# TrafficFlowBench Task 1·2·3 — Kaggle 통합 노트북

**`codex/task-123`의 변경을 이 노트북 하나에 포함했습니다.** Git clone, 별도 프로젝트 파일, Google Drive 연결은 필요하지 않습니다. 원본 코드는 아래 셀에 읽을 수 있는 형태로 들어 있으며 메모리에서 불러옵니다. 데이터와 기존 체크포인트는 Kaggle의 **Add Input**으로 연결해 주세요.

- Task 1·3: MagiNet, `temporal_direct`, ΔN 손실 비교, 관측 문맥·채널별 공간 보정, 과거 구간 검색을 포함합니다.
- Task 2: 공개 `T` 입력, 44특징, epoch별 표본 재추출, 조건별 간격, pooled MLP, 날짜 분산 검증을 포함합니다.
- 최종 제출: Task 1·2 예측과 **기존 Task 4 CSV**를 연결합니다. Task 4 파일을 지정하지 않으면 기존 `λ=0.05, p=0` NNLS로 채울 수 있습니다. Task 3의 별도 제출 행은 없습니다.

**실행 순서**

1. Kaggle에 이 `.ipynb`를 가져오고 대회 데이터를 Add Input으로 연결합니다. `corridors/`, `task1/`, `task2/`가 있는 배포본이 필요합니다.
2. 첫 설정 셀의 `RELEASE_ROOT`는 자동 탐색을 사용하거나 실제 입력 경로로 지정합니다.
3. 기본 `RUN_MODE="smoke"`로 한 패널·작은 모델·2 epoch의 실행 흐름을 점검합니다. 이 모드의 Task 2는 명시적으로 `scan`을 사용하고, MagiNet의 물리 손실 준비 기간 5 epoch에는 State만 학습합니다.
4. 전체 학습은 `RUN_MODE="train"`, `PANELS=None`으로 실행합니다. Task 2의 기본 backend는 원본과 같은 **Mamba2**입니다. 필요한 경우 Internet을 켜서 고정 버전 패키지를 설치합니다. GPU에서 동작하지 않으면 학습 전에 오류로 안내하며 backend를 몰래 바꾸지 않습니다. `QUEUE_BACKEND="scan"`은 직접 선택할 수 있지만 Mamba2와 같은 모델은 아닙니다.
5. 이전 Notebook의 Output을 Add Input으로 연결하고 `RESTORE_ROOT`를 이전 `trafficflow_task123/` 폴더로 지정하면 재개할 수 있습니다. `RESUME=True`에서 epoch는 **누적 목표**입니다. 과제별 체크포인트가 없으면 새로 시작하고, 미완료 모델은 재개하며, 목표 epoch를 채웠거나 조기 종료된 모델은 그대로 사용합니다. 예측만 할 때는 `RUN_MODE="predict"`입니다.

입력 원본은 수정하지 않습니다. 캐시·체크포인트·보고서는 `/kaggle/working/trafficflow_task123/`, 완성된 제출은 `/kaggle/working/submission.csv`에 저장합니다. Kaggle의 Save Version으로 Output을 보존해 주세요.

원본: [task 123 커밋](https://github.com/mmt-1234/trafficflowbench/commit/cdf888214fe8a54b73302f3988b764b74b3f30a4). State는 로컬 Task 1 지표이며 Queue·Physics는 대리 지표입니다. 이 노트북에 대회 데이터로 새로 측정한 점수는 들어 있지 않습니다. [Kaggle 실행 안내](https://www.kaggle.com/docs/notebooks), [Mamba 구현](https://github.com/state-spaces/mamba).


## 1. 실행 설정

`MAGINET_EXPERIMENT`는 새 `channel_graph`를 기본으로 사용합니다. `baseline`, `temporal_direct`, `temporal_direct_dn003`, `temporal_direct_dnzero`, `temporal_direct_learned_graph`, `temporal_direct_dn003_learned_graph`, `temporal_direct_dn003_context_graph`도 선택할 수 있습니다. 새 학습의 선택 기준은 State로 통일합니다. 기존 실행을 재개할 때는 저장된 기준을 유지합니다.

`QUEUE_FLAGS`를 비우면 기존 입력·표본·선형 출력층을 사용합니다. 단독 옵션 비교 시 `QUEUE_TAG`를 바꾸어 별도 결과 폴더에 저장해 주세요. 모든 비교에서 검증 날짜 분산 설정과 학습 예산을 맞춰야 합니다. `USE_RETRIEVAL=True`는 1월 전반에서 검색 방법·속도/교통량 혼합 비율을 선택하고 후반·예측에 고정 적용합니다. 2월은 이미 개발에 사용한 추가 진단 구간입니다.


In [16]:
from pathlib import Path

RUN_MODE = "train"             # "smoke", "train", "predict"
RELEASE_ROOT = None             # 예: "/kaggle/input/your-data/kaggle_public"
WORK_ROOT = Path("/kaggle/working/trafficflow_task123")
PANELS = None                   # None: 전체. smoke는 D12_I5_S 등 한 패널을 자동 선택합니다.
RESTORE_ROOT = None             # 이전 Output의 trafficflow_task123/ 폴더
RESUME = False                  # 과제별 새 학습/재개/완료 판단. epoch는 누적 목표입니다.

MAGINET_EXPERIMENT = "temporal_direct_dn003_channel_graph"
MAGINET_EPOCHS = 50
MAGINET_BATCH_SIZE = 2
QUEUE_EPOCHS = 75
QUEUE_BACKEND = "mamba2"         # 전체 학습용. smoke는 "scan" 고정
QUEUE_TAG = "enhanced"
QUEUE_FLAGS = ["--queue-origin", "--queue-resample", "--queue-condition-gap",
               "--queue-changes", "--queue-head", "pooled-mlp"]
VALIDATION_WINDOWS = 20
SMOKE_EPOCHS = 2
USE_RETRIEVAL = True
RUN_FEBRUARY = False
MAKE_PREDICTIONS = True
MAKE_SUBMISSION = True          # 전체 패널 예측에서만 완성합니다.
TASK4_FILES = []                # validation/private를 모두 포함하는 Task 4 CSV 경로 목록
BUILD_TASK4_BASELINE = True     # TASK4_FILES가 없을 때 기존 NNLS 사용
INSTALL_MISSING_PACKAGES = True
CPU_THREADS = 2
SEED = 42


## 2. 패키지와 입력 경로

Kaggle에 설치된 PyTorch를 사용합니다. 부족한 일반 패키지만 설치하며, Mamba2의 고정 버전은 전체 학습·해당 체크포인트 예측에서 필요할 때 설치합니다. 인터넷을 사용할 수 없다면 패키지를 미리 설치하거나 wheel을 Input으로 연결해 주세요. 대회 데이터 전체를 복사하지 않고 쓰기 가능한 폴더에서 원본을 참조하므로 Queue의 과거 평균 캐시도 읽기 전용 Input 밖에 저장됩니다.


In [17]:
import ast
import contextlib
import gc
import hashlib
import importlib.util
import json
import os
import shlex
import shutil
import subprocess
import sys
import types

if RUN_MODE not in {"smoke", "train", "predict"}:
    raise ValueError("RUN_MODE를 smoke/train/predict 중에서 지정해 주세요.")
packages = {"numpy": "numpy", "pandas": "pandas>=2.0", "pyarrow": "pyarrow", "scipy": "scipy"}
missing = [requirement for name, requirement in packages.items() if importlib.util.find_spec(name) is None]
if missing:
    if not INSTALL_MISSING_PACKAGES:
        raise RuntimeError(f"필요한 패키지를 먼저 설치해 주세요: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import numpy as np
import pandas as pd
import torch
if tuple(int(n) for n in torch.__version__.split("+")[0].split(".")[:2]) < (2, 3):
    raise RuntimeError("PyTorch 2.3 이상이 필요합니다. Kaggle의 현재 GPU 이미지를 사용해 주세요.")
if int(pd.__version__.split(".")[0]) < 2:
    raise RuntimeError("pandas 2.0 이상이 필요합니다.")
torch.set_num_threads(CPU_THREADS)
WORK_ROOT = Path(WORK_ROOT).resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def locate_release(requested, search_root=Path("/kaggle/input")):
    if requested is not None:
        p = Path(requested).expanduser()
        candidates = [p, p / "kaggle_public"]
    else:
        candidates = [p.parent for p in search_root.rglob("submission_key.csv")]
    matches = sorted({p.resolve() for p in candidates
                      if all((p / name).is_dir() for name in ("corridors", "task1", "task2"))})
    if len(matches) != 1:
        raise ValueError(f"배포본 경로를 하나로 지정해 주세요. RELEASE_ROOT 후보: {matches}")
    return matches[0]

INPUT_RELEASE = locate_release(RELEASE_ROOT)
RELEASE = WORK_ROOT / "data" / "release"
RELEASE.mkdir(parents=True, exist_ok=True)
for source in INPUT_RELEASE.iterdir():
    destination = RELEASE / source.name
    if destination.is_symlink() or destination.exists():
        if not destination.is_symlink() or destination.resolve() != source.resolve():
            raise FileExistsError(f"다른 입력이 연결돼 있습니다. 새 WORK_ROOT를 사용해 주세요: {destination}")
    else:
        destination.symlink_to(source, target_is_directory=source.is_dir())
RESULTS = WORK_ROOT / "reports" / ("smoke" if RUN_MODE == "smoke" else "full")
if RESTORE_ROOT is not None:
    previous = Path(RESTORE_ROOT) / "reports"
    if not previous.is_dir():
        raise FileNotFoundError(f"이전 trafficflow_task123/reports 폴더가 필요합니다: {previous}")
    target = WORK_ROOT / "reports"
    if target.exists() and any(target.iterdir()):
        raise FileExistsError("현재 결과를 덮어쓸 수 없습니다. 빈 WORK_ROOT에서 복원해 주세요.")
    shutil.copytree(previous, target, dirs_exist_ok=True)
all_panels = sorted(p.name for p in (RELEASE / "corridors").iterdir() if p.is_dir())
selected_panels = list(PANELS) if PANELS is not None else None
if RUN_MODE == "smoke" and selected_panels is None:
    eligible = [p for p in all_panels if p not in {"D12_I405_N", "D12_I405_S"}]
    selected_panels = ["D12_I5_S" if "D12_I5_S" in eligible else eligible[0]]
if selected_panels is not None and not set(selected_panels) <= set(all_panels):
    raise ValueError("PANELS에 입력에 없는 패널이 있습니다.")
if selected_panels == [] or not set(selected_panels or all_panels) - {"D12_I405_N", "D12_I405_S"}:
    raise ValueError("Task 2 대상 패널을 하나 이상 선택해 주세요.")
EFFECTIVE_BACKEND = "scan" if RUN_MODE == "smoke" else QUEUE_BACKEND
MAGINET_DIR = RESULTS / "maginet" / MAGINET_EXPERIMENT
RETRIEVAL_SELECTION = MAGINET_DIR / "retrieval" / "selected_retrieval.json"
QUEUE_DIR = RESULTS / "queue" / f"{QUEUE_TAG}-{EFFECTIVE_BACKEND}"
QUEUE_CHECKPOINT = QUEUE_DIR / "queue.pt"
if RUN_MODE == "predict":
    queue_saved = torch.load(QUEUE_CHECKPOINT, map_location="cpu", weights_only=True)
    EFFECTIVE_BACKEND = queue_saved["model_config"]["backend"]
if RUN_MODE == "train" and not torch.cuda.is_available():
    raise RuntimeError("전체 학습에는 GPU를 켜 주세요. CPU에서는 smoke로 흐름을 확인할 수 있습니다.")
if EFFECTIVE_BACKEND.startswith("mamba") and importlib.util.find_spec("mamba_ssm") is None:
    if not INSTALL_MISSING_PACKAGES:
        raise RuntimeError("Mamba 패키지가 필요합니다. 설치하거나 QUEUE_BACKEND를 명시적으로 바꿔 주세요.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops==0.8.2", "ninja==1.13.2",
                    "apache-tvm-ffi==0.1.9"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation",
                    "git+https://github.com/state-spaces/mamba.git@e9594ce1c732d97440f0332fdc43170a2294dbfa"], check=True)
CACHE_DIR = WORK_ROOT / "cache" / "maginet"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
QUEUE_PRECISION = "bf16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "fp32"
print({"input": str(INPUT_RELEASE), "writable_release": str(RELEASE), "mode": RUN_MODE,
       "panels": selected_panels or all_panels, "queue_backend": EFFECTIVE_BACKEND,
       "queue_precision": QUEUE_PRECISION, "torch": str(torch.__version__),
       "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 39.3 MB/s eta 0:00:00
  Cloning https://github.com/state-spaces/mamba.git (to revision e9594ce1c732d97440f0332fdc43170a2294dbfa) to /tmp/pip-req-build-7rlu3s66


  Running command git clone --filter=blob:none --quiet https://github.com/state-spaces/mamba.git /tmp/pip-req-build-7rlu3s66
  Running command git rev-parse -q --verify 'sha^e9594ce1c732d97440f0332fdc43170a2294dbfa'
  Running command git fetch -q https://github.com/state-spaces/mamba.git e9594ce1c732d97440f0332fdc43170a2294dbfa


  Resolved https://github.com/state-spaces/mamba.git to commit e9594ce1c732d97440f0332fdc43170a2294dbfa
  Preparing metadata (pyproject.toml): started


  Running command git submodule update --init --recursive -q


  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 28.5 MB/s eta 0:00:00
  Created wheel for mamba_ssm: filename=mamba_ssm-2.3.2.post1-py3-none-any.whl size=223780 sha256=63c6117dca91af0ccca555f87bd09cd7c8aee8f26678456da7b1a16445d91f39
  Stored in directory: /root/.cache/pip/wheels/51/82/75/5f613916d84b63263b4096e6c725f8f83b51de8e8b78c0762d
Successfully built mamba_ssm
{'input': '/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggl

## 3. 내장 소스

아래 여섯 셀은 원격 커밋의 원본 코드입니다. 셀을 펼쳐 확인하거나 수정할 수 있습니다. 실행 시 내부 import 이름만 이 노트북 전용 이름으로 바꾸고, 파일을 생성하지 않고 메모리에 정의합니다. 원본 SHA-256은 불러오기 전에 검사합니다. 이후 셀은 이 함수들을 직접 호출하며 외부 프로세스나 외부 프로젝트를 실행하지 않습니다.


In [18]:
SOURCE_COMMIT = "cdf888214fe8a54b73302f3988b764b74b3f30a4"
SOURCE_HASHES = {'data': 'fbc430d06cb4848dfd9f2e0966ac43e7c1e9cb3944c6c69518724d5e47323258', 'fast_model': 'b2a2ca6facba4e49f94eecc56fc3caf5adfdae36d4d215618d71f4c51c9c85a6', 'learning': '0958f4eadc59715ab5ab07885e84a2595440c2436fb1a8aeed64c006a04bd12f', 'historical_retrieval': '7089b3bac861b4aac54d6b89b615e12b965fd679ffd6f311fecc038d787555f4', 'maginet': '79f9765081fce047d7ce93309f1b5b090086ed4b4c57b00536535b3ec2c53024', 'run': '0c676b90a32e203eb10139c25124a3a9d21c4d0c9dc5a4a6b35ca6ab25399a0e'}
MODULE_NAMES = {name: "_trafficflow_task123_" + name for name in SOURCE_HASHES}
MODULES = {}

def load_source(name, source):
    if hashlib.sha256(source.encode()).hexdigest() != SOURCE_HASHES[name]:
        raise ValueError(f"내장 원본이 바뀌었습니다. 수정 의도가 있다면 해당 해시도 갱신해 주세요: {name}")
    tree = ast.parse(source)
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom) and node.module in MODULE_NAMES:
            node.module = MODULE_NAMES[node.module]
    module = types.ModuleType(MODULE_NAMES[name])
    module.__file__ = str(WORK_ROOT / "src" / f"{name}.py")
    sys.modules[module.__name__] = module
    exec(compile(tree, module.__file__, "exec"), module.__dict__)
    MODULES[name] = module
    print(f"내장 코드 로드: {name} ({len(source.splitlines())}줄)")


In [19]:
# src/data.py — 원본 cdf888214fe8a54b73302f3988b764b74b3f30a4
load_source('data', r'''"""Released panel data to tensors, with masks applied before aggregation."""

from __future__ import annotations

from dataclasses import dataclass, field
import hashlib
import json
from pathlib import Path
import heapq

import numpy as np
import pandas as pd


def scan_layout(link_ids: list[str], topology: pd.DataFrame) -> tuple[list[str], tuple[tuple[int, int], ...]]:
    """Topological order within each weakly connected component; cycles are stable."""
    links = set(link_ids)
    outgoing = {link: set() for link in links}
    neighbors = {link: set() for link in links}
    for row in topology.itertuples(index=False):
        target = str(row.link_id)
        if target not in links:
            continue
        for source in str(getattr(row, "incoming_link_ids", "")).split(";"):
            source = source.strip()
            if source in links and source != target:
                outgoing[source].add(target)
                neighbors[source].add(target)
                neighbors[target].add(source)
        for dest in str(getattr(row, "outgoing_link_ids", "")).split(";"):
            dest = dest.strip()
            if dest in links and dest != target:
                outgoing[target].add(dest)
                neighbors[target].add(dest)
                neighbors[dest].add(target)
    seen: set[str] = set()
    ordered: list[str] = []
    segments: list[tuple[int, int]] = []
    for root in sorted(links):
        if root in seen:
            continue
        component = {root}
        pending = [root]
        seen.add(root)
        while pending:
            for neighbor in neighbors[pending.pop()]:
                if neighbor not in seen:
                    seen.add(neighbor)
                    component.add(neighbor)
                    pending.append(neighbor)
        indegree = {link: 0 for link in component}
        for source in component:
            for target in outgoing[source] & component:
                indegree[target] += 1
        ready = [link for link in component if indegree[link] == 0]
        heapq.heapify(ready)
        part = []
        while ready:
            source = heapq.heappop(ready)
            part.append(source)
            for target in outgoing[source] & component:
                indegree[target] -= 1
                if indegree[target] == 0:
                    heapq.heappush(ready, target)
        part.extend(sorted(component - set(part)))
        start = len(ordered)
        ordered.extend(part)
        segments.append((start, len(ordered)))
    return ordered, tuple(segments)


@dataclass
class PanelInfo:
    name: str
    links: list[str]
    segments: tuple[tuple[int, int], ...]
    link_index: dict[str, int]
    free_speed: np.ndarray
    lanes: np.ndarray
    lengths: np.ndarray
    cutoffs: np.ndarray
    station_lanes: dict[str, float]
    capacity: np.ndarray | None = None
    jam_density: np.ndarray | None = None
    position: np.ndarray | None = None
    edges: np.ndarray = field(default_factory=lambda: np.empty((2, 0), dtype=np.int64))
    ramp_map: dict[str, tuple[int, int]] = field(default_factory=dict)

    @classmethod
    def load(cls, release: Path, panel: str) -> "PanelInfo":
        panel_dir = release / "corridors" / panel
        candidates = sorted((panel_dir / "train" / "mainline_states").glob("**/*.parquet"))
        if not candidates:
            raise FileNotFoundError(f"No original train observations under {panel_dir}")
        observed = pd.read_parquet(candidates[0], columns=["link_id"]).link_id.astype(str).unique().tolist()
        network = panel_dir / "network"
        topology = pd.read_csv(network / "lwr_mainline_topology.csv", dtype=str)
        # Latent mainline connectors carry no detector value, but preserve the
        # road between observed links during the spatial scan.
        links, segments = scan_layout(sorted(set(observed) | set(topology.link_id.dropna().astype(str))), topology)
        frame = pd.read_csv(network / "links.csv", dtype={"link_id": str}).drop_duplicates("link_id").set_index("link_id")
        fd_path = network / "fd_parameters.csv"
        fd = pd.read_csv(fd_path, dtype={"link_id": str, "station_id": str}) if fd_path.exists() else pd.DataFrame()
        grouped = fd.groupby("link_id").mean(numeric_only=True) if not fd.empty else pd.DataFrame()

        def parameter(column: str, default: float) -> np.ndarray:
            raw = pd.to_numeric(frame[column], errors="coerce").reindex(links) if column in frame else pd.Series(default, index=links)
            if column in grouped:
                raw = pd.to_numeric(grouped[column], errors="coerce").reindex(links).fillna(raw)
            return raw.fillna(default).to_numpy(dtype=np.float32)

        free_speed = parameter("free_speed_kmh", 105.0)
        lanes = np.clip(parameter("lanes", 1.0), 1, 16)
        lengths = parameter("length_km", 1.0)
        cutoffs = 0.6 * free_speed
        if "v_cut" in grouped:
            cutoffs = pd.to_numeric(grouped.v_cut, errors="coerce").reindex(links).fillna(pd.Series(cutoffs, index=links)).to_numpy(dtype=np.float32)
        station_lanes: dict[str, float] = {}
        if "station_id" in fd and "lanes" in fd:
            for row in fd.dropna(subset=["station_id", "lanes"]).itertuples(index=False):
                station_lanes[str(row.station_id)] = float(row.lanes)
        lookup = {link: i for i, link in enumerate(links)}
        pairs = set()
        for row in topology.itertuples(index=False):
            target = str(row.link_id)
            for source in str(getattr(row, "incoming_link_ids", "")).split(";"):
                if source in lookup and target in lookup and source != target:
                    pairs.add((lookup[source], lookup[target]))
            for dest in str(getattr(row, "outgoing_link_ids", "")).split(";"):
                if target in lookup and dest in lookup and dest != target:
                    pairs.add((lookup[target], lookup[dest]))
        positions = topology.drop_duplicates("link_id").set_index("link_id")
        position = pd.to_numeric(positions.get("order_index", pd.Series(dtype=float)), errors="coerce").reindex(links)
        position = position.fillna(pd.Series(np.arange(len(links)), index=links)).to_numpy(np.float32)
        position = (position - position.min()) / max(float(np.ptp(position)), 1.0)
        ramp_map = {}
        ramp_path = network / "ramp_attachment_map.csv"
        if ramp_path.exists():
            for row in pd.read_csv(ramp_path, dtype=str).itertuples(index=False):
                if row.nearest_mainline_link_id in lookup:
                    if row.ramp_type not in {"OR", "FR", "on", "off"}:
                        raise ValueError(f"Unknown ramp type: {row.ramp_type}")
                    ramp_map[row.ramp_link_id] = (lookup[row.nearest_mainline_link_id], int(row.ramp_type in {"FR", "off"}))
        return cls(panel, links, segments, lookup, free_speed, lanes, lengths, cutoffs, station_lanes,
                   parameter("capacity_vph", 6000.0), parameter("k_jam", 400.0), position,
                   np.array(sorted(pairs), dtype=np.int64).reshape(-1, 2).T, ramp_map)


@dataclass
class StationDay:
    timestamps: list[str]
    station_ids: list[str]
    station_links: list[str]
    station_to_link: np.ndarray
    lanes: np.ndarray
    speed: np.ndarray
    flow_lane: np.ndarray
    eligible: np.ndarray
    occupancy: np.ndarray | None = None
    quality: np.ndarray | None = None

    @classmethod
    def load(cls, path: Path, panel: PanelInfo) -> "StationDay":
        columns = ["timestamp", "station_id", "link_id", "speed_kmh", "flow_vph", "is_score_eligible"]
        # Optional published features also exist in Task 2 histories.
        import pyarrow.parquet as pq
        available = set(pq.read_schema(path).names) if path.stat().st_size else set(columns)
        columns += [c for c in ("occupancy", "pct_observed") if c in available]
        frame = pd.read_parquet(path, columns=columns)
        for column in ("timestamp", "station_id", "link_id"):
            frame[column] = frame[column].astype(str)
        if frame.duplicated(["timestamp", "station_id", "link_id"]).any():
            raise ValueError(f"Duplicate station/time rows in {path}")
        stations = frame[["station_id", "link_id"]].drop_duplicates()
        if stations.station_id.duplicated().any():
            raise ValueError(f"Station maps to multiple links in {path}")
        stations = stations.sort_values(["link_id", "station_id"])
        station_ids = stations.station_id.tolist()
        station_links = stations.link_id.tolist()
        unknown = set(station_links) - set(panel.link_index)
        if unknown:
            raise ValueError(f"Links absent from panel layout: {sorted(unknown)[:4]}")
        timestamps = sorted(frame.timestamp.unique().tolist())
        index = pd.MultiIndex.from_product([timestamps, station_ids], names=["timestamp", "station_id"])
        aligned = frame.set_index(["timestamp", "station_id"]).reindex(index)
        speed = pd.to_numeric(aligned.speed_kmh, errors="coerce").to_numpy(dtype=np.float32).reshape(len(timestamps), -1)
        flow = pd.to_numeric(aligned.flow_vph, errors="coerce").to_numpy(dtype=np.float32).reshape(len(timestamps), -1)
        lanes = np.array([panel.station_lanes.get(s, panel.lanes[panel.link_index[l]]) for s, l in zip(station_ids, station_links)], dtype=np.float32)
        lanes = np.clip(lanes, 1, 16)
        eligible = aligned.is_score_eligible.fillna(False).astype(bool).to_numpy(copy=True).reshape(len(timestamps), -1)
        eligible &= np.isfinite(speed) & np.isfinite(flow)
        occupancy = pd.to_numeric(aligned.occupancy, errors="coerce").to_numpy(np.float32).reshape(speed.shape) if "occupancy" in aligned else np.full_like(speed, np.nan)
        quality = pd.to_numeric(aligned.pct_observed, errors="coerce").fillna(0).to_numpy(np.float32).reshape(speed.shape) / 100 if "pct_observed" in aligned else eligible.astype(np.float32)
        return cls(timestamps, station_ids, station_links,
                   np.array([panel.link_index[l] for l in station_links], dtype=np.int64),
                   lanes, speed / 25.0, flow / lanes[None, :] / 600.0, eligible, occupancy, quality)


def station_features(day: StationDay, panel: PanelInfo, visible: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Interpolate each station after masking; never aggregate a hidden answer."""
    steps, stations = day.speed.shape
    context = np.zeros((steps, stations, 3), dtype=np.float32)
    position = np.arange(steps)
    for station in range(stations):
        good = np.flatnonzero(visible[:, station])
        if len(good):
            context[:, station, 0] = np.interp(position, good, day.speed[good, station])
            context[:, station, 1] = np.interp(position, good, day.flow_lane[good, station])
    context[:, :, 2] = day.lanes[None, :] / 5.0
    inputs = link_features(day.timestamps, day.speed, day.flow_lane, visible, day.station_to_link, panel)
    return inputs, context


def station_window_features(day: StationDay, panel: PanelInfo, visible: np.ndarray,
                            index: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    window = StationDay([day.timestamps[i] for i in index], day.station_ids, day.station_links,
                        day.station_to_link, day.lanes, day.speed[index], day.flow_lane[index], day.eligible[index])
    return station_features(window, panel, visible[index])


def link_features(timestamps: list[str], speed: np.ndarray, flow: np.ndarray, visible: np.ndarray,
                  station_to_link: np.ndarray, panel: PanelInfo) -> np.ndarray:
    steps = len(timestamps)
    nodes = len(panel.links)
    values = np.zeros((steps, nodes, 9), dtype=np.float32)
    counts = np.zeros((steps, nodes), dtype=np.float32)
    for station, link in enumerate(station_to_link):
        good = visible[:, station]
        values[good, link, 0] += speed[good, station]
        values[good, link, 1] += flow[good, station]
        counts[good, link] += 1
    values[:, :, 0:2] /= np.maximum(counts[:, :, None], 1)
    values[:, :, 2] = (counts > 0).astype(np.float32)
    values[:, :, 3] = panel.free_speed[None, :] / 125.0
    values[:, :, 4] = panel.lanes[None, :] / 5.0
    values[:, :, 5] = panel.lengths[None, :] / 5.0
    dates = pd.to_datetime(timestamps, utc=True, format="ISO8601")
    slot = np.asarray(dates.hour * 12 + dates.minute // 5)
    values[:, :, 6] = np.sin(2 * np.pi * slot / 288)[:, None]
    values[:, :, 7] = np.cos(2 * np.pi * slot / 288)[:, None]
    values[:, :, 8] = dates.dayofweek.to_numpy()[:, None] / 6.0
    return values


# The first nine channels retain the legacy order; v2 checkpoints store this list.
FEATURE_NAMES = ("speed", "flow_lane", "both_observed", "free_speed", "lanes", "length",
                 "time_sin", "time_cos", "weekday", "occupancy", "speed_observed", "flow_observed",
                 "occupancy_observed", "pct_observed", "speed_age", "flow_age", "occupancy_age",
                 "capacity", "road_position", "on_ramp_attached", "off_ramp_attached", "on_ramp_flow",
                 "off_ramp_flow", "on_ramp_valid", "off_ramp_valid", "prior_speed", "prior_flow_lane",
                 "prior_occupancy", "time_slot", "day_of_week", "relative_time")

QUEUE_CHANGE_NAMES = ("speed_margin",) + tuple(
    f"{channel}_{field}_{minutes}m" for minutes in (5, 15)
    for channel in ("speed", "flow_lane", "occupancy") for field in ("delta", "valid")
)


def queue_feature_names(changes=False):
    return FEATURE_NAMES + QUEUE_CHANGE_NAMES if changes else FEATURE_NAMES

STATE_CONTEXT_NAMES = ("speed", "flow_lane", "lanes") + tuple(
    f"{channel}_{field}" for channel in ("speed", "flow")
    for field in ("left", "right", "left_age", "right_age", "left_valid", "right_valid", "observed")
) + ("pct_observed", "prior_speed", "prior_flow_lane")


def aggregate_channels(day: StationDay, panel: PanelInfo, visible: np.ndarray | None = None):
    values = np.stack((day.speed, day.flow_lane,
                       day.occupancy if day.occupancy is not None else np.full_like(day.speed, np.nan)), -1)
    good = np.isfinite(values)
    if visible is not None:
        good &= visible[..., None] if visible.ndim == 2 else visible
    sums = np.zeros((len(day.timestamps), len(panel.links), 3), np.float32)
    counts = np.zeros_like(sums)
    quality = np.zeros(sums.shape[:2], np.float32)
    qcount = np.zeros_like(quality)
    for station, link in enumerate(day.station_to_link):
        sums[:, link] += np.where(good[:, station], values[:, station], 0)
        counts[:, link] += good[:, station]
        present = good[:, station].any(-1)
        q = day.quality[:, station] if day.quality is not None else present.astype(np.float32)
        # Observation quality remains public even when measured values are masked.
        quality[:, link] += np.where(np.isfinite(q), q, 0)
        qcount[:, link] += np.isfinite(q)
    return sums / np.maximum(counts, 1), counts > 0, quality / np.maximum(qcount, 1)


def features_v2(day: StationDay, panel: PanelInfo, profile: np.ndarray,
                ramp: np.ndarray | None = None, visible: np.ndarray | None = None):
    values, good, quality = aggregate_channels(day, panel, visible)
    steps, nodes = values.shape[:2]
    out = np.zeros((steps, nodes, len(FEATURE_NAMES)), np.float32)
    dates = pd.to_datetime(day.timestamps, utc=True, format="ISO8601")
    slots = np.asarray(dates.hour * 12 + dates.minute // 5)
    dow = dates.dayofweek.to_numpy()
    out[..., :2] = values[..., :2]
    out[..., 2] = good[..., :2].all(-1)
    out[..., 3] = panel.free_speed / 125
    out[..., 4] = panel.lanes / 5
    out[..., 5] = panel.lengths / 5
    out[..., 6] = np.sin(2 * np.pi * slots / 288)[:, None]
    out[..., 7] = np.cos(2 * np.pi * slots / 288)[:, None]
    out[..., 8] = dow[:, None] / 6
    out[..., 9] = values[..., 2]
    out[..., 10:13] = good
    out[..., 13] = quality
    elapsed = np.asarray((dates - dates[0]).total_seconds() / 300, dtype=np.float32)
    for channel in range(3):
        last = np.maximum.accumulate(np.where(good[..., channel], elapsed[:, None], -72), axis=0)
        out[..., 14 + channel] = np.minimum(elapsed[:, None] - last, 72) / 72
    out[..., 17] = (panel.capacity if panel.capacity is not None else np.full(nodes, 6000)) / 10000
    out[..., 18] = panel.position if panel.position is not None else np.arange(nodes) / max(nodes - 1, 1)
    for link, kind in panel.ramp_map.values():
        out[:, link, 19 + kind] = 1
    if ramp is not None:
        out[..., 21:23] = ramp[..., :2] / 600
        out[..., 23:25] = ramp[..., 2:]
    out[..., 25:28] = profile[dow * 288 + slots]
    out[..., 28] = slots[:, None] / 287
    out[..., 29] = dow[:, None] / 6
    out[..., 30] = np.arange(steps)[:, None] / max(steps - 1, 1)
    return out


def queue_features(day, panel, profile, ramp=None, origin=None, changes=False):
    """Build Queue inputs from the supplied past and optional released origin only."""
    x = features_v2(day, panel, profile, ramp)
    if origin is None and not changes:
        return x
    times = pd.to_datetime(day.timestamps, utc=True, format="ISO8601")
    if origin is not None:
        if origin.shape != (1, len(panel.links), len(FEATURE_NAMES)):
            raise ValueError("Unexpected released Queue origin shape")
        x = np.concatenate((x, origin), axis=0)
        times = times.append(pd.DatetimeIndex([times[-1] + pd.Timedelta(minutes=5)]))
        elapsed = np.asarray((times - times[0]).total_seconds() / 300, dtype=np.float32)
        for channel in range(3):
            last = np.maximum.accumulate(np.where(x[..., 10 + channel] > 0, elapsed[:, None], -72), axis=0)
            x[..., 14 + channel] = np.minimum(elapsed[:, None] - last, 72) / 72
        x[..., 30] = np.arange(len(x))[:, None] / (len(x) - 1)
    if not changes:
        return x
    extra = np.zeros((*x.shape[:2], len(QUEUE_CHANGE_NAMES)), np.float32)
    extra[..., 0] = np.where(x[..., 10] > 0, x[..., 0] - panel.cutoffs[None] / 25, 0)
    column = 1
    for lag in (1, 3):
        consecutive = np.asarray(times[lag:] - times[:-lag] == pd.Timedelta(minutes=5 * lag))
        for channel, position in enumerate((0, 1, 9)):
            good = (x[lag:, :, 10 + channel] > 0) & (x[:-lag, :, 10 + channel] > 0) & consecutive[:, None]
            extra[lag:, :, column] = np.where(good, x[lag:, :, position] - x[:-lag, :, position], 0)
            extra[lag:, :, column + 1] = good
            column += 2
    return np.concatenate((x, extra), axis=-1)


def slice_day(day: StationDay, index: np.ndarray) -> StationDay:
    return StationDay([day.timestamps[i] for i in index], day.station_ids, day.station_links,
                      day.station_to_link, day.lanes, day.speed[index], day.flow_lane[index], day.eligible[index],
                      None if day.occupancy is None else day.occupancy[index],
                      None if day.quality is None else day.quality[index])


def state_features_v2(day: StationDay, panel: PanelInfo, index: np.ndarray,
                      profile: np.ndarray, ramp: np.ndarray | None = None, rich=False):
    window = slice_day(day, index)
    x = features_v2(window, panel, profile, None if ramp is None else ramp[index])
    context = np.zeros((len(index), len(day.station_ids), len(STATE_CONTEXT_NAMES) if rich else 3), np.float32)
    dates = pd.to_datetime(window.timestamps, utc=True, format="ISO8601")
    slots = np.asarray(dates.dayofweek * 288 + dates.hour * 12 + dates.minute // 5)
    elapsed = np.asarray((dates - dates[0]).total_seconds() / 300, dtype=np.float32)
    position = np.arange(len(index))[:, None]
    for channel, values in enumerate((window.speed, window.flow_lane)):
        good = np.isfinite(values)
        prior = profile[slots[:, None], day.station_to_link[None], channel]
        # Search all stations together, within the masked window only.
        left = np.maximum.accumulate(np.where(good, position, -1), axis=0)
        right = np.minimum.accumulate(np.where(good, position, len(index))[::-1], axis=0)[::-1]
        left_valid, right_valid = left >= 0, right < len(index)
        left_safe, right_safe = np.clip(left, 0, len(index) - 1), np.clip(right, 0, len(index) - 1)
        left_value = np.take_along_axis(values, left_safe, axis=0)
        right_value = np.take_along_axis(values, right_safe, axis=0)
        first = np.where(left_valid, left_value, np.where(right_valid, right_value, prior)).astype(np.float64)
        last = np.where(right_valid, right_value, first).astype(np.float64)
        fraction = np.clip((position - left) / np.maximum(right - left, 1), 0, 1)
        # Match np.interp's index-based interpolation and constant endpoint extension.
        context[..., channel] = first + fraction * (last - first)
        if rich:
            offset = 3 + 7 * channel
            for side, safe, valid, endpoint in ((0, left_safe, left_valid, left_value),
                                               (1, right_safe, right_valid, right_value)):
                context[..., offset + side] = np.where(valid, endpoint, prior)
                context[..., offset + 2 + side] = np.where(valid, np.minimum(np.abs(elapsed[:, None] - elapsed[safe]), 72) / 72, 1)
                context[..., offset + 4 + side] = valid
            context[..., offset + 6] = good
            context[..., 18 + channel] = prior
    context[..., 2] = day.lanes / 5
    if rich:
        context[..., 17] = (np.clip(np.nan_to_num(window.quality), 0, 1)
                            if window.quality is not None else np.isfinite(window.speed).astype(np.float32))
    return x, context


def ramp_features(release: Path, panel: PanelInfo, split: str, timestamps: list[str]) -> np.ndarray:
    out = np.zeros((len(timestamps), len(panel.links), 4), np.float32)
    out[..., 2:] = 1  # A road without an attached ramp has known zero ramp flow.
    expected = np.zeros((len(panel.links), 2), np.float32)
    for link, kind in panel.ramp_map.values():
        expected[link, kind] += 1
    counts = np.zeros_like(out[..., :2])
    date = timestamps[0][:10]
    root = release / "corridors" / panel.name / split / "ramp_states"
    paths = sorted(set(root.glob(f"**/*{date}*.parquet")) | set(root.glob(f"**/*{date.replace('-', '_')}*.parquet")))
    time_index = {value: i for i, value in enumerate(timestamps)}
    for path in paths:
        frame = pd.read_parquet(path, columns=["timestamp", "ramp_link_id", "flow_vph", "pct_observed"])
        frame = frame[frame.timestamp.isin(time_index)].drop_duplicates(["timestamp", "ramp_link_id"])
        for row in frame.itertuples(index=False):
            if row.ramp_link_id in panel.ramp_map and pd.notna(row.flow_vph) and row.pct_observed >= 75:
                link, kind = panel.ramp_map[row.ramp_link_id]
                index = time_index[row.timestamp]
                out[index, link, kind] += float(row.flow_vph)
                counts[index, link, kind] += 1
    out[..., 2:] = counts >= expected[None]
    out[..., :2] *= out[..., 2:]
    return out


class TargetIndex:
    """Compact, cached exact template targets; never infer targets from nulls."""
    def __init__(self, release: Path, panel: str, station_ids: list[str]):
        source = release / "task1" / panel / "train" / "sample_submission_state.csv"
        signature = hashlib.sha256(json.dumps([source.stat().st_size, source.stat().st_mtime_ns, station_ids]).encode()).hexdigest()[:16]
        folder = release.parent / "processed_v2"
        folder.mkdir(parents=True, exist_ok=True)
        cache = folder / f"{panel}-targets-{signature}.npz"
        if not cache.exists():
            print(f"Preparing exact Task 1 targets: {panel}", flush=True)
            lookup = {station: i for i, station in enumerate(station_ids)}
            by_date = {}
            regimes = {}
            for frame in pd.read_csv(source, usecols=["timestamp", "station_id", "mask_regime"], dtype=str, chunksize=200000):
                positions = frame.station_id.map(lookup)
                if positions.isna().any():
                    raise ValueError(f"Template has unknown stations: {panel}")
                slot = frame.timestamp.str[11:13].astype(np.int64) * 12 + frame.timestamp.str[14:16].astype(np.int64) // 5
                frame = frame.assign(_date=frame.timestamp.str[:10], _cell=(slot * len(station_ids) + positions).astype(np.uint32))
                for date, part in frame.groupby("_date", sort=False):
                    unique = part.mask_regime.unique()
                    if len(unique) != 1 or date in regimes and regimes[date] != unique[0]:
                        raise ValueError(f"Multiple regimes in {panel} {date}")
                    regimes[date] = unique[0]
                    by_date.setdefault(date, []).append(part._cell.to_numpy())
            arrays = {date: np.concatenate(parts) for date, parts in by_date.items()}
            if any(len(np.unique(cells)) != len(cells) for cells in arrays.values()):
                raise ValueError(f"Duplicate Task 1 template targets: {panel}")
            arrays["dates"] = np.array(list(regimes))
            arrays["regimes"] = np.array(list(regimes.values()))
            temporary = cache.with_suffix(".pending.npz")
            np.savez_compressed(temporary, **arrays)
            temporary.replace(cache)
        self.archive = np.load(cache, allow_pickle=False)
        self.regimes = dict(zip(self.archive["dates"].tolist(), self.archive["regimes"].tolist()))
        self.stations = station_ids

    def get(self, day: StationDay):
        date = day.timestamps[0][:10]
        if day.station_ids != self.stations or date not in self.regimes:
            raise ValueError(f"Target mapping changed or absent: {date}")
        mask = np.zeros(day.speed.shape, bool)
        slots = {int(value[11:13]) * 12 + int(value[14:16]) // 5: i for i, value in enumerate(day.timestamps)}
        cells = self.archive[date]
        mask[[slots[int(slot)] for slot in cells // len(self.stations)], cells % len(self.stations)] = True
        if (mask & ~day.eligible).any():
            raise ValueError(f"Template target is ineligible: {date}")
        return self.regimes[date], mask
''')


내장 코드 로드: data (476줄)


In [20]:
# src/fast_model.py — 원본 cdf888214fe8a54b73302f3988b764b74b3f30a4
load_source('fast_model', r'''"""FAST-style traffic models with legacy scan and official Mamba backends."""

from __future__ import annotations

import torch
from torch import nn
from torch.nn import functional as F

from data import STATE_CONTEXT_NAMES


class SpatialScan(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        self.parameters_for_step = nn.Linear(width, 3 * width)
        self.log_decay = nn.Parameter(torch.zeros(width))
        self.local = nn.Conv1d(width, width, 3, padding=1, groups=width)
        self.project = nn.Linear(width, width)

    def forward(self, x: torch.Tensor, segments: tuple[tuple[int, int], ...], reverse: bool) -> torch.Tensor:
        # Reverse inside each component, then scan all components together.
        order = None
        if reverse:
            order = torch.tensor([i for start, stop in segments for i in range(stop - 1, start - 1, -1)],
                                 device=x.device)
            x = x.index_select(2, order)
        count = x.shape[2]
        starts = torch.tensor([start for start, _ in segments], device=x.device)
        ends = torch.tensor([stop - 1 for _, stop in segments], device=x.device)
        left_mask = torch.ones(count, dtype=x.dtype, device=x.device)
        right_mask = torch.ones(count, dtype=x.dtype, device=x.device)
        left_mask[starts] = 0
        right_mask[ends] = 0

        # Equivalent to the depthwise width-3 convolution, with zeros at each
        # component boundary rather than borrowing a neighbor from another road.
        left = torch.cat((torch.zeros_like(x[:, :, :1]), x[:, :, :-1]), dim=2) * left_mask[None, None, :, None]
        right = torch.cat((x[:, :, 1:], torch.zeros_like(x[:, :, :1])), dim=2) * right_mask[None, None, :, None]
        weights = self.local.weight[:, 0]
        local = left * weights[:, 0] + x * weights[:, 1] + right * weights[:, 2] + self.local.bias

        delta, source, readout = self.parameters_for_step(x).chunk(3, -1)
        decay = torch.exp(-F.softplus(delta) * F.softplus(self.log_decay))
        emission = (1 - decay) * torch.tanh(source)
        decay = decay * left_mask[None, None, :, None]

        # Parallel prefix of affine recurrences: s_i = decay_i * s_(i-1) + emission_i.
        # A zero decay at each start resets the state between components.
        state = emission
        step = 1
        while step < count:
            next_state = state[:, :, step:] + decay[:, :, step:] * state[:, :, :-step]
            next_decay = decay[:, :, step:] * decay[:, :, :-step]
            state = torch.cat((state[:, :, :step], next_state), dim=2)
            decay = torch.cat((decay[:, :, :step], next_decay), dim=2)
            step *= 2
        scanned = state * torch.sigmoid(readout) * torch.sigmoid(local)
        if reverse:
            scanned = scanned.index_select(2, order)
        return self.project(scanned)


class FASTBlock(nn.Module):
    def __init__(self, width: int, heads: int):
        super().__init__()
        self.time_a = nn.MultiheadAttention(width, heads, batch_first=True)
        self.time_b = nn.MultiheadAttention(width, heads, batch_first=True)
        self.forward_scan = SpatialScan(width)
        self.reverse_scan = SpatialScan(width)
        self.norm_a = nn.LayerNorm(width)
        self.norm_s = nn.LayerNorm(width)
        self.norm_b = nn.LayerNorm(width)

    @staticmethod
    def attend(x: torch.Tensor, attention: nn.MultiheadAttention) -> torch.Tensor:
        batch, steps, nodes, width = x.shape
        sequence = x.transpose(1, 2).reshape(batch * nodes, steps, width)
        attended, _ = attention(sequence, sequence, sequence, need_weights=False)
        return attended.reshape(batch, nodes, steps, width).transpose(1, 2)

    def forward(self, x: torch.Tensor, segments: tuple[tuple[int, int], ...], bidirectional: bool) -> torch.Tensor:
        x = self.norm_a(x + self.attend(x, self.time_a))
        spatial = self.forward_scan(x, segments, False)
        if bidirectional:
            spatial = 0.5 * (spatial + self.reverse_scan(x, segments, True))
        x = self.norm_s(x + spatial)
        return self.norm_b(x + self.attend(x, self.time_b))


class FASTEncoder(nn.Module):
    def __init__(self, features: int = 9, width: int = 64, blocks: int = 2, heads: int = 4):
        super().__init__()
        self.input = nn.Linear(features, width)
        self.blocks = nn.ModuleList(FASTBlock(width, heads) for _ in range(blocks))

    def forward(self, x: torch.Tensor, segments: tuple[tuple[int, int], ...], bidirectional: bool = True) -> torch.Tensor:
        hidden = self.input(x)
        outputs = []
        for block in self.blocks:
            hidden = block(hidden, segments, bidirectional)
            outputs.append(hidden)
        return torch.stack(outputs).mean(0)


class StateModel(nn.Module):
    def __init__(self, width: int = 64, blocks: int = 2):
        super().__init__()
        self.encoder = FASTEncoder(width=width, blocks=blocks)
        self.head = nn.Linear(width + 3, 2)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, links: torch.Tensor, station_context: torch.Tensor,
                station_to_link: torch.Tensor, segments: tuple[tuple[int, int], ...],
                bidirectional: bool = True) -> torch.Tensor:
        hidden = self.encoder(links, segments, bidirectional)[:, :, station_to_link]
        correction = self.head(torch.cat((hidden, station_context), dim=-1))
        return F.relu(station_context[..., :2] + correction)


class QueueModel(nn.Module):
    def __init__(self, width: int = 64, blocks: int = 2):
        super().__init__()
        self.encoder = FASTEncoder(width=width, blocks=blocks)
        self.head = nn.Linear(width, 12)

    def forward(self, history: torch.Tensor, segments: tuple[tuple[int, int], ...],
                bidirectional: bool = True) -> tuple[torch.Tensor, torch.Tensor]:
        last = self.encoder(history, segments, bidirectional)[:, -1]
        output = self.head(last).reshape(last.shape[0], last.shape[1], 6, 2).transpose(1, 2)
        return output[..., 0], F.softplus(output[..., 1])


class MambaSpatial(nn.Module):
    """Official Mamba over independently padded road components."""
    def __init__(self, width: int, backend: str, state: int = 64):
        super().__init__()
        try:
            if backend == "mamba1":
                from mamba_ssm import Mamba
                self.ssm = Mamba(d_model=width, d_state=state, d_conv=2, expand=1)
            else:
                from mamba_ssm import Mamba2
                # The torch-convolution path also works without causal-conv1d.
                self.ssm = Mamba2(d_model=width, d_state=state, d_conv=2, expand=1,
                                  headdim=min(32, width), chunk_size=32, use_mem_eff_path=False)
        except ImportError as error:
            raise RuntimeError("Install requirements-mamba.txt in WSL; no implicit scan fallback is used") from error
        self.layouts = {}

    def forward(self, x, segments, reverse=False):
        batch, steps, nodes, width = x.shape
        key = segments, reverse, x.device
        if key not in self.layouts:
            buckets = {}
            for start, stop in segments:
                length = 1 << max((stop - start - 1).bit_length(), 0)
                indices = list(range(start, stop))
                if reverse:
                    indices.reverse()
                buckets.setdefault(length, []).append(indices)
            layouts = []
            for length, components in sorted(buckets.items()):
                gather = torch.zeros((len(components), length), dtype=torch.long, device=x.device)
                valid = torch.zeros_like(gather, dtype=torch.bool)
                for row, indices in enumerate(components):
                    gather[row, :len(indices)] = torch.tensor(indices, device=x.device)
                    valid[row, :len(indices)] = True
                layouts.append((gather, valid, gather[valid]))
            self.layouts[key] = layouts
        out = torch.zeros_like(x)
        for gather, valid, destination in self.layouts[key]:
            packed = x[:, :, gather] * valid[None, None, :, :, None]
            packed = packed.reshape(batch * steps * len(gather), gather.shape[1], width).contiguous()
            scanned = self.ssm(packed).reshape(batch, steps, *gather.shape, width)
            out = out.index_copy(2, destination, scanned[:, :, valid].to(out.dtype))
        return out


class GraphResidual(nn.Module):
    def __init__(self, width, mode="mean"):
        super().__init__()
        if mode not in ("mean", "reliability"):
            raise ValueError(f"Unknown graph mode: {mode}")
        self.mode = mode
        self.up = nn.Linear(width, width, bias=False)
        self.down = nn.Linear(width, width, bias=False)
        if mode == "reliability":
            self.edge_gate = nn.Linear(2 * width + 14, 1)
            self.mix_gate = nn.Linear(width + 7, 1)
            nn.init.zeros_(self.edge_gate.weight)
            nn.init.zeros_(self.edge_gate.bias)
            nn.init.zeros_(self.mix_gate.weight)
            nn.init.constant_(self.mix_gate.bias, -1)

    def forward(self, x, edges, observation=None):
        out = torch.zeros_like(x)
        if self.mode == "reliability":
            if observation is None or observation.shape[-1] < 17:
                raise ValueError("Reliability graph needs the public channel masks, quality, and ages")
            masks = observation[..., 10:13].clamp(0, 1)
            quality = observation[..., 13:14].clamp(0, 1)
            ages = observation[..., 14:17].clamp(0, 1)
            evidence = torch.cat((masks, quality, ages), -1).to(x.dtype)
            confidence = masks.mean(-1, keepdim=True) * quality
        if edges.numel():
            for source, target, projection in ((edges[0], edges[1], self.up), (edges[1], edges[0], self.down)):
                degree = torch.bincount(target, minlength=x.shape[2]).clamp_min(1)
                neighbor = x.index_select(2, source)
                gathered = projection(neighbor)
                if self.mode == "reliability":
                    edge_input = torch.cat((neighbor, x.index_select(2, target),
                                            evidence.index_select(2, source), evidence.index_select(2, target)), -1)
                    weight = self.edge_gate(edge_input).sigmoid() * confidence.index_select(2, source)
                    gathered = gathered * weight
                # Keep absolute confidence: normalizing by weight sum would
                # cancel quality when a node has only one neighbor.
                gathered = gathered / degree[target][None, None, :, None]
                out = out.index_add(2, target, gathered.to(out.dtype))
        if self.mode == "reliability":
            out = out * self.mix_gate(torch.cat((x, evidence), -1)).sigmoid()
        return out


class TrafficBlock(nn.Module):
    def __init__(self, width, heads, dropout, backend, graph, graph_mode="mean"):
        super().__init__()
        self.time_a = nn.MultiheadAttention(width, heads, dropout=dropout, batch_first=True)
        self.time_b = nn.MultiheadAttention(width, heads, dropout=dropout, batch_first=True)
        self.ffn_a = nn.Sequential(nn.Linear(width, 2 * width), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * width, width))
        self.ffn_b = nn.Sequential(nn.Linear(width, 2 * width), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * width, width))
        self.norms = nn.ModuleList(nn.LayerNorm(width) for _ in range(5))
        cls = (lambda: SpatialScan(width)) if backend == "scan" else (lambda: MambaSpatial(width, backend))
        self.forward_scan, self.reverse_scan = cls(), cls()
        self.graph = GraphResidual(width, graph_mode) if graph else None
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, segments, edges, bidirectional, observation=None):
        x = self.norms[0](x + self.dropout(FASTBlock.attend(x, self.time_a)))
        x = self.norms[1](x + self.dropout(self.ffn_a(x)))
        spatial = self.forward_scan(x, segments, False)
        if bidirectional:
            spatial = 0.5 * (spatial + self.reverse_scan(x, segments, True))
        if self.graph is not None:
            spatial = spatial + self.graph(x, edges, observation)
        x = self.norms[2](x + self.dropout(spatial))
        x = self.norms[3](x + self.dropout(FASTBlock.attend(x, self.time_b)))
        return self.norms[4](x + self.dropout(self.ffn_b(x)))


def haar_split(x):
    """One orthonormal Haar level on time, replicating an odd final sample."""
    if x.shape[1] % 2:
        x = torch.cat((x, x[:, -1:]), 1)
    even, odd = x[:, ::2], x[:, 1::2]
    return (even + odd) / 2 ** 0.5, (even - odd) / 2 ** 0.5


def haar_merge(low, high, steps):
    return torch.stack(((low + high) / 2 ** 0.5, (low - high) / 2 ** 0.5), 2).flatten(1, 2)[:, :steps]


class WaveletResidual(nn.Module):
    """Learn separate trend/detail filters, then reconstruct every time step."""
    def __init__(self, width, levels):
        super().__init__()
        self.levels = levels
        self.bands = nn.ModuleList(nn.Sequential(
            nn.Conv1d(width, width, 3, padding=1, groups=width), nn.GELU(), nn.Conv1d(width, width, 1)
        ) for _ in range(levels + 1))
        self.gain = nn.Parameter(torch.tensor(-2.0))

    @staticmethod
    def filter(x, band):
        batch, steps, nodes, width = x.shape
        y = band(x.permute(0, 2, 3, 1).reshape(batch * nodes, width, steps))
        return y.reshape(batch, nodes, width, steps).permute(0, 3, 1, 2)

    def forward(self, x):
        details, lengths = [], []
        low = x
        for _ in range(self.levels):
            lengths.append(low.shape[1])
            low, high = haar_split(low)
            details.append(high)
        result = self.filter(low, self.bands[0])
        for level in reversed(range(self.levels)):
            result = haar_merge(result, self.filter(details[level], self.bands[level + 1]), lengths[level])
        return self.gain.sigmoid() * result


class AdaptiveGraphResidual(nn.Module):
    """Panel-local learned dependencies, separate from physical road edges."""
    def __init__(self, width, neighbors):
        super().__init__()
        self.neighbors = neighbors
        rank = min(width, 16)
        self.source = nn.Linear(width, rank, bias=False)
        self.target = nn.Linear(width, rank, bias=False)
        self.project = nn.Linear(width, width, bias=False)
        self.gate = nn.Linear(width + 7, 1)
        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, -2)

    def layout(self, identities):
        count = identities.shape[0]
        # ponytail: dense scores fit current panels; use blocked top-k for larger networks.
        scores = self.target(identities) @ self.source(identities).T / self.source.out_features ** 0.5
        scores = scores.masked_fill(torch.eye(count, dtype=torch.bool, device=scores.device), -torch.inf)
        values, neighbors = scores.topk(min(self.neighbors, count - 1), dim=-1)
        return neighbors, values.softmax(-1)

    def forward(self, x, layout, observation):
        neighbors, weights = layout
        if neighbors.shape[1] == 0:
            return torch.zeros_like(x)
        evidence = observation[..., 10:17].clamp(0, 1).to(x.dtype)
        # Temporal inference remains usable when the selected node is currently missing.
        confidence = 0.25 + 0.75 * evidence[..., :3].mean(-1) * evidence[..., 3]
        message = torch.zeros_like(x)
        for neighbor in range(neighbors.shape[1]):
            source = neighbors[:, neighbor]
            weight = weights[:, neighbor].to(x.dtype)[None, None] * confidence.index_select(2, source)
            message = message + x.index_select(2, source) * weight[..., None]
        return self.gate(torch.cat((x, evidence), -1)).sigmoid() * self.project(message)


class TrafficEncoder(nn.Module):
    def __init__(self, features, node_count, width=64, blocks=2, heads=4,
                 dropout=0.1, backend="mamba2", graph=False, missing_aware=False, graph_mode="mean",
                 adaptive_graph_k=0, wavelet_levels=0, wavelet_input="embedded"):
        super().__init__()
        if width % heads or backend.startswith("mamba") and width % min(width, 32):
            raise ValueError("width must be divisible by attention heads and the Mamba head dimension")
        if graph_mode not in ("mean", "reliability") or graph_mode != "mean" and not graph:
            raise ValueError("Reliability graph mode requires graph=True")
        if (missing_aware or graph_mode == "reliability") and features < 17:
            raise ValueError("Missing-aware features require channel masks, quality, and ages")
        if adaptive_graph_k < 0 or adaptive_graph_k == 1 or not 0 <= wavelet_levels <= 3:
            raise ValueError("Adaptive graph neighbors must be 0 or at least 2; wavelet levels must lie in [0, 3]")
        if (adaptive_graph_k or wavelet_levels) and features < 31:
            raise ValueError("Adaptive graph and wavelet require the v2 observation/prior features")
        if wavelet_input not in ("embedded", "signals"):
            raise ValueError("Unknown wavelet input")
        # Configs without this field retain the original checkpoint architecture.
        self.wavelet_input = wavelet_input
        self.input = nn.Linear(features, width)
        self.node = nn.Embedding(node_count + 1, width, padding_idx=0)
        self.slot = nn.Embedding(288, width)
        self.weekday = nn.Embedding(7, width)
        self.relative = nn.Embedding(72, width)
        self.blocks = nn.ModuleList(TrafficBlock(width, heads, dropout, backend, graph, graph_mode) for _ in range(blocks))
        self.skip = nn.Linear(width * blocks, width)
        self.dropout = nn.Dropout(dropout)
        self.missing_embedding = nn.Parameter(torch.empty(3, width)) if missing_aware else None
        if self.missing_embedding is not None:
            nn.init.normal_(self.missing_embedding, std=0.02)
        self.adaptive_graph = AdaptiveGraphResidual(width, adaptive_graph_k) if adaptive_graph_k else None
        self.wavelet = WaveletResidual(3 if wavelet_input == "signals" else width, wavelet_levels) if wavelet_levels else None
        self.wavelet_project = nn.Linear(3, width, bias=False) if wavelet_levels and wavelet_input == "signals" else None

    def project_input(self, x, fill_missing=False):
        if self.missing_embedding is None and self.wavelet is None:
            return self.input(x)
        masks = x[..., 10:13].clamp(0, 1)
        clean = x.clone()
        for channel, position in enumerate((0, 1, 9)):
            fallback = x[..., 25 + channel] if fill_missing else 0
            clean[..., position] = torch.where(masks[..., channel] > 0, x[..., position], fallback)
        # Separate trainable tokens for missing speed, flow, and occupancy.
        result = self.input(clean)
        return result if self.missing_embedding is None else result + (1 - masks) @ self.missing_embedding

    def forward(self, x, node_ids, segments, edges, bidirectional=True):
        steps = x.shape[1]
        if steps > 72:
            raise ValueError("The input window must contain at most 72 steps")
        slot = (x[..., 28] * 287).round().long().clamp(0, 287)
        weekday = (x[..., 29] * 6).round().long().clamp(0, 6)
        projected = self.project_input(x)
        identities = self.node(node_ids)
        embedding = (identities[None, None].to(projected.dtype) + self.slot(slot).to(projected.dtype)
                     + self.weekday(weekday).to(projected.dtype)
                     + self.relative(torch.arange(steps, device=x.device))[None, :, None].to(projected.dtype))
        hidden = self.dropout(projected + embedding)
        if self.wavelet is not None:
            if self.wavelet_input == "signals":
                # DWT sees only continuous measurements; masks and embeddings stay in the main path.
                signals = torch.where(x[..., 10:13] > 0, x[..., (0, 1, 9)], x[..., 25:28])
                hidden = hidden + self.wavelet_project(self.wavelet(signals))
            else:
                hidden = hidden + self.wavelet(self.project_input(x, fill_missing=True) + embedding)
        layout = self.adaptive_graph.layout(identities) if self.adaptive_graph is not None else None
        outputs = []
        for block in self.blocks:
            hidden = block(hidden, segments, edges, bidirectional, x)
            if self.adaptive_graph is not None:
                hidden = hidden + self.adaptive_graph(hidden, layout, x)
            outputs.append(hidden)
        return self.skip(torch.cat(outputs, -1))


class TrafficStateModel(nn.Module):
    def __init__(self, station_decoder="linear", **config):
        super().__init__()
        if station_decoder not in ("linear", "gated", "split"):
            raise ValueError("Unknown station decoder")
        self.station_decoder = station_decoder
        self.encoder = TrafficEncoder(**config)
        width = config.get("width", 64)
        if station_decoder == "split":
            self.head = nn.ModuleList(nn.Sequential(nn.Linear(width + len(STATE_CONTEXT_NAMES), width),
                                                   nn.GELU(), nn.Linear(width, 1)) for _ in range(2))
            self.gate = nn.ModuleList(nn.Linear(width + len(STATE_CONTEXT_NAMES), 1) for _ in range(2))
            for head, gate in zip(self.head, self.gate):
                nn.init.zeros_(head[-1].weight)
                nn.init.zeros_(head[-1].bias)
                nn.init.zeros_(gate.weight)
                nn.init.constant_(gate.bias, -1)
        elif station_decoder == "gated":
            self.head = nn.Sequential(nn.Linear(width + len(STATE_CONTEXT_NAMES), width), nn.GELU(), nn.Linear(width, 2))
            self.gate = nn.Linear(width + len(STATE_CONTEXT_NAMES), 2)
            nn.init.zeros_(self.gate.weight)
            nn.init.constant_(self.gate.bias, -1)
            nn.init.zeros_(self.head[-1].weight)
            nn.init.zeros_(self.head[-1].bias)
        else:
            self.head = nn.Linear(width + 3, 2)
            nn.init.zeros_(self.head.weight)
            nn.init.zeros_(self.head.bias)

    def forward(self, x, context, station_link, node_ids, segments, edges, bidirectional=True):
        hidden = self.encoder(x, node_ids, segments, edges, bidirectional)[:, :, station_link]
        expected = len(STATE_CONTEXT_NAMES) if self.station_decoder != "linear" else 3
        if context.shape[-1] != expected:
            raise ValueError(f"Station decoder requires {expected} context features")
        combined = torch.cat((hidden, context), -1)
        if self.station_decoder == "split":
            correction = torch.cat([head(combined) * gate(combined).sigmoid()
                                    for head, gate in zip(self.head, self.gate)], -1)
        else:
            correction = self.head(combined)
            if self.station_decoder == "gated":
                correction = correction * self.gate(combined).sigmoid()
        return F.relu(context[..., :2] + correction)


class TrafficQueueModel(nn.Module):
    def __init__(self, station_decoder="linear", queue_head="linear", queue_changes=False, **config):
        super().__init__()
        if queue_head not in ("linear", "pooled-mlp"):
            raise ValueError("Unknown Queue head")
        self.queue_head = queue_head
        self.encoder = TrafficEncoder(**config)
        width = config.get("width", 64)
        self.head = (nn.Linear(width, 12) if queue_head == "linear" else
                     nn.Sequential(nn.Linear(2 * width, width), nn.GELU(),
                                   nn.Dropout(config.get("dropout", 0.1)), nn.Linear(width, 12)))

    def forward(self, x, node_ids, segments, edges, bidirectional=True):
        history = self.encoder(x, node_ids, segments, edges, bidirectional)
        hidden = history[:, -1]
        if self.queue_head == "pooled-mlp":
            hidden = torch.cat((hidden, history.mean(1)), dim=-1)
        output = self.head(hidden).reshape(hidden.shape[0], hidden.shape[1], 6, 2).transpose(1, 2)
        return output[..., 0], F.softplus(output[..., 1])
''')


내장 코드 로드: fast_model (466줄)


In [21]:
# src/learning.py — 원본 cdf888214fe8a54b73302f3988b764b74b3f30a4
load_source('learning', r'''"""Versioned training, exact-template State validation, and Queue proxy validation."""
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
import hashlib
import itertools
import json
from pathlib import Path
import re
import sys
import time

import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F

from data import (FEATURE_NAMES, STATE_CONTEXT_NAMES, PanelInfo, StationDay, TargetIndex, aggregate_channels,
                  features_v2, queue_features, queue_feature_names, ramp_features, slice_day, state_features_v2)
from fast_model import TrafficQueueModel, TrafficStateModel


def stable_seed(*parts):
    return int.from_bytes(hashlib.blake2b("|".join(map(str, parts)).encode(), digest_size=8).digest(), "big")


@lru_cache(maxsize=64)
def file_index(release: Path, name: str):
    result = {}
    for path in sorted((release / "corridors" / name / "train" / "mainline_states").glob("**/*.parquet")):
        match = re.search(r"\d{4}[-_]\d{2}[-_]\d{2}", path.name)
        date = match.group().replace("_", "-") if match else str(pd.read_parquet(path, columns=["timestamp"]).timestamp.iloc[0])[:10]
        if date in result:
            raise ValueError(f"Duplicate day: {name} {date}")
        result[date] = path
    return result


def period_files(release, name, period, maximum=None):
    limits = {"fit": ("2030-06-01", "2030-12-31"), "select": ("2031-01-01", "2031-01-31"),
              "holdout": ("2031-02-01", "2031-02-28")}
    low, high = limits[period]
    paths = [path for date, path in file_index(release, name).items() if low <= date <= high]
    return paths[:maximum] if maximum else paths


def signature(paths):
    return hashlib.sha256(json.dumps([(p.name, p.stat().st_size, p.stat().st_mtime_ns) for p in paths]).encode()).hexdigest()


def fit_profile(release, panel, paths):
    """Fit-only weekday/slot means; the checkpoint embeds the resulting array."""
    key = hashlib.sha256(("eligible-v2|" + signature(paths) + json.dumps(panel.links)).encode()).hexdigest()[:16]
    folder = release.parent / "processed_v2"
    folder.mkdir(parents=True, exist_ok=True)
    cache = folder / f"{panel.name}-profile-{key}.npy"
    if cache.exists():
        return np.load(cache, allow_pickle=False)
    print(f"Preparing fit-only means: {panel.name} days={len(paths)}", flush=True)
    sums = np.zeros((2016, len(panel.links), 3), np.float64)
    counts = np.zeros_like(sums)
    for number, path in enumerate(paths, 1):
        day = StationDay.load(path, panel)
        values, valid, _ = aggregate_channels(day, panel, day.eligible)
        dates = pd.to_datetime(day.timestamps, utc=True)
        slots = np.asarray(dates.dayofweek * 288 + dates.hour * 12 + dates.minute // 5)
        np.add.at(sums, slots, values * valid)
        np.add.at(counts, slots, valid)
        if number % 50 == 0:
            print(f"fit-only means {panel.name}: {number}/{len(paths)}", flush=True)
    global_mean = sums.sum((0, 1)) / np.maximum(counts.sum((0, 1)), 1)
    link_mean = np.divide(sums.sum(0), counts.sum(0), out=np.broadcast_to(global_mean, sums.shape[1:]).copy(), where=counts.sum(0) > 0)
    result = np.divide(sums, counts, out=np.broadcast_to(link_mean, sums.shape).copy(), where=counts > 0).astype(np.float32)
    temporary = cache.with_suffix(".pending.npy")
    np.save(temporary, result)
    temporary.replace(cache)
    return result


def center_windows(steps):
    for start in range(0, steps, 12):
        yield np.clip(np.arange(start - 30, start + 42), 0, steps - 1), slice(start, min(start + 12, steps))


def state_examples(context, path):
    panel, release = context["panel"], context["release"]
    truth = StationDay.load(path, panel)
    match = re.search(r"\d{4}[-_]\d{2}[-_]\d{2}", path.name)
    if match and truth.timestamps[0][:10] != match.group().replace("_", "-"):
        raise ValueError(f"File date does not match observations: {path}")
    regime, targets = context["targets"].get(truth)
    masked = StationDay.load(context["masked"][path.name], panel)
    if masked.timestamps != truth.timestamps or masked.station_ids != truth.station_ids:
        raise ValueError(f"Masked/unmasked mapping differs: {path}")
    if np.isfinite(masked.speed[targets]).any() or np.isfinite(masked.flow_lane[targets]).any():
        raise ValueError("A template answer is visible in the masked input")
    ramp = ramp_features(release, panel, "train", truth.timestamps)
    samples = []
    for index, absolute in center_windows(len(truth.timestamps)):
        if not targets[absolute].any():
            continue
        x, station = state_features_v2(masked, panel, index, context["profile"], ramp,
                                      rich=context.get("station_decoder", "linear") != "linear")
        length = absolute.stop - absolute.start
        y = np.zeros((12, len(truth.station_ids), 2), np.float32)
        mask = np.zeros(y.shape[:2], bool)
        eligible = np.zeros_like(mask)
        observed = np.zeros_like(mask)
        y[:length] = np.stack((truth.speed[absolute], truth.flow_lane[absolute]), -1)
        mask[:length] = targets[absolute]
        eligible[:length] = truth.eligible[absolute]
        observed[:length] = np.isfinite(masked.speed[absolute]) & np.isfinite(masked.flow_lane[absolute])
        slots = pd.to_datetime(truth.timestamps[absolute], utc=True)
        slot_ids = np.asarray(slots.dayofweek * 288 + slots.hour * 12 + slots.minute // 5)
        prior = np.zeros_like(y)
        prior[:length] = context["profile"][slot_ids][:, truth.station_to_link, :2]
        samples.append({"x": x, "context": station, "truth": y, "mask": mask,
                        "eligible": eligible, "observed": observed, "baseline": station[30:42, :, :2],
                        "prior": prior, "date": truth.timestamps[0][:10], "regime": regime,
                        "station_link": truth.station_to_link, "lanes": truth.lanes})
    return samples


@lru_cache(maxsize=32)
def masked_file_index(release, name, split):
    folder = release / "corridors" / name / split / "mainline_states_masked"
    if not folder.is_dir():
        raise FileNotFoundError(f"Released Queue origin needs masked inputs: {folder}")
    result = {}
    for path in sorted(folder.glob("**/*.parquet")):
        match = re.search(r"\d{4}[-_]\d{2}[-_]\d{2}", path.name)
        if not match:
            raise ValueError(f"Masked file has no date: {path}")
        date = match.group().replace("_", "-")
        if date in result:
            raise ValueError(f"Duplicate masked day: {name} {split} {date}")
        result[date] = path
    return result


def released_origin_features(context, timestamp, ramp, cache):
    """Read exactly T from the released masked layer, never fill it from truth."""
    panel, date = context["panel"], timestamp[:10]
    missing = np.full((1, len(panel.links)), np.nan, np.float32)
    origin = StationDay([timestamp], panel.links, panel.links, np.arange(len(panel.links)), panel.lanes,
                        missing, missing.copy(), np.zeros_like(missing, bool), missing.copy(), np.zeros_like(missing))
    path = context["origin_paths"].get(date)
    if path is not None:
        if date not in cache:
            cache[date] = StationDay.load(path, panel)
        day = cache[date]
        if timestamp in day.timestamps:
            origin = slice_day(day, np.array([day.timestamps.index(timestamp)]))
    return features_v2(origin, panel, context["profile"], ramp)


def queue_examples(context, path, args, period, epoch=0):
    panel = context["panel"]
    day = StationDay.load(path, panel)
    steps, nodes = len(day.timestamps), len(panel.links)
    values, _, _ = aggregate_channels(day, panel, day.eligible)
    speed = values[..., 0]
    minimum = np.full((steps, nodes), np.inf, np.float32)
    valid = np.zeros_like(minimum, bool)
    for station, link in enumerate(day.station_to_link):
        good = day.eligible[:, station]
        minimum[:, link] = np.minimum(minimum[:, link], np.where(good, day.speed[:, station] * 25, np.inf))
        valid[:, link] |= good
    queues = (minimum <= panel.cutoffs) & valid  # Any eligible station, not mean speed.
    candidates = {"onset": [], "ongoing": [], "normal": []}
    for origin in range(12, steps - 6):
        history, future = queues[origin - 12:origin], queues[origin + 1:origin + 7]
        if valid[origin - 12:origin].mean() < args.history_coverage or valid[origin + 1:origin + 7].mean() < args.future_coverage:
            continue
        kind = ("onset" if not history.any() else "ongoing" if history[-1].any() else None) if future.any() else "normal"
        if kind is not None:
            candidates[kind].append(origin)
    seed_parts = (args.seed, panel.name, path.name, "queue")
    if period == "fit" and getattr(args, "queue_resample", False):
        seed_parts += (epoch,)
    rng = np.random.default_rng(stable_seed(*seed_parts))
    condition_gap = getattr(args, "queue_condition_gap", False) if period == "fit" else getattr(args, "queue_validation_spread", False)
    chosen = []
    for kind in ("onset", "ongoing", "normal"):
        limit = args.normal_windows if kind == "normal" else args.windows_per_day
        for origin in rng.permutation(candidates[kind]):
            if sum(k == kind for k, _ in chosen) >= limit:
                break
            if all(abs(int(origin) - earlier) * 5 >= args.window_gap_minutes
                   for other_kind, earlier in chosen if not condition_gap or other_kind == kind):
                chosen.append((kind, int(origin)))
    ramp = ramp_features(context["release"], panel, "train", day.timestamps)
    samples, origin_cache = [], {}
    for kind, origin in chosen:
        index = np.arange(origin - 12, origin)
        future = slice(origin + 1, origin + 7)
        released = (released_origin_features(context, day.timestamps[origin], ramp[origin:origin + 1], origin_cache)
                    if getattr(args, "queue_origin", False) else None)
        x = queue_features(slice_day(day, index), panel, context["profile"], ramp[index], released,
                           getattr(args, "queue_changes", False))
        hard = queues[future].astype(np.float32)
        margin = panel.cutoffs[None] - minimum[future]
        soft = 1 / (1 + np.exp(-np.clip(margin / max(args.label_temperature, 1e-6), -30, 30))) if args.label_temperature > 0 else hard
        weight = np.minimum(np.abs(margin) / args.uncertainty_margin, 1) if args.uncertainty_margin > 0 else np.ones_like(hard)
        weight = np.where(valid[future], np.maximum(weight, 0.1), 0).astype(np.float32)
        samples.append({"x": x, "truth": hard, "soft": soft.astype(np.float32), "speed": speed[future],
                        "mask": valid[future], "weight": weight, "baseline": np.broadcast_to(queues[origin - 1], hard.shape).copy(),
                        "condition": kind, "date": day.timestamps[0][:10], "origin": origin})
    return samples


def tensor_batch(samples, keys, device):
    out = {}
    for key in keys:
        value = torch.from_numpy(np.stack([sample[key] for sample in samples]))
        out[key] = value.pin_memory().to(device, non_blocking=True) if device == "cuda" else value
    return out


def physics_loss(prediction, truth, mask, eligible, observed, station_link, station_lanes, panel):
    """Supervise vehicle counts/differences on completed, observable link fields."""
    complete = torch.where(mask[..., None], prediction, torch.nan_to_num(truth))
    active = eligible & (observed | mask)
    count = torch.zeros((*active.shape[:2], len(panel.links)), device=prediction.device)
    count = count.index_add(2, station_link, active.float())
    def links(field):
        return torch.zeros_like(count).index_add(2, station_link, field * active) / count.clamp_min(1)
    v = links(complete[..., 0] * 25)
    q = links(complete[..., 1] * 600 * station_lanes[None, None])
    vt = links(torch.nan_to_num(truth[..., 0]) * 25)
    qt = links(torch.nan_to_num(truth[..., 1]) * 600 * station_lanes[None, None])
    device = prediction.device
    length = torch.as_tensor(panel.lengths, device=device)
    free = torch.as_tensor(panel.free_speed, device=device)
    capacity = torch.as_tensor(panel.capacity if panel.capacity is not None else np.full(len(panel.links), 6000), device=device)
    jam = torch.as_tensor(panel.jam_density if panel.jam_density is not None else np.full(len(panel.links), 400), device=device)
    scale = (capacity / free.clamp_min(1) * length).clamp_min(1)
    n = q / v.clamp_min(1) * length
    nt = qt / vt.clamp_min(1) * length
    good = count > 0
    loss_n = (((n - nt) / scale).square() * good).sum() / good.sum().clamp_min(1)
    pairs = good[:, 1:] & good[:, :-1]
    loss_dn = ((((n[:, 1:] - n[:, :-1]) - (nt[:, 1:] - nt[:, :-1])) / scale).square() * pairs).sum() / pairs.sum().clamp_min(1)
    density = q / v.clamp_min(1)
    wave = capacity / (jam - capacity / free.clamp_min(1)).clamp_min(1)
    fd_flow = torch.minimum(free * density, wave * (jam - density)).clamp_min(0)
    loss_fd = (((q - fd_flow) / capacity.clamp_min(1)).square() * good).sum() / good.sum().clamp_min(1)
    return loss_n, loss_dn, loss_fd


def state_loss(prediction, batch, mapping, lanes, panel, args, normalizer=None, aux_weight=1.0):
    squared = (prediction.float() - torch.nan_to_num(batch["truth"])).square()
    mask = batch["mask"]
    weights = batch.get("regime_weight")
    denominator = (len(prediction) if weights is not None else mask.sum().clamp_min(1)) if normalizer is None else max(normalizer, 1)
    error = (0.54 * squared[..., 0] + 0.46 * squared[..., 1]) * mask
    loss = (error if weights is None else error * weights[:, None, None]).sum() / denominator
    if args.lambda_n or args.lambda_dn or args.lambda_fd:
        n, dn, fd = physics_loss(prediction.float(), batch["truth"], mask, batch["eligible"], batch["observed"], mapping, lanes, panel)
        loss = loss + aux_weight * (args.lambda_n * n + args.lambda_dn * dn + args.lambda_fd * fd)
    return loss


def queue_loss(logits, speeds, batch, conditions, args, condition_counts=None):
    logits, speeds = logits.float(), speeds.float()
    weights = batch["weight"] * batch["mask"]
    bce = (F.binary_cross_entropy_with_logits(logits, batch["soft"], reduction="none") * weights).sum((1, 2)) / weights.sum((1, 2)).clamp_min(1)
    probability = logits.sigmoid() * batch["mask"]
    truth = batch["truth"] * batch["mask"]
    intersection = (probability * truth).sum((1, 2))
    union = (probability + truth - probability * truth).sum((1, 2))
    soft_iou = (1 - (intersection + 1e-6) / (union + 1e-6)) * (truth.sum((1, 2)) > 0)
    speed = (F.smooth_l1_loss(speeds, batch["speed"], reduction="none") * batch["mask"]).sum((1, 2)) / batch["mask"].sum((1, 2)).clamp_min(1)
    losses = bce + args.soft_iou_weight * soft_iou + args.speed_weight * speed
    groups = []
    for kind in ("onset", "ongoing", "normal"):
        indices = [i for i, condition in enumerate(conditions) if condition == kind]
        if indices:
            value = losses[indices].mean() if condition_counts is None else losses[indices].sum() / condition_counts[kind]
            groups.append(value * (args.normal_weight if kind == "normal" else 1.0))
    return torch.stack(groups).mean() if condition_counts is None else torch.stack(groups).sum() / len(condition_counts)


def family_mean(scores):
    families = {}
    for panel, score in scores.items():
        families.setdefault(panel.rsplit("_", 1)[0], []).append(score)
    return float(np.mean([np.mean(values) for values in families.values()])) if families else 0.0


def state_metrics(records, method="model"):
    groups = {}
    for row in records:
        key = row["panel"], row["regime"]
        group = groups.setdefault(key, np.zeros(3, np.float64))
        group += [row[f"{method}_speed_sse"], row[f"{method}_flow_sse"], row["count"]]
    detail = []
    panels = {}
    for (panel, regime), (sv, sq, count) in groups.items():
        rv, rq = np.sqrt(sv / max(count, 1)), np.sqrt(sq / max(count, 1))
        score = float(0.54 * max(0, 1 - rv) + 0.46 * max(0, 1 - rq)) if count else 0.0
        panels.setdefault(panel, {})[regime] = score
        detail.append({"panel": panel, "regime": regime, "count": int(count), "rmse_speed_kmh": float(rv * 25),
                       "rmse_flow_vph_per_lane": float(rq * 600), "State": score})
    # An absent regime has no evidence and receives zero, matching the evaluator.
    scores = {panel: float(np.mean([values.get(r, 0) for r in ("R1", "R2", "R3")])) for panel, values in panels.items()}
    return {"State": family_mean(scores), "panels": scores, "regimes": detail}


def iou(predicted, truth, mask):
    predicted, truth = predicted & mask, truth.astype(bool) & mask
    union = np.logical_or(predicted, truth).sum()
    return float(np.logical_and(predicted, truth).sum() / union) if union else 0.0


def queue_metrics(records, threshold, baseline=False):
    groups = {}
    normal = []
    for row in records:
        score = row["persistence_iou"] if baseline else row["ious"][str(threshold)]
        if row["condition"] == "normal":
            normal.append(row["persistence_false_positive"] if baseline else row["normal_false_positive"][str(threshold)])
        else:
            groups.setdefault((row["panel"], row["condition"]), []).append(score)
    panels = {row["panel"] for row in records}
    by_condition = {kind: {panel: float(np.mean(groups.get((panel, kind), [0]))) for panel in panels} for kind in ("onset", "ongoing")}
    scores = {panel: 0.5 * (by_condition["onset"][panel] + by_condition["ongoing"][panel]) for panel in panels}
    return {"proxy_IoU": family_mean(scores), "onset": family_mean(by_condition["onset"]),
            "ongoing": family_mean(by_condition["ongoing"]), "panels": scores,
            "condition_counts": {kind: sum(row["condition"] == kind for row in records) for kind in ("onset", "ongoing", "normal")},
            "normal_false_positive": float(np.mean(normal)) if normal else None}


def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_suffix(path.suffix + ".pending")
    pending.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    pending.replace(path)


def event(path, **values):
    values = {"time_utc": pd.Timestamp.now(tz="UTC").isoformat(), **values}
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(values, ensure_ascii=False, allow_nan=False) + "\n")
    print(json.dumps(values, ensure_ascii=False), flush=True)


def model_from_checkpoint(checkpoint, device):
    expected = queue_feature_names(checkpoint["model_config"].get("queue_changes", False)) if checkpoint["task"] == "queue" else FEATURE_NAMES
    if checkpoint.get("format_version") != 2 or tuple(checkpoint["feature_names"]) != expected:
        raise ValueError("Expected a v2 checkpoint with the current feature contract")
    cls = TrafficStateModel if checkpoint["task"] == "state" else TrafficQueueModel
    if checkpoint["task"] == "state" and checkpoint["model_config"].get("station_decoder", "linear") != "linear":
        if tuple(checkpoint.get("state_context_names", ())) != STATE_CONTEXT_NAMES:
            raise ValueError("State checkpoint has an incompatible station context contract")
    model = cls(**checkpoint["model_config"]).to(device)
    model.load_state_dict(checkpoint["weights"])
    return model


def check_model_requests(args, config, queue=True):
    for key in ("graph", "missing_aware"):
        if getattr(args, key, False) and not config.get(key, False):
            raise ValueError(f"The checkpoint has {key}=False; use a new checkpoint to change the model")
    if args.graph_mode != "mean" and args.graph_mode != config.get("graph_mode", "mean"):
        raise ValueError("The checkpoint has a different graph mode; use a new checkpoint to change the model")
    for key, default in (("station_decoder", "linear"), ("adaptive_graph_k", 0), ("wavelet_levels", 0)):
        requested = getattr(args, key, default)
        if requested != default and requested != config.get(key, default):
            raise ValueError(f"The checkpoint has a different {key}; use a new checkpoint to change the model")
    if getattr(args, "wavelet_input", None) is not None and args.wavelet_input != config.get("wavelet_input", "embedded"):
        raise ValueError("The checkpoint has a different wavelet_input; use a new checkpoint to change the model")
    if queue:
        for key, default in (("queue_head", "linear"), ("queue_changes", False)):
            requested = getattr(args, key, default)
            if requested != default and requested != config.get(key, default):
                raise ValueError(f"The checkpoint has a different {key}; use a new checkpoint to change the model")


def fit_regime_weights(item):
    """Weight SSE so its window mean equals the fit-only macro regime MSE."""
    targets = item["targets"]
    counts, windows = {}, 0
    fit_paths = set(item["fit_paths"])
    for date, path in file_index(item["release"], item["panel"].name).items():
        if path not in fit_paths:
            continue
        regime = targets.regimes[date]
        cells = targets.archive[date]
        counts[regime] = counts.get(regime, 0) + len(cells)
        windows += len(np.unique((cells // len(targets.stations)) // 12))
    counts = {regime: count for regime, count in counts.items() if count}
    if not counts or set(counts) - {"R1", "R2", "R3"}:
        raise ValueError("Regime balancing requires nonempty R1/R2/R3 fit targets")
    return {regime: windows / (len(counts) * count) for regime, count in counts.items()}


def contexts(args, task, checkpoint=None):
    release = args.release_root.resolve()
    available = sorted(path.name for path in (release / "corridors").iterdir() if path.is_dir())
    names = sorted(set(args.panel or (checkpoint["trained_panels"] if checkpoint else available)))
    if set(names) - set(available):
        raise ValueError(f"Unknown panels: {set(names) - set(available)}")
    if task == "queue":
        names = [name for name in names if name not in {"D12_I405_N", "D12_I405_S"}]
    if not names:
        raise ValueError("No panels for this task")
    result = {}
    for name in names:
        panel = PanelInfo.load(release, name)
        fit_paths = period_files(release, name, "fit", args.max_days)
        if not fit_paths:
            raise ValueError(f"No fit days for {name}")
        if checkpoint:
            if name not in checkpoint["profiles"]:
                raise ValueError(f"Checkpoint has no fit-only profile for {name}")
            profile = checkpoint["profiles"][name].cpu().numpy()
            if checkpoint["links"][name] != panel.links:
                raise ValueError(f"Road layout changed: {name}")
        else:
            profile = fit_profile(release, panel, fit_paths)
        context = {"panel": panel, "release": release, "profile": profile, "fit_paths": fit_paths}
        if task == "queue" and getattr(args, "queue_origin", False):
            context["origin_paths"] = masked_file_index(release, name, "train")
            fit_set = set(fit_paths)
            context["origin_fit_paths"] = [context["origin_paths"][date] for date, path in sorted(file_index(release, name).items())
                                           if path in fit_set and date in context["origin_paths"]]
        if task == "state":
            first = StationDay.load(fit_paths[0], panel)
            context["targets"] = TargetIndex(release, name, first.station_ids)
            context["masked"] = {path.name: path for path in (release / "corridors" / name / "train" / "mainline_states_masked").glob("**/*.parquet")}
            config = checkpoint["model_config"] if checkpoint else vars(args)
            context["station_decoder"] = config.get("station_decoder", "linear")
            if getattr(args, "state_regime_balance", False):
                context["regime_weights"] = fit_regime_weights(context)
        result[name] = context
    return result


def schedule(context, period, args, epoch):
    rng = np.random.default_rng(stable_seed(args.seed, epoch, period))
    names = list(context)
    if period == "fit":
        rng.shuffle(names)
    paths = []
    for name in names:
        days = period_files(args.release_root, name, period, args.max_days)
        if period == "fit":
            rng.shuffle(days)
        elif getattr(args, "queue_validation_spread", False) and days:
            # Visit evenly spaced dates first; run_period takes at most one window per condition/day.
            first = np.linspace(0, len(days) - 1, min(len(days), args.validation_windows)).round().astype(int).tolist()
            chosen = set(first)
            days = [days[i] for i in first] + [path for i, path in enumerate(days) if i not in chosen]
        paths.append([(name, path) for path in days])
    return [item for row in itertools.zip_longest(*paths) for item in row if item is not None]


def prepared_days(context, items, args, task, period, epoch=0):
    def prepare(item):
        name, path = item
        start = time.perf_counter()
        samples = state_examples(context[name], path) if task == "state" else queue_examples(context[name], path, args, period, epoch)
        return name, path, samples, time.perf_counter() - start
    if args.workers == 0:
        for item in items:
            yield prepare(item)
    else:
        # Bounded queue: do not retain the whole dataset in WSL's small RAM.
        with ThreadPoolExecutor(max_workers=args.workers) as pool:
            iterator = iter(items)
            pending = [pool.submit(prepare, item) for item in itertools.islice(iterator, args.workers + 1)]
            while pending:
                future = pending.pop(0)
                result = future.result()
                item = next(iterator, None)
                if item is not None:
                    pending.append(pool.submit(prepare, item))
                yield result


def batch_size(args, name, nodes, task):
    overrides = dict(value.split("=", 1) for value in args.panel_batch)
    if name in overrides:
        size = int(overrides[name])
    elif args.batch_size:
        size = args.batch_size
    else:
        size = (8 if nodes <= 128 else 2) if task == "state" else (48 if nodes <= 128 else 12)
    if size < 1:
        raise ValueError("Batch size must be positive")
    return size


def update_size(args, name, nodes, task):
    requested = getattr(args, "update_windows", None)
    if requested is None:
        return batch_size(args, name, nodes, task)
    if requested < 0:
        raise ValueError("--update-windows must be nonnegative")
    return requested or (8 if task == "state" else 48)


def device_layout(context, vocabulary, device):
    for name, item in context.items():
        panel = item["panel"]
        item["node_ids"] = torch.tensor([vocabulary.get(f"{name}|{link}", 0) for link in panel.links], device=device)
        item["edges"] = torch.as_tensor(panel.edges, device=device)


def run_period(model, context, vocabulary, args, task, period, epoch, device, optimizer=None, log=None, global_step=0):
    training = optimizer is not None
    model.train(training)
    records = []
    pending = {name: [] for name in context}
    totals = torch.zeros(2, device=device)
    physics_totals = torch.zeros(4, device=device)
    seen_conditions = {name: {kind: 0 for kind in ("onset", "ongoing", "normal")} for name in context}
    thresholds = tuple(float(t) for t in (0.3, 0.4, 0.5, 0.6))
    if period == "holdout":
        thresholds = (args.fixed_threshold,)
    started = time.perf_counter()
    preparation_seconds = 0.0
    windows = 0
    updates_by_panel = {name: 0 for name in context}
    device_layout(context, vocabulary, device)

    def process_batch(name, samples, normalizer=None, condition_counts=None, aux_weight=1.0):
        nonlocal windows
        panel = context[name]["panel"]
        if log and windows == 0:
            event(log, event="first_batch", epoch=epoch + 1, period=period, panel=name,
                  batch_size=len(samples), backend=args.model_config["backend"],
                  note="The first CUDA call may compile Triton kernels")
        if task == "state":
            mapping = torch.as_tensor(samples[0]["station_link"], device=device)
            lanes = torch.as_tensor(samples[0]["lanes"], device=device)
            if any(not np.array_equal(sample["station_link"], samples[0]["station_link"]) for sample in samples):
                raise ValueError("Station mapping differs inside a batch")
            batch = tensor_batch(samples, ("x", "context", "truth", "mask", "eligible", "observed"), device)
            if training and getattr(args, "state_regime_balance", False):
                batch["regime_weight"] = torch.tensor([context[name]["regime_weights"][s["regime"]]
                                                      for s in samples], device=device)
        else:
            batch = tensor_batch(samples, ("x", "truth", "soft", "speed", "mask", "weight"), device)
        with torch.set_grad_enabled(training):
            with torch.autocast(device, dtype=torch.bfloat16, enabled=args.use_bf16):
                if task == "state":
                    prediction = model(batch["x"], batch["context"], mapping, context[name]["node_ids"], panel.segments, context[name]["edges"], args.bidirectional)[:, 30:42]
                else:
                    logits, speeds = model(batch["x"], context[name]["node_ids"], panel.segments, context[name]["edges"], args.bidirectional)
            loss = (state_loss(prediction, batch, mapping, lanes, panel, args, normalizer, aux_weight)
                    if task == "state" else queue_loss(logits, speeds, batch, [s["condition"] for s in samples], args, condition_counts))
            if training:
                loss.backward()
        windows += len(samples)
        if not training:
            if task == "state":
                physics = physics_loss(prediction.float(), batch["truth"], batch["mask"], batch["eligible"], batch["observed"], mapping, lanes, panel)
                physics_totals[:3] += torch.stack(physics).detach() * len(samples)
                physics_totals[3] += len(samples)
                outputs = prediction.float().cpu().numpy()
                for sample, output in zip(samples, outputs):
                    row = {"panel": name, "date": sample["date"], "regime": sample["regime"], "count": int(sample["mask"].sum())}
                    for method, estimate in (("model", output), ("interpolation", sample["baseline"]), ("historical", sample["prior"])):
                        squared = (estimate - np.nan_to_num(sample["truth"])) ** 2
                        row[f"{method}_speed_sse"] = float(squared[..., 0][sample["mask"]].sum())
                        row[f"{method}_flow_sse"] = float(squared[..., 1][sample["mask"]].sum())
                    records.append(row)
            else:
                probabilities, future_speed = logits.float().sigmoid().cpu().numpy(), speeds.float().cpu().numpy()
                for sample, probability, predicted_speed in zip(samples, probabilities, future_speed):
                    record = {"panel": name, "date": sample["date"], "condition": sample["condition"], "origin": sample["origin"],
                              "persistence_iou": iou(sample["baseline"], sample["truth"], sample["mask"]),
                              "persistence_false_positive": float((sample["baseline"] & sample["mask"]).sum() / max(sample["mask"].sum(), 1)),
                              "ious": {str(t): iou(probability >= t, sample["truth"], sample["mask"]) for t in thresholds},
                              "normal_false_positive": {str(t): float(((probability >= t) & sample["mask"]).sum() / max(sample["mask"].sum(), 1)) for t in thresholds},
                              "speed_sse_kmh": float((((predicted_speed - sample["speed"]) * 25) ** 2)[sample["mask"]].sum()),
                              "count": int(sample["mask"].sum())}
                    records.append(record)
        return loss.detach()

    def process(name, samples):
        nonlocal global_step
        count = len(samples)
        size = batch_size(args, name, len(context[name]["panel"].links), task) if training else count
        accumulated = training and getattr(args, "update_windows", None) is not None
        normalizer = ((count if getattr(args, "state_regime_balance", False) else
                       sum(np.count_nonzero(sample["mask"]) for sample in samples))
                      if accumulated and task == "state" else None)
        conditions = {kind: sum(sample["condition"] == kind for sample in samples)
                      for kind in {sample["condition"] for sample in samples}} if accumulated and task == "queue" else None
        if training:
            optimizer.zero_grad(set_to_none=True)
        total_loss = torch.zeros((), device=device)
        for start in range(0, count, size):
            part = samples[start:start + size]
            total_loss += process_batch(name, part, normalizer, conditions, len(part) / count)
        if training:
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()
            global_step += 1
            updates_by_panel[name] += 1
        totals[0] += total_loss * count
        totals[1] += count
        if log and training and global_step % args.log_every == 0:
            values = totals.cpu().tolist()
            event(log, event="progress", epoch=epoch + 1, period=period, panel=name, step=global_step,
                  windows=windows, mean_loss=values[0] / max(values[1], 1), elapsed_seconds=time.perf_counter() - started)

    items = schedule(context, period, args, epoch)
    if not items:
        raise ValueError(f"No days in {period}")
    for number, (name, path, samples, prep) in enumerate(prepared_days(context, items, args, task, period, epoch), 1):
        preparation_seconds += prep
        if not training and task == "queue":
            selected = []
            daily_seen = set()
            for sample in samples:
                kind = sample["condition"]
                if (seen_conditions[name][kind] < args.validation_windows
                        and (not getattr(args, "queue_validation_spread", False) or kind not in daily_seen)):
                    selected.append(sample)
                    daily_seen.add(kind)
                    seen_conditions[name][kind] += 1
            samples = selected
        pending[name].extend(samples)
        size = (update_size if training else batch_size)(args, name, len(context[name]["panel"].links), task)
        while len(pending[name]) >= size:
            process(name, pending[name][:size])
            del pending[name][:size]
        if log and (number == 1 or number % 50 == 0 or number == len(items)):
            event(log, event="data_progress", epoch=epoch + 1, period=period, days=number, total_days=len(items), panel=name, windows=windows)
    for name, samples in pending.items():
        if samples:
            process(name, samples)
    values = totals.cpu().tolist()
    if not values[1] or not np.isfinite(values).all():
        raise RuntimeError(f"No usable windows or non-finite loss in {period}")
    elapsed = time.perf_counter() - started
    report = {"task": task, "period": period, "loss": values[0] / values[1], "windows": windows,
              "elapsed_seconds": elapsed, "preparation_seconds_sum": preparation_seconds,
              "windows_per_second": windows / elapsed, "daily_records": records,
              "optimizer_updates": sum(updates_by_panel.values()), "updates_by_panel": updates_by_panel}
    if not training:
        if task == "state":
            report.update(state_metrics(records))
            report["baselines"] = {method: state_metrics(records, method) for method in ("interpolation", "historical")}
            physical = physics_totals.cpu().tolist()
            report["physics_aux_normalized_mse"] = dict(zip(("N", "dN", "FD"), [v / max(physical[3], 1) for v in physical[:3]]))
        else:
            chosen = max(thresholds, key=lambda t: queue_metrics(records, t)["proxy_IoU"])
            report.update(queue_metrics(records, chosen))
            report["threshold"] = chosen
            report["baseline"] = queue_metrics(records, chosen, True)
            report["future_speed_rmse_kmh"] = float(np.sqrt(sum(row["speed_sse_kmh"] for row in records) / max(sum(row["count"] for row in records), 1)))
            keys = sorted((row["panel"], row["date"], row["condition"], row["origin"]) for row in records)
            report["window_signature"] = hashlib.sha256(json.dumps(keys).encode()).hexdigest()
            report["validation_spread"] = getattr(args, "queue_validation_spread", False)
    return report, global_step


def save_checkpoint(path, model, optimizer, scheduler, context, vocabulary, args, task, epoch, best, bad_epochs, step, report):
    payload = {"format_version": 2, "task": task, "weights": {k: v.detach().cpu() for k, v in model.state_dict().items()},
               "model_config": args.model_config,
               "feature_names": list(queue_feature_names(args.model_config.get("queue_changes", False)) if task == "queue" else FEATURE_NAMES),
               "node_vocab": vocabulary,
               "trained_panels": list(context), "links": {name: item["panel"].links for name, item in context.items()},
               "profiles": {name: torch.from_numpy(item["profile"]) for name, item in context.items()},
               "fit_signature": {name: signature(item["fit_paths"]) for name, item in context.items()},
               "training_args": {key: str(value) if isinstance(value, Path) else value for key, value in vars(args).items()},
               "epoch": epoch + 1, "global_step": step, "best_metric": best, "bad_epochs": bad_epochs,
               "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
               "rng_cpu": torch.get_rng_state(), "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
               "threshold": report.get("threshold", 0.5), "bidirectional": args.bidirectional,
               "metrics": {key: value for key, value in report.items() if key != "daily_records"},
               "environment": {"python": sys.version, "torch": str(torch.__version__), "cuda": torch.version.cuda}}
    if task == "state":
        payload["state_context_names"] = list(STATE_CONTEXT_NAMES if model.station_decoder != "linear" else STATE_CONTEXT_NAMES[:3])
    elif getattr(args, "queue_origin", False):
        payload["origin_fit_signature"] = {name: signature(item["origin_fit_paths"]) for name, item in context.items()}
    temporary = path.with_suffix(path.suffix + ".pending")
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, temporary)
    temporary.replace(path)


def train(args, task):
    if min(args.epochs, args.cpu_threads, args.log_every) < 1 or args.grad_clip <= 0:
        raise ValueError("Epochs, threads, log interval, and gradient clipping must be positive")
    if min(args.lambda_n, args.lambda_dn, args.lambda_fd, args.normal_weight, args.soft_iou_weight, args.speed_weight, args.patience) < 0:
        raise ValueError("Loss weights and early-stopping patience must be nonnegative")
    if not 0 <= args.history_coverage <= 1 or not 0 <= args.future_coverage <= 1:
        raise ValueError("Coverage thresholds must lie in [0, 1]")
    if args.update_windows is not None and args.update_windows < 0:
        raise ValueError("--update-windows must be nonnegative")
    if args.graph_mode != "mean" and not args.graph and not (args.resume or args.init_state):
        raise ValueError("--graph-mode reliability requires --graph")
    if args.adaptive_graph_k < 0 or args.adaptive_graph_k == 1 or not 0 <= args.wavelet_levels <= 3:
        raise ValueError("--adaptive-graph-k must be 0 or at least 2; --wavelet-levels must lie in [0, 3]")
    if min(args.validation_windows, args.windows_per_day) < 1 or args.normal_windows < 0 or args.window_gap_minutes < 0:
        raise ValueError("Queue window limits must be positive, and normal windows/gap nonnegative")
    torch.set_num_threads(args.cpu_threads)
    torch.manual_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    args.use_bf16 = device == "cuda" and (args.bf16 or args.precision == "bf16" or args.precision == "auto" and torch.cuda.is_bf16_supported())
    if (args.bf16 or args.precision == "bf16") and (device != "cuda" or not torch.cuda.is_bf16_supported()):
        raise ValueError("BF16 training requires a supported CUDA GPU")
    if args.backend != "scan" and device != "cuda":
        raise ValueError("Official Mamba training requires CUDA; use --backend scan only for CPU contract tests")
    resumed = torch.load(args.resume, map_location="cpu", weights_only=True) if args.resume else None
    if resumed and resumed["task"] != task:
        raise ValueError("Resume checkpoint task mismatch")
    if resumed:
        check_model_requests(args, resumed["model_config"])
        for key in ("queue_origin", "queue_resample", "queue_condition_gap", "queue_validation_spread"):
            saved = resumed["training_args"].get(key, False)
            if getattr(args, key, False) and not saved:
                raise ValueError(f"Resume must retain {key}; use a new checkpoint to change Queue inputs/sampling")
            setattr(args, key, saved)
        args.queue_head = resumed["model_config"].get("queue_head", "linear")
        args.queue_changes = resumed["model_config"].get("queue_changes", False)
        if args.state_regime_balance and not resumed["training_args"].get("state_regime_balance", False):
            raise ValueError("Resume must retain the loss; use a new checkpoint for regime balancing")
        # Restore settings that determine the data/loss; only the requested epoch budget and paths may change.
        for key in ("seed", "bidirectional", "windows_per_day", "normal_windows", "window_gap_minutes", "validation_windows",
                    "history_coverage", "future_coverage", "label_temperature", "uncertainty_margin", "normal_weight",
                    "soft_iou_weight", "speed_weight", "lambda_n", "lambda_dn", "lambda_fd", "max_days", "precision", "bf16",
                    "batch_size", "panel_batch", "grad_clip"):
            setattr(args, key, resumed["training_args"][key])
        args.use_bf16 = resumed["training_args"]["use_bf16"]
        # Older v2 checkpoints keep their original per-microbatch updates.
        args.update_windows = resumed["training_args"].get("update_windows")
        args.state_regime_balance = resumed["training_args"].get("state_regime_balance", False)
    context = contexts(args, task, resumed)
    if resumed:
        if set(context) != set(resumed["trained_panels"]):
            raise ValueError("Resume must retain the trained panel set")
        for name, item in context.items():
            if signature(item["fit_paths"]) != resumed["fit_signature"][name]:
                raise ValueError(f"Fit dataset changed since the checkpoint: {name}")
            if args.queue_origin and signature(item["origin_fit_paths"]) != resumed["origin_fit_signature"][name]:
                raise ValueError(f"Released Queue origin dataset changed since the checkpoint: {name}")
        vocabulary = resumed["node_vocab"]
        args.model_config = resumed["model_config"]
        model = model_from_checkpoint(resumed, device)
    else:
        vocabulary = {f"{name}|{link}": i + 1 for i, (name, link) in enumerate((name, link) for name in context for link in context[name]["panel"].links)}
        args.model_config = {"features": len(FEATURE_NAMES), "node_count": len(vocabulary), "width": args.width,
                             "blocks": args.blocks, "heads": args.heads, "dropout": args.dropout,
                             "backend": args.backend, "graph": args.graph,
                             "missing_aware": args.missing_aware, "graph_mode": args.graph_mode,
                             "station_decoder": args.station_decoder, "adaptive_graph_k": args.adaptive_graph_k,
                             "wavelet_levels": args.wavelet_levels, "wavelet_input": args.wavelet_input or "signals"}
        queue_config = {"queue_head": args.queue_head, "queue_changes": args.queue_changes,
                        "features": len(queue_feature_names(args.queue_changes))} if task == "queue" else {}
        args.model_config.update(queue_config)
        transfer = torch.load(args.init_state, map_location="cpu", weights_only=True) if task == "queue" and args.init_state else None
        if transfer:
            if transfer.get("format_version") != 2 or transfer["task"] != "state":
                raise ValueError("Transfer needs a v2 State checkpoint")
            check_model_requests(args, transfer["model_config"], queue=False)
            if any(name not in transfer["profiles"] for name in context):
                raise ValueError("The State checkpoint must cover the Queue panels")
            # Transfer fit-only statistics and identity mapping with the encoder.
            vocabulary = transfer["node_vocab"]
            args.model_config = {**transfer["model_config"], **queue_config}
            for name, item in context.items():
                if transfer["links"][name] != item["panel"].links:
                    raise ValueError("Transfer road layout mismatch")
                if transfer["fit_signature"][name] != signature(item["fit_paths"]):
                    raise ValueError("Transfer must use the same fit dates and data")
                item["profile"] = transfer["profiles"][name].numpy()
        model = (TrafficStateModel if task == "state" else TrafficQueueModel)(**args.model_config).to(device)
        if transfer:
            weights = {key.removeprefix("encoder."): value for key, value in transfer["weights"].items() if key.startswith("encoder.")}
            if args.queue_changes:
                expanded = torch.zeros_like(model.encoder.input.weight, device="cpu")
                expanded[:, :len(FEATURE_NAMES)] = weights["input.weight"]
                weights["input.weight"] = expanded
            model.encoder.load_state_dict(weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, fused=device == "cuda")
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=args.scheduler_patience, factor=0.5)
    start_epoch, best, bad_epochs, step = 0, -float("inf"), 0, 0
    if resumed:
        optimizer.load_state_dict(resumed["optimizer"])
        scheduler.load_state_dict(resumed["scheduler"])
        start_epoch, best, bad_epochs, step = resumed["epoch"], resumed["best_metric"], resumed["bad_epochs"], resumed["global_step"]
        torch.set_rng_state(resumed["rng_cpu"])
        if device == "cuda" and resumed["rng_cuda"]:
            torch.cuda.set_rng_state_all(resumed["rng_cuda"])
    if args.epochs <= start_epoch:
        raise ValueError(f"--epochs is the total budget and must exceed completed epoch {start_epoch}")
    log = args.checkpoint.with_suffix(".jsonl")
    event(log, event="start", task=task, device=device, model=args.model_config, bf16=args.use_bf16,
          panels=list(context), epochs=args.epochs, start_epoch=start_epoch, parameters=sum(p.numel() for p in model.parameters()),
          update_windows=args.update_windows,
          state_regime_weights={name: item.get("regime_weights", {}) for name, item in context.items()},
          effective_update_windows={name: update_size(args, name, len(item["panel"].links), task) for name, item in context.items()})
    for epoch in range(start_epoch, args.epochs):
        fit, step = run_period(model, context, vocabulary, args, task, "fit", epoch, device, optimizer, log, step)
        select, _ = run_period(model, context, vocabulary, args, task, "select", epoch, device, log=log, global_step=step)
        metric = select["State"] if task == "state" else select["proxy_IoU"]
        improved = metric > best
        best = max(best, metric)
        bad_epochs = 0 if improved else bad_epochs + 1
        scheduler.step(metric)
        event(log, event="epoch", epoch=epoch + 1, fit_loss=fit["loss"], select_metric=metric, best_metric=best,
              learning_rate=optimizer.param_groups[0]["lr"], fit_seconds=fit["elapsed_seconds"],
              fit_windows_per_second=fit["windows_per_second"], improved=improved,
              fit_windows=fit["windows"], optimizer_updates=fit["optimizer_updates"], updates_by_panel=fit["updates_by_panel"])
        write_json(args.checkpoint.with_suffix(f".epoch{epoch + 1}.json"), select)
        save_checkpoint(args.checkpoint.with_name(args.checkpoint.stem + ".last.pt"), model, optimizer, scheduler, context, vocabulary, args, task, epoch, best, bad_epochs, step, select)
        if improved:
            save_checkpoint(args.checkpoint, model, optimizer, scheduler, context, vocabulary, args, task, epoch, best, bad_epochs, step, select)
        if args.patience and bad_epochs >= args.patience:
            event(log, event="early_stop", epoch=epoch + 1, patience=args.patience)
            break
    print(f"Best checkpoint: {args.checkpoint}; latest/resume: {args.checkpoint.stem}.last.pt", flush=True)


def evaluate(args):
    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=True)
    if checkpoint.get("format_version") != 2:
        raise ValueError("evaluate needs a v2 checkpoint; retain the previous audit for legacy scores")
    for key, value in checkpoint["training_args"].items():
        if not hasattr(args, key) or getattr(args, key) is None:
            setattr(args, key, value)
    if args.validation_windows is None:
        args.validation_windows = checkpoint["training_args"]["validation_windows"]
    if args.validation_windows < 1:
        raise ValueError("--validation-windows must be positive")
    args.queue_validation_spread = bool(getattr(args, "queue_validation_spread", False))
    args.seed = checkpoint["training_args"]["seed"]
    args.fixed_threshold = checkpoint["threshold"]
    args.bidirectional = checkpoint["bidirectional"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.set_num_threads(args.cpu_threads)
    args.use_bf16 = device == "cuda" and checkpoint["training_args"]["use_bf16"]
    context = contexts(args, checkpoint["task"], checkpoint)
    model = model_from_checkpoint(checkpoint, device).eval()
    report, _ = run_period(model, context, checkpoint["node_vocab"], args, checkpoint["task"], args.period, 0, device)
    if args.period == "select" and checkpoint["task"] == "queue":
        # Evaluation of a saved checkpoint uses its selected threshold.
        report.update(queue_metrics(report["daily_records"], checkpoint["threshold"]))
        report["threshold"] = checkpoint["threshold"]
    if args.bootstrap:
        rows = report["daily_records"]
        dates = sorted({row["date"] for row in rows})
        by_date = {date: [row for row in rows if row["date"] == date] for date in dates}
        rng = np.random.default_rng(args.seed)
        differences = []
        for _ in range(args.bootstrap):
            sample = [row for date in rng.choice(dates, len(dates), replace=True) for row in by_date[date]]
            if checkpoint["task"] == "state":
                differences.append(state_metrics(sample)["State"] - state_metrics(sample, "interpolation")["State"])
            else:
                t = checkpoint["threshold"]
                differences.append(queue_metrics(sample, t)["proxy_IoU"] - queue_metrics(sample, t, True)["proxy_IoU"])
        report["paired_day_bootstrap_delta_95pct"] = np.percentile(differences, [2.5, 97.5]).tolist()
    report["checkpoint_epoch"] = checkpoint["epoch"]
    write_json(args.output, report)
    print(json.dumps({key: value for key, value in report.items() if key != "daily_records"}, ensure_ascii=False, indent=2))


def inference_context(checkpoint, release, name, device):
    panel = PanelInfo.load(release, name)
    if name not in checkpoint["profiles"] or checkpoint["links"][name] != panel.links:
        raise ValueError(f"Checkpoint does not cover this panel/layout: {name}")
    return panel, checkpoint["profiles"][name].numpy(), torch.tensor(
        [checkpoint["node_vocab"].get(f"{name}|{link}", 0) for link in panel.links], device=device), torch.as_tensor(panel.edges, device=device)


def predict_state_v2(model, checkpoint, release, split, output, selected=None):
    device = next(model.parameters()).device.type
    parts = []
    for name in selected or checkpoint["trained_panels"]:
        panel, profile, node_ids, edges = inference_context(checkpoint, release, name, device)
        template = pd.read_csv(release / "task1" / name / split / "sample_submission_state.csv", dtype=str)
        keys = ["panel", "timestamp", "station_id", "link_id", "mask_regime"]
        if template[keys].duplicated().any():
            raise ValueError(f"Duplicate State template keys: {name}")
        by_date = {date: frame for date, frame in template.groupby(template.timestamp.str[:10])}
        root = release / "corridors" / name / split / "mainline_states_masked"
        for path in sorted(root.glob("**/*.parquet")):
            day = StationDay.load(path, panel)
            date = day.timestamps[0][:10]
            target = by_date.pop(date, None)
            if target is None:
                continue
            if len(target.mask_regime.unique()) != 1 or path.parent.name != f"mask_regime={target.mask_regime.iloc[0]}":
                raise ValueError(f"State regime mismatch: {path}")
            ramp = ramp_features(release, panel, split, day.timestamps)
            windows = list(center_windows(len(day.timestamps)))
            prediction = np.zeros((*day.speed.shape, 2), np.float32)
            size = 8 if len(panel.links) <= 128 else 2
            mapping = torch.as_tensor(day.station_to_link, device=device)
            for start in range(0, len(windows), size):
                chunk = windows[start:start + size]
                xs, cs = zip(*(state_features_v2(day, panel, index, profile, ramp,
                              rich=checkpoint.get("model_config", {}).get("station_decoder", "linear") != "linear") for index, _ in chunk))
                batch = tensor_batch([{"x": x, "context": c} for x, c in zip(xs, cs)], ("x", "context"), device)
                with torch.inference_mode(), torch.autocast(device, dtype=torch.bfloat16,
                    enabled=device == "cuda" and checkpoint["training_args"]["use_bf16"]):
                    result = model(batch["x"], batch["context"], mapping, node_ids, panel.segments, edges, checkpoint["bidirectional"])[:, 30:42].float().cpu().numpy()
                for values, (_, absolute) in zip(result, chunk):
                    prediction[absolute] = values[:absolute.stop - absolute.start]
            if not np.isfinite(prediction).all():
                raise RuntimeError("Non-finite State prediction")
            times = {time: i for i, time in enumerate(day.timestamps)}
            stations = {station: i for i, station in enumerate(day.station_ids)}
            ti, si = target.timestamp.map(times), target.station_id.map(stations)
            if ti.isna().any() or si.isna().any():
                raise ValueError("Template key absent from State input")
            ti, si = ti.to_numpy(int), si.to_numpy(int)
            if not np.array_equal(np.array(day.station_links)[si], target.link_id.to_numpy()):
                raise ValueError("Template station/link mapping mismatch")
            result = target.copy()
            result["speed_kmh"] = prediction[ti, si, 0] * 25
            result["flow_vph"] = prediction[ti, si, 1] * 600 * day.lanes[si]
            parts.append(result[keys + ["speed_kmh", "flow_vph"]])
        if by_date:
            raise ValueError(f"Missing masked days: {sorted(by_date)}")
    if not parts:
        raise ValueError("No State predictions")
    output.parent.mkdir(parents=True, exist_ok=True)
    pd.concat(parts, ignore_index=True).to_csv(output, index=False)


def history_day(frame, panel):
    times = sorted(frame.timestamp.unique().tolist())
    if len(times) != 12 or frame.duplicated(["timestamp", "link_id"]).any():
        raise ValueError("Task 2 history must have twelve unique time/link cells")
    if not set(frame.link_id).issubset(panel.link_index):
        raise ValueError("Task 2 history has unknown links")
    index = pd.MultiIndex.from_product([times, panel.links], names=["timestamp", "link_id"])
    aligned = frame.set_index(["timestamp", "link_id"]).reindex(index)
    shape = len(times), len(panel.links)
    speed = pd.to_numeric(aligned.speed_kmh, errors="coerce").to_numpy(np.float32).reshape(shape) / 25
    flow = pd.to_numeric(aligned.flow_vph, errors="coerce").to_numpy(np.float32).reshape(shape) / panel.lanes / 600
    occupancy = pd.to_numeric(aligned.occupancy, errors="coerce").to_numpy(np.float32).reshape(shape)
    quality = pd.to_numeric(aligned.pct_observed, errors="coerce").fillna(0).to_numpy(np.float32).reshape(shape) / 100
    eligible = aligned.is_score_eligible.fillna(False).astype(bool).to_numpy().reshape(shape)
    return StationDay(times, panel.links, panel.links, np.arange(len(panel.links)), panel.lanes,
                      speed, flow, eligible & np.isfinite(speed) & np.isfinite(flow), occupancy, quality)


def predict_queue_v2(model, checkpoint, release, split, output, selected=None, probabilities=False):
    device = next(model.parameters()).device.type
    parts = []
    for name in selected or checkpoint["trained_panels"]:
        root = release / "task2" / name / split
        if not (root / "sample_submission_queue.csv").exists():
            continue
        panel, profile, node_ids, edges = inference_context(checkpoint, release, name, device)
        include_origin = checkpoint["training_args"].get("queue_origin", False)
        changes = checkpoint.get("model_config", {}).get("queue_changes", False)
        origin_context = {"panel": panel, "profile": profile}
        if include_origin:
            origin_context["origin_paths"] = masked_file_index(release, name, split)
        origin_cache = {}
        template = pd.read_csv(root / "sample_submission_queue.csv", dtype=str)
        index = pd.read_csv(root / "window_index.csv", dtype=str).set_index("window_id")
        history = pd.read_parquet(root / "window_history.parquet")
        for col in ("window_id", "timestamp", "link_id"):
            history[col] = history[col].astype(str)
        histories = {key: frame for key, frame in history.groupby("window_id")}
        inputs, targets = [], []
        for window_id, target in template.groupby("window_id", sort=False):
            if window_id not in histories or window_id not in index.index:
                raise ValueError(f"No history/index for {window_id}")
            day = history_day(histories[window_id], panel)
            origin = pd.Timestamp(index.loc[window_id, "forecast_origin"])
            expected = pd.date_range(origin - pd.Timedelta(minutes=60), periods=12, freq="5min").strftime("%Y-%m-%dT%H:%M:%SZ").tolist()
            future = pd.date_range(origin + pd.Timedelta(minutes=5), periods=6, freq="5min").strftime("%Y-%m-%dT%H:%M:%SZ").tolist()
            if day.timestamps != expected or sorted(target.timestamp.unique()) != future:
                raise ValueError(f"Unexpected Task 2 time boundary: {window_id}")
            if not set(target.link_id).issubset(panel.link_index):
                raise ValueError("Unknown Queue target link")
            timestamp = origin.strftime("%Y-%m-%dT%H:%M:%SZ")
            ramp_times = day.timestamps + ([timestamp] if include_origin else [])
            ramp = ramp_features(release, panel, split, ramp_times)
            released = released_origin_features(origin_context, timestamp, ramp[-1:], origin_cache) if include_origin else None
            inputs.append({"x": queue_features(day, panel, profile, ramp[:12], released, changes)})
            targets.append((target, future))
        size = 48 if len(panel.links) <= 128 else 12
        for start in range(0, len(inputs), size):
            batch = tensor_batch(inputs[start:start + size], ("x",), device)
            with torch.inference_mode(), torch.autocast(device, dtype=torch.bfloat16,
                enabled=device == "cuda" and checkpoint["training_args"]["use_bf16"]):
                logits, _ = model(batch["x"], node_ids, panel.segments, edges, checkpoint["bidirectional"])
                values = logits.float().sigmoid().cpu().numpy()
            if not np.isfinite(values).all():
                raise RuntimeError("Non-finite Queue prediction")
            for probability, (target, future) in zip(values, targets[start:start + size]):
                result = target.copy()
                ti = result.timestamp.map({time: i for i, time in enumerate(future)}).to_numpy(int)
                li = result.link_id.map(panel.link_index).to_numpy(int)
                result["queue_pred"] = probability[ti, li] if probabilities else (probability[ti, li] >= checkpoint["threshold"]).astype(int)
                parts.append(result[["window_id", "timestamp", "link_id", "queue_pred"]])
    if not parts:
        raise ValueError("No Queue predictions")
    result = pd.concat(parts, ignore_index=True)
    if result[["window_id", "timestamp", "link_id"]].duplicated().any():
        raise ValueError("Duplicate Queue keys")
    output.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output, index=False)
''')


내장 코드 로드: learning (1009줄)


In [22]:
# src/historical_retrieval.py — 원본 cdf888214fe8a54b73302f3988b764b74b3f30a4
load_source('historical_retrieval', r'''"""Fit-only historical patterns for optional frozen MagiNet inference."""
import hashlib
import json
from pathlib import Path

import numpy as np


CONFIG = dict(neighbors=4, window=36, stride=18, shifts=[-2, 0, 2], min_observations=8)
METHODS = ('global', 'local', 'local_gated')


class HistoricalRetrieval:
    def __init__(self, paths):
        paths = sorted(map(Path, paths))
        if not paths or len({p.stem for p in paths}) != len(paths):
            raise ValueError('중복 없는 fit 날짜별 NPZ가 필요합니다.')
        self.dates = [p.stem for p in paths]
        residuals, valid = [], []
        for path in paths:
            with np.load(path, allow_pickle=False) as day:
                active = day['active'].astype(bool)
                residual = np.where(active[..., None], day['truth'] - day['aux'][..., 8:10], 0.)
                if residual.shape != (*active.shape, 2) or not np.isfinite(residual).all():
                    raise ValueError(f'잘못된 fit 관측 배열입니다: {path}')
                residuals.append(residual.astype(np.float32))
                valid.append(active)
        # ponytail: keep one fit memory in RAM; persist only if rebuilding becomes a bottleneck.
        self.residuals, self.valid = np.stack(residuals), np.stack(valid)
        digest = hashlib.sha256(json.dumps(self.dates).encode())
        digest.update(self.residuals.tobytes())
        digest.update(self.valid.tobytes())
        self.sha256 = digest.hexdigest()

    def candidates(self, day, network, methods=METHODS):
        if not set(methods) <= set(METHODS):
            raise ValueError('지원하지 않는 과거 검색 방법입니다.')
        values, visible, prior = day['values'][..., :2], day['visible'][..., :2], day['aux'][..., 8:10]
        if (values.shape != prior.shape or visible.shape != prior.shape or network.shape != prior.shape
                or prior.shape != self.residuals.shape[1:]):
            raise ValueError('과거 검색의 시각·관측소·채널 구성이 다릅니다.')
        if self.dates[-1] >= str(day['timestamps'][0])[:10]:
            raise ValueError('과거 검색은 추론 날짜보다 이른 fit 자료만 사용해야 합니다.')
        visible = visible.astype(bool) & np.isfinite(values)
        residual = np.where(visible, values - prior, 0.)
        results = {}
        if 'global' in methods:
            estimate = np.zeros_like(prior)
            for channel in (0, 1):
                estimate[..., channel], _ = self._retrieve(residual, visible, 0, len(prior), channel, [0])
            results['global'] = np.maximum(prior + estimate, 0.)
        if set(methods) & {'local', 'local_gated'}:
            sums = np.zeros_like(prior, dtype=np.float64)
            gated = np.zeros_like(sums)
            weights = np.zeros((len(prior), 1, 1), dtype=np.float64)
            width = min(CONFIG['window'], len(prior))
            starts = sorted(set(range(0, len(prior) - width + 1, CONFIG['stride'])) | {len(prior) - width})
            taper = np.hanning(width + 2)[1:-1, None]
            for start in starts:
                stop = start + width
                for channel in (0, 1):
                    estimate, confidence = self._retrieve(residual, visible, start, stop, channel, CONFIG['shifts'])
                    candidate = np.maximum(prior[start:stop, :, channel] + estimate, 0.)
                    sums[start:stop, :, channel] += taper * candidate
                    gated[start:stop, :, channel] += taper * (
                        network[start:stop, :, channel] + confidence * (candidate - network[start:stop, :, channel]))
                weights[start:stop] += taper[..., None]
            if 'local' in methods:
                results['local'] = (sums / weights).astype(np.float32)
            if 'local_gated' in methods:
                results['local_gated'] = (gated / weights).astype(np.float32)
        return results

    def _retrieve(self, residual, visible, start, stop, channel, shifts):
        known = visible[start:stop, :, channel]
        query = residual[start:stop, :, channel]
        if known.sum() < CONFIG['min_observations']:
            return np.zeros_like(query), 0.
        candidates, masks, scores = [], [], []
        indices = np.arange(start, stop)
        for shift in shifts:
            shifted = indices + shift
            inside = (shifted >= 0) & (shifted < self.residuals.shape[1])
            indices_safe = shifted.clip(0, self.residuals.shape[1] - 1)
            patterns = self.residuals[..., channel][:, indices_safe]
            valid = self.valid[:, indices_safe] & inside[None, :, None]
            shared = valid & known[None]
            count = shared.sum(axis=(1, 2))
            error = np.where(shared, patterns - query, 0.)
            distance = np.square(error).sum(axis=(1, 2), dtype=np.float64) / np.maximum(count, 1)
            distance[count < max(CONFIG['min_observations'], known.sum() // 2)] = np.inf
            candidates.append(patterns)
            masks.append(valid)
            scores.append(distance)
        scores = np.stack(scores)
        # One shift per historical day prevents the same day occupying all nearest slots.
        best_shift = scores.argmin(axis=0)
        distance = scores[best_shift, np.arange(len(self.dates))]
        neighbors = np.argsort(distance, kind='stable')[:CONFIG['neighbors']]
        neighbors = neighbors[np.isfinite(distance[neighbors])]
        if not len(neighbors):
            return np.zeros_like(query), 0.
        patterns = np.stack([candidates[best_shift[n]][n] for n in neighbors])
        valid = np.stack([masks[best_shift[n]][n] for n in neighbors])
        estimate = np.where(valid, patterns, 0.).sum(axis=0) / np.maximum(valid.sum(axis=0), 1)
        # Compare only public observations; target truth/active are never read here.
        reference = np.square(query[known]).mean(dtype=np.float64)
        mismatch = np.square((estimate - query)[known]).mean(dtype=np.float64)
        uncertainty = np.where(valid, np.square(patterns - estimate), 0.).sum(dtype=np.float64) / max(valid.sum(), 1)
        confidence = np.clip(1 - mismatch / (reference + 1e-8), 0, 1) * reference / (reference + uncertainty + 1e-8)
        return estimate, float(confidence)

    def apply(self, day, network, method, weights):
        candidate = self.candidates(day, network, (method,))[method]
        return (network + np.asarray(weights, np.float32) * (candidate - network)).astype(np.float32)
''')


내장 코드 로드: historical_retrieval (115줄)


In [23]:
# src/maginet.py — 원본 cdf888214fe8a54b73302f3988b764b74b3f30a4
load_source('maginet', r'''"""Local MagiNet Task 1/3 runner, adapted from task13_maginet_colab.ipynb.

Legacy presets retain the notebook architecture. Optional context/channel graph
paths and State-based selection are documented with local CLI commands in README.md.
"""
from __future__ import annotations

import argparse
from types import SimpleNamespace

from dataclasses import dataclass, field
from pathlib import Path, PurePosixPath
from functools import lru_cache, partial
import hashlib, heapq, json, random, re, shutil, tempfile, time, zipfile

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

PROJECT_ROOT = Path(__file__).resolve().parents[1]
RELEASE_ROOT = PROJECT_ROOT / 'data' / 'kaggle_public'
INPUT_NORMALIZATION = 'standard'  # 'fixed': 기존 /25·/600, 'standard': fit 관측으로 입력 표준화
EXPERIMENT = 'temporal_direct'
# 한 번에 한 요소만 비교합니다. 검증된 조합은 이 표에 별도 이름으로 추가하세요.
EXPERIMENTS = {
    'baseline':        (0.10, False, False, False),
    'dn_003':          (0.03, False, False, False),
    'dn_zero':         (0.00, False, False, False),
    'temporal_direct': (0.10, True,  False, False),
    'mask_resample':   (0.10, False, True,  False),
    'learned_graph':   (0.10, False, False, True),
    'temporal_direct_dn003': (0.03, True, False, False),
    'temporal_direct_dnzero': (0.00, True, False, False),
    'temporal_direct_learned_graph': (0.10, True, False, True),
    'temporal_direct_dn003_learned_graph': (0.03, True, False, True),
    'temporal_direct_dn003_context_graph': (0.03, True, False, True),
    'temporal_direct_dn003_channel_graph': (0.03, True, False, True),
}
SPATIAL_EXPERIMENTS = {
    'temporal_direct_dn003_context_graph': 'context',
    'temporal_direct_dn003_channel_graph': 'channel',
}
SPATIAL_MODE = 'legacy'
if EXPERIMENT not in EXPERIMENTS:
    raise ValueError(f'알 수 없는 실험: {EXPERIMENT}')
DN_WEIGHT, TEMPORAL_DIRECT, MASK_RESAMPLE, LEARNED_GRAPH = EXPERIMENTS[EXPERIMENT]
REPORT_BASE = PROJECT_ROOT / 'reports' / 'maginet'
REPORT_DIR = REPORT_BASE / EXPERIMENT
CACHE_STORE_DIR = PROJECT_ROOT / 'data' / 'maginet_cache'
CACHE_METADATA_DIR = CACHE_STORE_DIR / 'metadata'
SAVED_CACHE_ZIP = None
CACHE_DIR = CACHE_STORE_DIR  # ZIP을 한 번 풀 로컬 폴더
CACHE_MODE = 'build'             # 'reuse': 누락 시 중단, 'build': 없는 날짜만 원본에서 생성
CACHE_PROFILE_DIR = CACHE_STORE_DIR / 'legacy_profiles'  # 이전 config.json·fit_profile-*.npy
PANELS = None                    # None: 전체 패널 공동 학습
MAX_TRAIN_DAYS = None            # 실행 점검 시 12 이상으로 줄일 수 있습니다.
DAYS_PER_REGIME = None          # 점검 시 각 기간·regime의 날짜 수를 제한합니다.
DATE_OVERRIDES = {}
SEED = 42
WINDOW, STRIDE = 72, 36
WIDTH, BLOCKS, HEADS, CHEB_K = 64, 3, 4, 3
GRAPH_DIM = 128
DROPOUT = 0.05
MGL_THRESHOLD = 0.5              # 공식 구현의 missing graph 판정 시작값
BATCH_SIZE = 2
BATCH_POLICY = 'fixed'           # adaptive: 관측소 128개 이하 8, 초과 4
PANEL_BATCH = {}                  # 명시한 패널의 배치가 policy보다 우선합니다.
UPDATES_PER_GROUP = 24           # epoch당 패널×regime 업데이트
EPOCHS, PATIENCE = 50, 8
LR, WEIGHT_DECAY = 1e-3, 1e-4
WARMUP_EPOCHS, PHYSICS_RAMP_EPOCHS = 5, 5
LOSS_WEIGHTS = dict(N=0.05, dN=DN_WEIGHT, FD=0.0)
# 공식 Task 1·3 비중을 참고한 대리 선택식입니다. 공식 Physics 점수가 아닙니다.
SELECT_WEIGHTS = dict(State=0.70, FD_proxy=0.10, LWR_proxy=0.20)
CHECKPOINT_METRIC = 'selection_proxy'
ALPHAS = (0.0, 0.25, 0.5, 0.75, 1.0)
TRAIN_MODEL = True              # False: 현재 실험의 저장된 best로 평가·오차 분해만 실행
RESUME = False                   # last_maginet.pt에서 재개; EPOCHS를 마지막 epoch보다 크게 설정
LWR_MODES = {}                   # 공개 기본값 A; 패널별 B/C 실험 시 명시

REGIMES = ('R1', 'R2', 'R3')
KEYS = ['panel', 'timestamp', 'station_id', 'link_id', 'mask_regime']
SCALE = np.array([25., 600.], dtype=np.float32)
PERIODS = dict(fit=('2030-06-01', '2030-12-31'), early=('2031-01-01', '2031-01-15'),
               blend=('2031-01-16', '2031-01-31'), february=('2031-02-01', '2031-02-28'))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
assert tuple(map(int, re.match(r'(\d+)\.(\d+)', torch.__version__).groups())) >= (2, 3)
assert 12 <= WINDOW <= 288 and 1 <= STRIDE <= WINDOW
assert WIDTH % HEADS == 0 and WIDTH % 2 == 0 and CHEB_K >= 1 and BLOCKS >= 1
assert BATCH_SIZE >= 1 and UPDATES_PER_GROUP >= 1 and EPOCHS >= 1
assert INPUT_NORMALIZATION in ('fixed', 'standard') and CACHE_MODE in ('reuse', 'build')
assert all(v >= 0 for v in LOSS_WEIGHTS.values()) and sum(SELECT_WEIGHTS.values()) > 0
assert all(v >= 0 for v in SELECT_WEIGHTS.values())
assert WARMUP_EPOCHS >= 0 and PHYSICS_RAMP_EPOCHS >= 0
assert len(set(ALPHAS)) == len(ALPHAS) and all(0 <= a <= 1 for a in ALPHAS) and 0.0 in ALPHAS and 1.0 in ALPHAS

def reset_seed(seed=None):
    seed = SEED if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def scan_layout(link_ids: list[str], topology: pd.DataFrame) -> list[str]:
    """Topological order within each weakly connected component; cycles are stable."""
    links = set(link_ids)
    outgoing = {link: set() for link in links}
    neighbors = {link: set() for link in links}
    for row in topology.itertuples(index=False):
        target = str(row.link_id)
        if target not in links:
            continue
        for source in str(getattr(row, "incoming_link_ids", "")).split(";"):
            source = source.strip()
            if source in links and source != target:
                outgoing[source].add(target)
                neighbors[source].add(target)
                neighbors[target].add(source)
        for dest in str(getattr(row, "outgoing_link_ids", "")).split(";"):
            dest = dest.strip()
            if dest in links and dest != target:
                outgoing[target].add(dest)
                neighbors[target].add(dest)
                neighbors[dest].add(target)
    seen: set[str] = set()
    ordered: list[str] = []
    for root in sorted(links):
        if root in seen:
            continue
        component = {root}
        pending = [root]
        seen.add(root)
        while pending:
            for neighbor in neighbors[pending.pop()]:
                if neighbor not in seen:
                    seen.add(neighbor)
                    component.add(neighbor)
                    pending.append(neighbor)
        indegree = {link: 0 for link in component}
        for source in component:
            for target in outgoing[source] & component:
                indegree[target] += 1
        ready = [link for link in component if indegree[link] == 0]
        heapq.heapify(ready)
        part = []
        while ready:
            source = heapq.heappop(ready)
            part.append(source)
            for target in outgoing[source] & component:
                indegree[target] -= 1
                if indegree[target] == 0:
                    heapq.heappush(ready, target)
        part.extend(sorted(component - set(part)))
        ordered.extend(part)
    return ordered

@dataclass
class PanelInfo:
    name: str
    links: list[str]
    link_index: dict[str, int]
    free_speed: np.ndarray
    lanes: np.ndarray
    lengths: np.ndarray
    station_lanes: dict[str, float]
    capacity: np.ndarray | None = None
    jam_density: np.ndarray | None = None
    edges: np.ndarray = field(default_factory=lambda: np.empty((2, 0), dtype=np.int64))
    ramp_map: dict[str, tuple[int, int]] = field(default_factory=dict)

    @classmethod
    def load(cls, release: Path, panel: str) -> "PanelInfo":
        panel_dir = release / "corridors" / panel
        candidates = sorted((panel_dir / "train" / "mainline_states").glob("**/*.parquet"))
        if not candidates:
            raise FileNotFoundError(f"No original train observations under {panel_dir}")
        observed = pd.read_parquet(candidates[0], columns=["link_id"]).link_id.astype(str).unique().tolist()
        network = panel_dir / "network"
        topology = pd.read_csv(network / "lwr_mainline_topology.csv", dtype=str)
        # Latent mainline connectors carry no detector value, but preserve the
        # road between observed links during the spatial scan.
        links = scan_layout(sorted(set(observed) | set(topology.link_id.dropna().astype(str))), topology)
        frame = pd.read_csv(network / "links.csv", dtype={"link_id": str}).drop_duplicates("link_id").set_index("link_id")
        fd_path = network / "fd_parameters.csv"
        fd = pd.read_csv(fd_path, dtype={"link_id": str, "station_id": str}) if fd_path.exists() else pd.DataFrame()
        grouped = fd.groupby("link_id").mean(numeric_only=True) if not fd.empty else pd.DataFrame()

        def parameter(column: str, default: float) -> np.ndarray:
            raw = pd.to_numeric(frame[column], errors="coerce").reindex(links) if column in frame else pd.Series(default, index=links)
            if column in grouped:
                raw = pd.to_numeric(grouped[column], errors="coerce").reindex(links).fillna(raw)
            return raw.fillna(default).to_numpy(dtype=np.float32)

        free_speed = parameter("free_speed_kmh", 105.0)
        lanes = np.clip(parameter("lanes", 1.0), 1, 16)
        lengths = parameter("length_km", 1.0)
        station_lanes: dict[str, float] = {}
        if "station_id" in fd and "lanes" in fd:
            for row in fd.dropna(subset=["station_id", "lanes"]).itertuples(index=False):
                station_lanes[str(row.station_id)] = float(row.lanes)
        lookup = {link: i for i, link in enumerate(links)}
        pairs = set()
        for row in topology.itertuples(index=False):
            target = str(row.link_id)
            for source in str(getattr(row, "incoming_link_ids", "")).split(";"):
                if source in lookup and target in lookup and source != target:
                    pairs.add((lookup[source], lookup[target]))
            for dest in str(getattr(row, "outgoing_link_ids", "")).split(";"):
                if target in lookup and dest in lookup and dest != target:
                    pairs.add((lookup[target], lookup[dest]))
        ramp_map = {}
        ramp_path = network / "ramp_attachment_map.csv"
        if ramp_path.exists():
            for row in pd.read_csv(ramp_path, dtype=str).itertuples(index=False):
                if row.nearest_mainline_link_id in lookup:
                    if row.ramp_type not in {"OR", "FR", "on", "off"}:
                        raise ValueError(f"Unknown ramp type: {row.ramp_type}")
                    ramp_map[row.ramp_link_id] = (lookup[row.nearest_mainline_link_id], int(row.ramp_type in {"FR", "off"}))
        return cls(panel, links, lookup, free_speed, lanes, lengths, station_lanes,
                   parameter("capacity_vph", 6000.0), np.maximum(parameter("k_jam", np.nan), 1.05 * parameter("capacity_vph", 6000.0) / np.maximum(free_speed, 1)),
                   np.array(sorted(pairs), dtype=np.int64).reshape(-1, 2).T, ramp_map)

@dataclass
class StationDay:
    timestamps: list[str]
    station_ids: list[str]
    station_links: list[str]
    station_to_link: np.ndarray
    lanes: np.ndarray
    speed: np.ndarray
    flow_lane: np.ndarray
    eligible: np.ndarray
    occupancy: np.ndarray | None = None
    quality: np.ndarray | None = None

    @classmethod
    def load(cls, path: Path, panel: PanelInfo) -> "StationDay":
        columns = ["timestamp", "station_id", "link_id", "speed_kmh", "flow_vph", "is_score_eligible"]
        # Optional published features also exist in Task 2 histories.
        import pyarrow.parquet as pq
        available = set(pq.read_schema(path).names) if path.stat().st_size else set(columns)
        columns += [c for c in ("occupancy", "pct_observed") if c in available]
        frame = pd.read_parquet(path, columns=columns)
        for column in ("timestamp", "station_id", "link_id"):
            frame[column] = frame[column].astype(str)
        if frame.duplicated(["timestamp", "station_id", "link_id"]).any():
            raise ValueError(f"Duplicate station/time rows in {path}")
        stations = frame[["station_id", "link_id"]].drop_duplicates()
        if stations.station_id.duplicated().any():
            raise ValueError(f"Station maps to multiple links in {path}")
        stations = stations.sort_values(["link_id", "station_id"])
        station_ids = stations.station_id.tolist()
        station_links = stations.link_id.tolist()
        unknown = set(station_links) - set(panel.link_index)
        if unknown:
            raise ValueError(f"Links absent from panel layout: {sorted(unknown)[:4]}")
        timestamps = sorted(frame.timestamp.unique().tolist())
        index = pd.MultiIndex.from_product([timestamps, station_ids], names=["timestamp", "station_id"])
        aligned = frame.set_index(["timestamp", "station_id"]).reindex(index)
        speed = pd.to_numeric(aligned.speed_kmh, errors="coerce").to_numpy(dtype=np.float32).reshape(len(timestamps), -1)
        flow = pd.to_numeric(aligned.flow_vph, errors="coerce").to_numpy(dtype=np.float32).reshape(len(timestamps), -1)
        lanes = np.array([panel.station_lanes.get(s, panel.lanes[panel.link_index[l]]) for s, l in zip(station_ids, station_links)], dtype=np.float32)
        lanes = np.clip(lanes, 1, 16)
        eligible = aligned.is_score_eligible.fillna(False).astype(bool).to_numpy(copy=True).reshape(len(timestamps), -1)
        eligible &= np.isfinite(speed) & np.isfinite(flow)
        occupancy = pd.to_numeric(aligned.occupancy, errors="coerce").to_numpy(np.float32).reshape(speed.shape) if "occupancy" in aligned else np.full_like(speed, np.nan)
        quality = pd.to_numeric(aligned.pct_observed, errors="coerce").fillna(0).to_numpy(np.float32).reshape(speed.shape) / 100 if "pct_observed" in aligned else eligible.astype(np.float32)
        return cls(timestamps, station_ids, station_links,
                   np.array([panel.link_index[l] for l in station_links], dtype=np.int64),
                   lanes, speed / 25.0, flow / lanes[None, :] / 600.0, eligible, occupancy, quality)

def aggregate_channels(day: StationDay, panel: PanelInfo, visible: np.ndarray | None = None):
    values = np.stack((day.speed, day.flow_lane,
                       day.occupancy if day.occupancy is not None else np.full_like(day.speed, np.nan)), -1)
    good = np.isfinite(values)
    if visible is not None:
        good &= visible[..., None] if visible.ndim == 2 else visible
    sums = np.zeros((len(day.timestamps), len(panel.links), 3), np.float32)
    counts = np.zeros_like(sums)
    quality = np.zeros(sums.shape[:2], np.float32)
    qcount = np.zeros_like(quality)
    for station, link in enumerate(day.station_to_link):
        sums[:, link] += np.where(good[:, station], values[:, station], 0)
        counts[:, link] += good[:, station]
        present = good[:, station].any(-1)
        q = day.quality[:, station] if day.quality is not None else present.astype(np.float32)
        # Observation quality remains public even when measured values are masked.
        quality[:, link] += np.where(np.isfinite(q), q, 0)
        qcount[:, link] += np.isfinite(q)
    return sums / np.maximum(counts, 1), counts > 0, quality / np.maximum(qcount, 1)

def ramp_files(release, panel, split, date):
    root = release / 'corridors' / panel.name / split / 'ramp_states'
    return sorted(set(root.glob(f"**/*{date}*.parquet"))
                  | set(root.glob(f"**/*{date.replace('-', '_')}*.parquet")))


def ramp_features(release: Path, panel: PanelInfo, split: str, timestamps: list[str]) -> np.ndarray:
    out = np.zeros((len(timestamps), len(panel.links), 4), np.float32)
    out[..., 2:] = 1  # A road without an attached ramp has known zero ramp flow.
    expected = np.zeros((len(panel.links), 2), np.float32)
    for link, kind in panel.ramp_map.values():
        expected[link, kind] += 1
    counts = np.zeros_like(out[..., :2])
    time_index = {value: i for i, value in enumerate(timestamps)}
    for path in ramp_files(release, panel, split, timestamps[0][:10]):
        frame = pd.read_parquet(path, columns=["timestamp", "ramp_link_id", "flow_vph", "pct_observed", "is_score_eligible"])
        frame = frame[frame.timestamp.isin(time_index)].drop_duplicates(["timestamp", "ramp_link_id"])
        for row in frame.itertuples(index=False):
            if row.ramp_link_id in panel.ramp_map and np.isfinite(row.flow_vph) and row.pct_observed >= 75 and bool(row.is_score_eligible):
                link, kind = panel.ramp_map[row.ramp_link_id]
                index = time_index[row.timestamp]
                out[index, link, kind] += float(row.flow_vph)
                counts[index, link, kind] += 1
    out[..., 2:] = counts >= expected[None]
    out[..., :2] *= out[..., 2:]
    return out

def dated_files(root):
    result = {}
    for path in sorted(Path(root).rglob("*.parquet")):
        match = re.search(r"\d{4}[-_]\d{2}[-_]\d{2}", path.name)
        if match is None:
            raise ValueError(f"Date missing from filename: {path}")
        date = match.group().replace("_", "-")
        if date in result:
            raise ValueError(f"Duplicate date: {date}")
        result[date] = path
    if not result:
        raise FileNotFoundError(f"No parquet files: {root}")
    return result

def fit_prior(panel, paths):
    """Only fit dates contribute to weekday/slot means and fallbacks."""
    sums = np.zeros((2016, len(panel.links), 3), np.float64)
    counts = np.zeros_like(sums)
    for number, path in enumerate(paths, 1):
        if number == 1 or number % 30 == 0 or number == len(paths):
            print(f"과거 평균 계산 {panel.name}: {number}/{len(paths)} days", flush=True)
        day = StationDay.load(path, panel)
        values, valid, _ = aggregate_channels(day, panel, day.eligible)
        dates = pd.to_datetime(day.timestamps, utc=True)
        slot = np.asarray(dates.dayofweek * 288 + dates.hour * 12 + dates.minute // 5)
        np.add.at(sums, slot, values * valid)
        np.add.at(counts, slot, valid)
    mean = sums.sum((0, 1)) / np.maximum(counts.sum((0, 1)), 1)
    link_mean = np.divide(sums.sum(0), counts.sum(0),
                          out=np.broadcast_to(mean, sums.shape[1:]).copy(),
                          where=counts.sum(0) > 0)
    return np.divide(sums, counts, out=np.broadcast_to(link_mean, sums.shape).copy(),
                     where=counts > 0).astype(np.float32)

class TargetIndex:
    def __init__(self, release, name, day, split='train'):
        path = release / 'task1' / name / split / 'sample_submission_state.csv'
        self.stations, self.links = day.station_ids, day.station_links
        self.archive, self.regimes = {}, {}
        station_lookup = {s: i for i, s in enumerate(self.stations)}
        parts = {}
        for frame in pd.read_csv(path, usecols=KEYS, dtype=str, chunksize=200_000):
            positions = frame.station_id.map(station_lookup)
            if positions.isna().any() or set(frame.panel) != {name}:
                raise ValueError(f'템플릿의 panel/station이 잘못되었습니다: {path}')
            if not np.array_equal(np.asarray(self.links)[positions.to_numpy(int)], frame.link_id.to_numpy()):
                raise ValueError(f'템플릿 station/link 연결이 다릅니다: {path}')
            times = pd.to_datetime(frame.timestamp, utc=True, format='ISO8601')
            if ((times.dt.minute % 5 != 0) | (times.dt.second != 0)).any():
                raise ValueError('템플릿은 5분 시각이어야 합니다.')
            slots = times.dt.hour * 12 + times.dt.minute // 5
            frame = frame.assign(_date=frame.timestamp.str[:10],
                                 _cell=(slots * len(self.stations) + positions).astype(np.uint32))
            for date, part in frame.groupby('_date', sort=False):
                regimes = part.mask_regime.unique()
                if len(regimes) != 1 or regimes[0] not in REGIMES:
                    raise ValueError(f'날짜별 regime이 잘못되었습니다: {date}')
                if date in self.regimes and self.regimes[date] != regimes[0]:
                    raise ValueError(f'날짜별 regime이 바뀝니다: {date}')
                self.regimes[date] = regimes[0]
                parts.setdefault(date, []).append(part._cell.to_numpy())
        if not parts:
            raise ValueError(f'빈 템플릿: {path}')
        self.archive = {d: np.concatenate(p) for d, p in parts.items()}
        if any(len(np.unique(a)) != len(a) for a in self.archive.values()):
            raise ValueError(f'중복 대상 셀: {path}')

    def get(self, day, require_eligible=True):
        date = day.timestamps[0][:10]
        if (day.station_ids, day.station_links) != (self.stations, self.links) or date not in self.archive:
            raise ValueError(f'대상 관측소 구성이 달라졌습니다: {date}')
        if any(t[:10] != date for t in day.timestamps):
            raise ValueError('하루 파일에 여러 날짜가 있습니다.')
        slots = {int(t[11:13]) * 12 + int(t[14:16]) // 5: i for i, t in enumerate(day.timestamps)}
        mask = np.zeros(day.speed.shape, bool)
        cells = self.archive[date]
        mask[[slots[int(s)] for s in cells // len(self.stations)], cells % len(self.stations)] = True
        if require_eligible and (mask & ~day.eligible).any():
            raise ValueError(f'정답이 없는 대상 셀: {date}')
        return self.regimes[date], mask

def available_panels(release):
    release = Path(release).expanduser().resolve()
    if not (release / 'corridors').is_dir() or not (release / 'task1').is_dir():
        raise FileNotFoundError(f'RELEASE_ROOT에 corridors/와 task1/이 필요합니다: {release}')
    available = sorted(p.name for p in (release / 'corridors').iterdir() if p.is_dir())
    names = available if PANELS is None else sorted(set(PANELS))
    if not names or set(names) - set(available):
        raise ValueError(f'잘못된 PANELS: {names}')
    return names

def station_graph(panel, day):
    # 무관측 connector를 따라 실제 도로에서 가장 가까운 관측 링크를 찾습니다.
    observed = set(day.station_to_link.tolist())
    neighbours = [set() for _ in panel.links]
    for a, b in panel.edges.T:
        neighbours[a].add(int(b))
        neighbours[b].add(int(a))
    links = np.eye(len(panel.links), dtype=bool)
    for origin in observed:
        pending, seen = list(neighbours[origin]), {origin}
        while pending:
            node = pending.pop()
            if node in seen:
                continue
            seen.add(node)
            if node in observed:
                links[origin, node] = True
            else:
                pending.extend(neighbours[node] - seen)
    return links[day.station_to_link[:, None], day.station_to_link[None, :]]

def prepare_panel(release, name, code_id, station_offset):
    started = time.perf_counter()
    panel = PanelInfo.load(release, name)
    files = dated_files(release / 'corridors' / name / 'train' / 'mainline_states')
    masked = dated_files(release / 'corridors' / name / 'train' / 'mainline_states_masked')
    dates = {p: [d for d in files if lo <= d <= hi] for p, (lo, hi) in PERIODS.items()}
    if name in DATE_OVERRIDES:
        saved_dates = DATE_OVERRIDES[name]
        if any(not set(saved_dates[p]) <= set(dates[p]) for p in PERIODS):
            raise ValueError(f'저장된 날짜가 로컬 데이터 기간과 다릅니다: {name}')
        dates = {p: list(saved_dates[p]) for p in PERIODS}
    if MAX_TRAIN_DAYS is not None:
        if MAX_TRAIN_DAYS < 1:
            raise ValueError('MAX_TRAIN_DAYS는 양수 또는 None이어야 합니다.')
        indices = np.linspace(0, len(dates['fit']) - 1, min(MAX_TRAIN_DAYS, len(dates['fit'])), dtype=int)
        dates['fit'] = [dates['fit'][i] for i in indices]
    if any(not dates[p] for p in ('fit', 'early', 'blend')):
        raise ValueError(f'학습·조기 종료·선택 구간이 모두 필요합니다: {name}')
    first = StationDay.load(files[dates['fit'][0]], panel)
    observed_links = np.unique(first.station_to_link)
    for attribute, fallback in (('free_speed', 100.), ('capacity', 1800. * panel.lanes), ('lengths', 1.)):
        values = getattr(panel, attribute)
        invalid = ~np.isfinite(values) | ((values < 0) if attribute == 'lengths' else (values <= 0))
        if invalid[observed_links].any():
            raise ValueError(f'관측 링크의 {attribute}가 잘못되었습니다: {name}')
        # 무관측 connector의 placeholder 값이 손실 연산에 NaN을 전파하지 않도록 합니다.
        setattr(panel, attribute, np.where(invalid, fallback, values).astype(np.float32))
    panel.jam_density = np.where(np.isfinite(panel.jam_density), panel.jam_density, 100 * panel.lanes)
    panel.jam_density = np.maximum(panel.jam_density, 1.05 * panel.capacity / panel.free_speed)
    target_index = TargetIndex(release, name, first)
    if DAYS_PER_REGIME is not None:
        for period, candidates in dates.items():
            chosen = []
            for regime in REGIMES:
                pool = [d for d in candidates if target_index.regimes.get(d) == regime]
                indices = np.linspace(0, len(pool) - 1, min(DAYS_PER_REGIME, len(pool)), dtype=int)
                chosen.extend(pool[i] for i in indices)
            dates[period] = sorted(chosen)
    for p in ('fit', 'early', 'blend'):
        if {target_index.regimes.get(d) for d in dates[p]} != set(REGIMES):
            raise ValueError(f'{name}/{p}의 R1·R2·R3가 부족합니다. 날짜 수를 늘려 주세요.')
        if not set(dates[p]) <= set(masked):
            raise ValueError(f'masked 날짜가 누락되었습니다: {name}/{p}')
    mode = LWR_MODES.get(name, 'A').upper()
    if mode not in ('A', 'B', 'C'):
        raise ValueError(f'지원하지 않는 LWR mode: {mode}')
    out = dict(release=release, panel=panel, files=files, masked=masked, dates=dates,
               targets=target_index, profile=load_fit_profile(panel, target_index, dates['fit'], files),
               panel_code=code_id, station_codes=np.arange(station_offset, station_offset + len(first.station_ids)),
               mapping=first.station_to_link, lanes=first.lanes, station_graph=station_graph(panel, first),
               mode=mode)
    topology = pd.read_csv(release / 'corridors' / name / 'network' / 'lwr_mainline_topology.csv', dtype=str)
    out['topology_nodes'] = {panel.link_index[l] for l in topology.link_id.dropna().astype(str)}
    out['preparation_seconds'] = time.perf_counter() - started
    print({'panel': name, 'stations': len(first.station_ids), 'days': {p: len(ds) for p, ds in dates.items()},
           'preparation_seconds': round(out['preparation_seconds'], 1)}, flush=True)
    return out

def link_mean_numpy(values, active, mapping, nodes):
    sums = np.zeros((len(values), nodes, values.shape[-1]), np.float64)
    counts = np.zeros((len(values), nodes), np.float64)
    np.add.at(sums.transpose(1, 0, 2), mapping, np.where(active[..., None], values, 0).transpose(1, 0, 2))
    np.add.at(counts.T, mapping, active.T)
    return sums / np.maximum(counts[..., None], 1), counts

def nearest_features(values, prior, elapsed):
    good = np.isfinite(values)
    t = np.arange(len(values))[:, None]
    left = np.maximum.accumulate(np.where(good, t, -1), axis=0)
    right = np.minimum.accumulate(np.where(good, t, len(values))[::-1], axis=0)[::-1]
    lv, rv = left >= 0, right < len(values)
    ls, rs = np.clip(left, 0, len(values) - 1), np.clip(right, 0, len(values) - 1)
    a, b = np.take_along_axis(values, ls, axis=0), np.take_along_axis(values, rs, axis=0)
    a = np.where(lv, a, np.where(rv, b, prior))
    b = np.where(rv, b, a)
    la, ra = elapsed[:, None] - elapsed[ls], elapsed[rs] - elapsed[:, None]
    fraction = np.divide(la, la + ra, out=np.zeros_like(la), where=(la + ra) > 0).clip(0, 1)
    base = a + fraction * (b - a)
    la, ra = np.where(lv, la, np.nan), np.where(rv, ra, np.nan)
    return base, la, ra

def macro_mean(panel_values):
    families = {}
    for name, value in panel_values.items():
        families.setdefault(name.rsplit('_', 1)[0], []).append(value)
    return float(np.mean([np.mean(values) for values in families.values()]))

AUX_NAMES = ('quality', 'speed_visible', 'flow_visible', 'occupancy_visible',
             'speed_left_age', 'speed_right_age', 'flow_left_age', 'flow_right_age',
             'profile_speed', 'profile_flow', 'profile_occupancy',
             'lanes', 'free_speed', 'capacity_per_lane', 'length',
             'ramp_in', 'ramp_out', 'ramp_in_valid', 'ramp_out_valid', 'on_attached', 'off_attached')


def restore_input_cache(names, archive_path, destination):
    destination = Path(destination).expanduser()
    destination.mkdir(parents=True, exist_ok=True)
    if archive_path is not None:
        archive_path = Path(archive_path).expanduser()
        if not archive_path.is_file():
            raise FileNotFoundError(f'저장 캐시 ZIP이 없습니다: {archive_path}. 새 캐시를 만들 때는 SAVED_CACHE_ZIP=None, CACHE_MODE="build"로 설정해 주세요.')
        with zipfile.ZipFile(archive_path) as archive:
            members = archive.infolist()
            for member in members:
                relative = PurePosixPath(member.filename)
                if (relative.is_absolute() or '..' in relative.parts or '\\' in member.filename or ':' in member.filename
                        or not (destination / member.filename).resolve().is_relative_to(destination.resolve())
                        or (member.external_attr >> 16) & 0o170000 == 0o120000):
                    raise ValueError(f'안전하지 않은 ZIP 경로: {member.filename}')
            fingerprint = hashlib.sha256(json.dumps(
                [(m.filename, m.file_size, m.CRC) for m in members], sort_keys=True).encode()).hexdigest()
            marker = destination / '.cache_source.json'
            if marker.exists() and json.loads(marker.read_text())['fingerprint'] != fingerprint:
                raise ValueError('다른 ZIP을 푼 폴더입니다. CACHE_DIR를 새 로컬 경로로 지정해 주세요.')
            pending = [m for m in members if not m.is_dir() and
                       (not marker.exists() or not (destination / m.filename).is_file())]
            if pending:
                needed = sum(m.file_size for m in pending)
                if shutil.disk_usage(destination).free < needed + 2 * 2**30:
                    raise RuntimeError(f'캐시 ZIP 복원에 {needed / 2**30:.1f} GiB + 여유 2 GiB가 필요합니다.')
                began = time.perf_counter()
                for number, member in enumerate(pending, 1):
                    target = destination / member.filename
                    target.parent.mkdir(parents=True, exist_ok=True)
                    temporary = target.with_suffix(target.suffix + '.tmp')
                    with archive.open(member) as src, temporary.open('wb') as dst:
                        shutil.copyfileobj(src, dst)
                    temporary.replace(target)
                    if number == 1 or number % 100 == 0 or number == len(pending):
                        print(f'저장 ZIP 복원: {number}/{len(pending)}파일', flush=True)
                marker_temporary = marker.with_suffix('.json.tmp')
                marker_temporary.write_text(json.dumps(dict(fingerprint=fingerprint, archive=str(archive_path))))
                marker_temporary.replace(marker)
                print({'cache_restore_seconds': round(time.perf_counter() - began, 1)})
            else:
                print('같은 ZIP이 이미 복원되어 로컬 NPZ를 재사용합니다.')
    roots = {p.parent.parent.parent for p in destination.rglob('*.npz')
             if p.parent.name in PERIODS and p.parent.parent.name in names
             and re.fullmatch(r'\d{4}-\d{2}-\d{2}\.npz', p.name)}
    if len(roots) > 1:
        raise ValueError('ZIP에 캐시 루트가 여러 개입니다. 필요한 폴더를 CACHE_DIR로 지정하고 SAVED_CACHE_ZIP=None으로 설정해 주세요.')
    if not roots and CACHE_MODE == 'reuse':
        raise ValueError('캐시의 <패널>/fit|early|blend/YYYY-MM-DD.npz 구조를 찾지 못했습니다. 다른 모델의 NPZ는 사용할 수 없습니다.')
    return next(iter(roots)) if roots else destination, destination


def source_signature(paths):
    records = [(Path(p).relative_to(RELEASE_ROOT).as_posix(), file_digest(p)) for p in sorted(set(paths))]
    return hashlib.sha256(json.dumps(records, sort_keys=True).encode()).hexdigest()


def profile_source_signature(panel, fit_dates, files):
    network = RELEASE_ROOT / 'corridors' / panel.name / 'network'
    return source_signature([files[d] for d in fit_dates] + list(network.glob('*.csv')))


def day_source_signature(context, date):
    paths = [context['files'][date], context['masked'][date]]
    paths += ramp_files(context['release'], context['panel'], 'train', date)
    digest = hashlib.sha256(source_signature(paths).encode())
    digest.update(np.sort(context['targets'].archive[date]).astype('<u4').tobytes())
    digest.update(context['targets'].regimes[date].encode())
    return digest.hexdigest()


def cache_is_current(path, source):
    marker = path.with_suffix('.sources.json')
    return marker.is_file() and json.loads(marker.read_text(encoding='utf-8')) == source


def write_cache_source(path, source):
    marker = path.with_suffix('.sources.json')
    temporary = marker.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(source), encoding='utf-8')
    temporary.replace(marker)


def load_fit_profile(panel, targets, fit_dates, files):
    metadata = dict(format='maginet-input-v1', panel=panel.name, links=panel.links,
                    stations=targets.stations, station_links=targets.links,
                    fit_dates=fit_dates, aux_names=list(AUX_NAMES), scale=SCALE.tolist())
    key = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()[:16]
    folder = CACHE_METADATA_DIR / 'profiles' / panel.name / key
    folder.mkdir(parents=True, exist_ok=True)
    profile_path, metadata_path = folder / 'fit_profile.npy', folder / 'cache_metadata.json'
    source = profile_source_signature(panel, fit_dates, files)
    legacy = CACHE_ROOT / panel.name
    legacy_metadata = legacy / 'cache_metadata.json'
    if legacy_metadata.exists() and json.loads(legacy_metadata.read_text()) != metadata:
        raise ValueError(f'이전 캐시의 학습 날짜·관측소·입력 형식이 다릅니다: {panel.name}')
    if not metadata_path.exists() and legacy_metadata.exists():
        legacy_profile = legacy / 'fit_profile.npy'
        if not legacy_profile.is_file():
            raise FileNotFoundError(f'이전 캐시 과거 평균이 누락되었습니다: {legacy_profile}')
        profile = np.load(legacy_profile, allow_pickle=False)
        if profile.shape != (2016, len(panel.links), 3) or not np.isfinite(profile).all():
            raise ValueError(f'이전 캐시 과거 평균의 형태·값이 잘못되었습니다: {panel.name}')
        temporary = profile_path.with_suffix('.npy.tmp')
        shutil.copyfile(legacy_profile, temporary)
        temporary.replace(profile_path)
        metadata_temporary = metadata_path.with_suffix('.json.tmp')
        metadata_temporary.write_text(json.dumps(metadata, ensure_ascii=False, sort_keys=True))
        metadata_temporary.replace(metadata_path)
    if metadata_path.exists():
        if json.loads(metadata_path.read_text()) != metadata:
            raise ValueError(f'저장 과거 평균의 학습 날짜·관측소·입력 형식이 다릅니다: {metadata_path}')
    if profile_path.is_file() and metadata_path.is_file() and cache_is_current(profile_path, source):
        profile = np.load(profile_path, allow_pickle=False)
    else:
        if CACHE_MODE == 'reuse':
            raise ValueError(f'{panel.name}: 과거 평균의 원본이 변경됐거나 출처 기록이 없습니다. --cache-mode build로 한 번 실행해 주세요.')
        profile = fit_prior(panel, [files[d] for d in fit_dates])
        candidates = [CACHE_PROFILE_DIR / f'fit_profile-{panel.name}.npy']
        candidates += sorted(CACHE_CONTAINER.rglob(f'fit_profile-{panel.name}.npy'))
        for candidate in dict.fromkeys(candidates):
            config_path = candidate.parent / 'config.json'
            if not candidate.is_file() or not config_path.is_file():
                continue
            saved = json.loads(config_path.read_text(encoding='utf-8')).get('panels', {}).get(panel.name, {})
            legacy_profile = np.load(candidate, allow_pickle=False)
            if (saved.get('stations') != targets.stations or saved.get('links') != targets.links
                    or saved.get('dates', {}).get('fit') != fit_dates
                    or hashlib.sha256(legacy_profile.tobytes()).hexdigest() != saved.get('profile_hash')
                    or not np.array_equal(legacy_profile, profile)):
                raise ValueError(f'이전 과거 평균이 현재 원본·관측소·fit 날짜와 다릅니다: {candidate}')
            break
        if profile.shape != (2016, len(panel.links), 3) or not np.isfinite(profile).all():
            raise ValueError(f'저장 과거 평균의 형태·값이 잘못되었습니다: {panel.name}')
        if not profile_path.is_file() or not np.array_equal(np.load(profile_path, allow_pickle=False), profile):
            temporary = profile_path.with_suffix('.npy.tmp')
            with temporary.open('wb') as stream:
                np.save(stream, profile, allow_pickle=False)
            temporary.replace(profile_path)
        metadata_temporary = metadata_path.with_suffix('.json.tmp')
        metadata_temporary.write_text(json.dumps(metadata, ensure_ascii=False, sort_keys=True))
        metadata_temporary.replace(metadata_path)
        write_cache_source(profile_path, source)
    if profile.shape != (2016, len(panel.links), 3) or not np.isfinite(profile).all():
        raise ValueError(f'캐시 과거 평균의 형태·값이 잘못되었습니다: {panel.name}')
    return profile.astype(np.float32, copy=False)


def validate_cache_targets(data, context, date):
    required = {'values', 'visible', 'aux', 'slot', 'weekday', 'base', 'truth', 'target',
                'active', 'observed', 'timestamps', 'ti', 'si', 'regime', 'ramp_valid', 'consecutive'}
    if required - set(data):
        raise ValueError(f'이 MagiNet 노트북의 NPZ 형식이 아닙니다: {date}, 누락 {sorted(required - set(data))}')
    target, timestamps = data['target'], data['timestamps']
    stations = len(context['targets'].stations)
    if (target.dtype != np.bool_ or target.ndim != 2 or target.shape != (len(timestamps), stations)
            or len(set(timestamps.tolist())) != len(timestamps)
            or any(str(t)[:10] != date for t in timestamps)):
        raise ValueError(f'캐시 날짜·관측소 형태가 다릅니다: {date}')
    ti, si = np.nonzero(target)
    cells = np.array([int(str(t)[11:13]) * 12 + int(str(t)[14:16]) // 5
                      for t in timestamps], np.int64)[ti] * stations + si
    expected = context['targets'].archive[date]
    if (not np.array_equal(np.sort(cells), np.sort(expected))
            or not np.array_equal(ti, data['ti']) or not np.array_equal(si, data['si'])
            or int(data['regime']) != REGIMES.index(context['targets'].regimes[date])):
        raise ValueError(f'캐시 대상 셀·regime이 템플릿과 다릅니다: {date}')
    if 'profile' in context:
        slot = data['weekday'] * 288 + data['slot']
        prior = context['profile'][slot[:, None], context['mapping'][None, :]].clip(-10, 10)
        if not np.allclose(data['aux'][..., 8:11], prior, atol=1e-6, rtol=1e-6):
            raise ValueError(f'캐시의 fit 과거 평균 특징이 현재 학습 구간과 다릅니다: {date}')
    return int(data['regime']), len(ti)


def fit_input_normalization(entries, cache_folder, mode=None, legacy_folder=None):
    mode = INPUT_NORMALIZATION if mode is None else mode
    fit = sorted((e for e in entries if e['period'] == 'fit'), key=lambda e: (e['panel'], e['date']))
    if not fit or mode not in ('fixed', 'standard'):
        raise ValueError('표준화에는 fit 캐시와 fixed/standard 모드가 필요합니다.')
    provenance = []
    for entry in fit:
        # NPZ의 배열 CRC는 ZIP 복원·경로 이동·수정 시각 변경 후에도 같습니다.
        with zipfile.ZipFile(entry['path']) as archive:
            arrays = [(name, archive.getinfo(name).file_size, archive.getinfo(name).CRC)
                      for name in ('values.npy', 'visible.npy')]
        provenance.append((entry['panel'], entry['date'], arrays))
    key = hashlib.sha256(json.dumps(dict(format='fit-visible-crc-v1', mode=mode, files=provenance),
                                   sort_keys=True).encode()).hexdigest()
    path = Path(cache_folder) / f'input_normalization-{key[:16]}.json'
    sources = ([Path(legacy_folder) / path.name] if legacy_folder is not None else [])
    sources.append(REPORT_BASE / 'full' / path.name)
    for source in sources:
        if path.exists():
            break
        if not source.is_file():
            continue
        saved = json.loads(source.read_text())
        if saved.get('source_signature') != key or saved.get('mode') != mode:
            raise ValueError('이전 실행의 입력 표준화 설정이 다릅니다.')
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix('.json.tmp')
        temporary.write_text(json.dumps(saved, indent=2, ensure_ascii=False))
        temporary.replace(path)
    if path.exists():
        saved = json.loads(path.read_text())
        if saved.get('source_signature') != key or saved.get('mode') != mode:
            raise ValueError('저장된 입력 표준화 설정이 다릅니다.')
        return saved
    count, total, square = np.zeros(3, np.int64), np.zeros(3, np.float64), np.zeros(3, np.float64)
    if mode == 'standard':
        for number, entry in enumerate(fit, 1):
            with np.load(entry['path'], allow_pickle=False) as data:
                values, visible = data['values'], data['visible']
                if (values.shape != visible.shape or visible.dtype != np.bool_
                        or values.ndim != 3 or values.shape[-1] != 3):
                    raise ValueError(f'표준화 입력 형태가 다릅니다: {entry["path"]}')
                if not np.isfinite(values[visible]).all():
                    raise ValueError(f'관측된 입력에 NaN/Inf가 있습니다: {entry["path"]}')
                observed = np.where(visible, values, 0).astype(np.float64)
                count += visible.sum((0, 1))
                total += observed.sum((0, 1))
                square += np.square(observed).sum((0, 1))
            if number == 1 or number % 100 == 0 or number == len(fit):
                print(f'fit 관측 입력 표준화: {number}/{len(fit)}일', flush=True)
    mean = total / np.maximum(count, 1)
    std = np.sqrt(np.maximum(square / np.maximum(count, 1) - mean**2, 0))
    std = np.where(std > 1e-6, std, 1.)
    saved = dict(mode=mode, mean=mean.tolist(), std=std.tolist(), count=count.tolist(),
                 source_signature=key, fitted_period='fit', channels=['speed/25', 'flow_per_lane/600', 'occupancy'])
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(saved, indent=2, ensure_ascii=False))
    temporary.replace(path)
    return saved


def attach_vocabulary(contexts):
    families = sorted({n.rsplit('_', 1)[0] for n in contexts})
    for name, c in contexts.items():
        family, direction = name.rsplit('_', 1)
        if direction not in ('N', 'S', 'E', 'W'):
            raise ValueError(f'알 수 없는 방향: {name}')
        c['family_code'] = families.index(family)
        c['direction_code'] = ('N', 'S', 'E', 'W').index(direction)


def chebyshev_supports(adjacency, order=None):
    order = CHEB_K if order is None else order
    adjacency = np.asarray(adjacency, np.float64)
    if adjacency.ndim != 2 or adjacency.shape[0] != adjacency.shape[1] or not np.allclose(adjacency, adjacency.T):
        raise ValueError('Chebyshev 그래프는 대칭 인접행렬이어야 합니다.')
    degree = adjacency.sum(-1)
    inv = 1 / np.sqrt(np.maximum(degree, 1))
    identity = np.eye(len(adjacency))
    laplacian = identity - inv[:, None] * adjacency * inv[None, :]
    largest = float(np.linalg.eigvalsh(laplacian).max())
    scaled = 2 * laplacian / largest - identity if largest > 1e-10 else -identity
    supports = [identity]
    if order > 1:
        supports.append(scaled)
    for _ in range(2, order):
        supports.append(2 * scaled @ supports[-1] - supports[-2])
    return np.stack(supports).astype(np.float32)


def input_arrays(day, context, split='train'):
    panel, mapping = context['panel'], day.station_to_link
    values = np.stack((day.speed, day.flow_lane, day.occupancy), -1)
    visible = np.isfinite(values)
    dates = pd.to_datetime(day.timestamps, utc=True, format='ISO8601')
    elapsed = np.asarray((dates - dates[0]).total_seconds() / 60, np.float32)
    if len(elapsed) < 2 or (np.diff(elapsed) <= 0).any():
        raise ValueError('하루 시각은 서로 다른 오름차순이어야 합니다.')
    slot, weekday = np.asarray(dates.hour * 12 + dates.minute // 5), np.asarray(dates.dayofweek)
    prior = context['profile'][(weekday * 288 + slot)[:, None], mapping[None, :]]
    baseline, ages = [], []
    for channel, data in enumerate((day.speed, day.flow_lane)):
        base, left_age, right_age = nearest_features(data, prior[..., channel], elapsed)
        baseline.append(base)
        ages += [np.nan_to_num(age / 360, nan=4.).clip(0, 4) for age in (left_age, right_age)]
    ramps = ramp_features(context['release'], panel, split, day.timestamps)
    r = ramps[:, mapping]
    attached = np.zeros((len(panel.links), 2), np.float32)
    for link, kind in panel.ramp_map.values():
        attached[link, kind] = 1
    spread = lambda v: np.broadcast_to(v, day.speed.shape)
    aux = np.stack((day.quality, visible[..., 0], visible[..., 1], visible[..., 2], *ages,
                    prior[..., 0], prior[..., 1], prior[..., 2], spread(day.lanes / 5),
                    spread(panel.free_speed[mapping] / 25), spread(panel.capacity[mapping] / panel.lanes[mapping] / 600),
                    spread(panel.lengths[mapping]), r[..., 0] / 600, r[..., 1] / 600, r[..., 2], r[..., 3],
                    spread(attached[mapping, 0]), spread(attached[mapping, 1])), -1)
    if aux.shape[-1] != len(AUX_NAMES) or np.isinf(values).any():
        raise ValueError('입력 배열이 잘못되었습니다.')
    # 수치상 placeholder 0은 mask로 제외되며 관측값 임베딩으로 사용되지 않습니다.
    values = np.where(visible, values, 0).astype(np.float32)
    aux = np.nan_to_num(aux, nan=0., posinf=10., neginf=-10.).clip(-10, 10).astype(np.float32)
    base = np.stack(baseline, -1).astype(np.float32)
    if not np.isfinite(base).all():
        raise ValueError('기준 보간이 유한하지 않습니다.')
    return dict(values=values, visible=visible, aux=aux, slot=slot.astype(np.int64),
                weekday=weekday.astype(np.int64), base=base,
                ramp_valid=(ramps[..., 2] > 0) & (ramps[..., 3] > 0))


def consecutive_five_minutes(timestamps):
    times = pd.to_datetime(timestamps, utc=True, format='ISO8601')
    return np.asarray((times[1:] - times[:-1]) == pd.Timedelta(minutes=5))


def make_day(context, date, split='train', target_index=None, input_files=None):
    panel = context['panel']
    path = context['masked'][date] if split == 'train' else input_files[date]
    day = StationDay.load(path, panel)
    if split == 'train':
        truth = StationDay.load(context['files'][date], panel)
        if (day.timestamps, day.station_ids, day.station_links) != (truth.timestamps, truth.station_ids, truth.station_links):
            raise ValueError(f'정답과 masked 입력이 다릅니다: {date}')
        regime, target = context['targets'].get(truth)
    else:
        truth = None
        regime, target = target_index.get(day, require_eligible=False)
    if (day.station_ids, day.station_links) != (context['targets'].stations, context['targets'].links):
        raise ValueError(f'관측소 구성이 다릅니다: {date}')
    path_regimes = [p.name.split('=', 1)[1] for p in path.parents if p.name.startswith('mask_regime=')]
    if path_regimes and path_regimes[0] != regime:
        raise ValueError(f'입력과 템플릿의 regime이 다릅니다: {date}')
    if np.isfinite(day.speed[target]).any() or np.isfinite(day.flow_lane[target]).any():
        raise ValueError(f'대상 정답이 입력에 보입니다: {date}')
    d = input_arrays(day, context, split)
    observed_values = np.stack((day.speed * 25, day.flow_lane * 600 * day.lanes), -1)
    observed = np.isfinite(observed_values).all(-1)
    active = (observed & (truth.eligible if truth is not None else day.eligible)) | target
    labels = np.stack((truth.speed, truth.flow_lane), -1) if truth is not None else np.zeros((*day.speed.shape, 2), np.float32)
    ti, si = np.nonzero(target)
    d.update(truth=np.nan_to_num(labels), target=target, active=active,
             observed=np.nan_to_num(observed_values), timestamps=np.asarray(day.timestamps), ti=ti, si=si,
             regime=np.array(REGIMES.index(regime), np.int32),
             consecutive=consecutive_five_minutes(day.timestamps))
    return d


def validate_resampling_occupancy(data):
    if not data['visible'][data['target'], 2].all():
        raise ValueError('mask_resample 중단: 기존 대상의 occupancy 원본이 NPZ에 없어 결측을 복원할 수 없습니다. 다른 EXPERIMENT를 사용하거나 원본 fit occupancy를 보관하는 전처리를 별도로 구현해 주세요.')


def cache_context(context, cache_root, date_overrides=None, periods=('fit', 'early', 'blend')):
    started, entries = time.perf_counter(), []
    for period in periods:
        dates = context['dates'][period] if date_overrides is None else date_overrides[period]
        folder = cache_root / context['panel'].name / period
        folder.mkdir(parents=True, exist_ok=True)
        for number, date in enumerate(dates, 1):
            path = folder / f'{date}.npz'
            source = day_source_signature(context, date)
            if not path.exists() or not cache_is_current(path, source):
                if CACHE_MODE == 'reuse':
                    if path.exists():
                        raise ValueError(f'캐시 원본이 변경됐거나 출처 기록이 없습니다: {path}. --cache-mode build로 실행해 주세요.')
                    raise FileNotFoundError(f'저장 캐시가 누락되었습니다: {path}. 원본에서 생성하려면 CACHE_MODE="build"로 변경해 주세요.')
                if shutil.disk_usage(cache_root).free < 2 * 2**30:
                    raise RuntimeError('임시 캐시 디스크 여유 공간이 2 GiB 미만입니다.')
                d = make_day(context, date)
                if TRAIN_MODEL and MASK_RESAMPLE and period == 'fit':
                    validate_resampling_occupancy(d)
                regime, targets = validate_cache_targets(d, context, date)
                same = False
                if path.exists():
                    with np.load(path, allow_pickle=False) as saved:
                        same = set(saved.files) == set(d) and all(np.array_equal(saved[k], v) for k, v in d.items())
                if not same:
                    temporary = path.with_suffix('.npz.tmp')
                    with temporary.open('wb') as stream:
                        np.savez_compressed(stream, **d)
                    temporary.replace(path)
                write_cache_source(path, source)
                load_day.cache_clear()
            else:
                with np.load(path, allow_pickle=False) as d:
                    regime, targets = validate_cache_targets(d, context, date)
                    if TRAIN_MODEL and MASK_RESAMPLE and period == 'fit':
                        validate_resampling_occupancy(d)
            entries.append(dict(panel=context['panel'].name, date=date, period=period, regime=regime,
                                targets=targets, path=str(path)))
            if number == 1 or number % 20 == 0 or number == len(dates):
                print(f"입력 캐시 {context['panel'].name}/{period}: {number}/{len(dates)}일", flush=True)
    context['cache_seconds'] = context.get('cache_seconds', 0.) + time.perf_counter() - started
    return entries


@lru_cache(maxsize=2)
def load_day(path):
    with np.load(path, allow_pickle=False) as d:
        result = {k: d[k] for k in d.files}
    values, visible, base, aux = (result[k] for k in ('values', 'visible', 'base', 'aux'))
    if (values.ndim != 3 or values.shape[-1] != 3 or visible.dtype != np.bool_
            or visible.shape != values.shape or base.shape != values.shape[:2] + (2,)
            or aux.shape != values.shape[:2] + (len(AUX_NAMES),)
            or not np.isfinite(values[visible]).all() or not np.isfinite(base).all()
            or not np.isfinite(aux).all() or (base < 0).any()
            or visible[result['target'], :2].any()):
        raise ValueError(f'캐시 배열 형태·관측 마스크·보간값이 잘못되었습니다: {path}')
    return result


def resample_fit_mask(d, context, rng, period):
    if period != 'fit':
        raise ValueError('결측 마스크 재표본화는 fit 학습에서만 사용할 수 있습니다.')
    validate_resampling_occupancy(d)
    eligible = d['active'].astype(bool)
    if not eligible.any() or d['truth'].shape != (*eligible.shape, 2) or not np.isfinite(d['truth'][eligible]).all():
        raise ValueError('마스크 재표본화에 필요한 fit 정답이 잘못되었습니다.')
    # 같은 링크의 관측소를 함께 가리고, 기존 자연 결측·Task 2 문맥 결측은 유지합니다.
    # shortcut: NPZ에 Task 2 history 제외 플래그가 없어 fit history 관측도 재마스킹합니다; 정확한 분포 비교 시 플래그를 캐시에 추가하세요.
    rate = (0.20, 0.30, 0.50)[int(d['regime'])]
    draws = rng.random((len(eligible), len(context['panel'].links)))
    target = (draws[:, context['mapping']] < rate) & eligible
    if not target.any():
        first_time, first_station = (indices[0] for indices in np.nonzero(eligible))
        target[first_time] = (context['mapping'] == context['mapping'][first_station]) & eligible[first_time]
    values, visible, aux = (d[key].copy() for key in ('values', 'visible', 'aux'))
    values[eligible, :2] = d['truth'][eligible]
    visible[eligible, :2] = True
    values[target, :2], visible[target, :2] = 0., False
    aux[..., 1:4] = visible
    dates = pd.to_datetime(d['timestamps'], utc=True, format='ISO8601')
    elapsed = np.asarray((dates - dates[0]).total_seconds() / 60, np.float32)
    baseline = []
    for channel in range(2):
        observed = np.where(visible[..., channel], values[..., channel], np.nan)
        base, left_age, right_age = nearest_features(observed, aux[..., 8+channel], elapsed)
        baseline.append(base)
        for offset, age in enumerate((left_age, right_age)):
            aux[..., 4 + channel * 2 + offset] = np.nan_to_num(age / 360, nan=4.).clip(0, 4)
    physical = d['observed'].copy()
    units = np.stack((np.full_like(context['lanes'], 25.), 600 * context['lanes']), -1)
    physical[eligible] = (d['truth'] * units)[eligible]
    physical[target] = 0.
    ti, si = np.nonzero(target)
    result = {key: value for key, value in d.items() if key != '_transition'}
    result.update(values=values, visible=visible, aux=aux, base=np.stack(baseline, -1).astype(np.float32),
                  target=target, observed=physical, ti=ti, si=si)
    if visible[target, :2].any() or not np.isfinite(result['base']).all():
        raise ValueError('재표본화한 마스크에 정답 노출 또는 잘못된 보간값이 있습니다.')
    return result


def window_starts(length):
    if length <= WINDOW:
        return [0]
    return sorted(set(range(0, length - WINDOW + 1, STRIDE)) | {length - WINDOW})

def masked_softmax(scores, valid):
    valid = valid.to(dtype=torch.bool)
    weights = scores.float().masked_fill(~valid, torch.finfo(torch.float32).min).softmax(-1)
    weights = torch.where(valid, weights, 0.)
    return weights / weights.sum(-1, keepdim=True).clamp_min(1e-8)


class AMSTEncoder(nn.Module):
    def __init__(self, station_count, panel_count, family_count):
        super().__init__()
        self.value_weight = nn.Parameter(torch.empty(3, WIDTH))
        self.value_bias = nn.Parameter(torch.zeros(3, WIDTH))
        self.missing_token = nn.Parameter(torch.empty(3, WIDTH))
        self.slot = nn.Embedding(288, WIDTH)
        self.weekday = nn.Embedding(7, WIDTH)
        self.relative = nn.Embedding(288, WIDTH)
        self.station = nn.Embedding(station_count, WIDTH)
        self.panel = nn.Embedding(panel_count, WIDTH)
        self.family = nn.Embedding(family_count, WIDTH)
        self.direction = nn.Embedding(4, WIDTH)
        self.regime = nn.Embedding(3, WIDTH)
        self.aux = nn.Linear(len(AUX_NAMES), WIDTH)
        self.base_embedding = nn.Linear(2, WIDTH)
        self.norm = nn.LayerNorm(WIDTH)
        self.dropout = nn.Dropout(DROPOUT)
        nn.init.xavier_uniform_(self.value_weight)
        nn.init.normal_(self.missing_token, std=0.02)

    def forward(self, values, visible, aux, base, slot, weekday, regime, ct):
        # values/visible: B×T×station×3; 결측 placeholder는 여기서 제외합니다.
        tokens = values[..., None] * self.value_weight + self.value_bias
        tokens = torch.where(visible[..., None], tokens, self.missing_token)
        h = tokens.sum(-2) / np.sqrt(3.)
        t = len(values[0])
        time_embedding = self.slot(slot) + self.weekday(weekday) + self.relative(torch.arange(t, device=values.device))[None]
        node_embedding = (self.station(ct['station_codes']) + self.panel(ct['panel_code'])
                          + self.family(ct['family_code']) + self.direction(ct['direction_code']))
        h = h + self.aux(aux) + self.base_embedding(base) + time_embedding[:, :, None] + node_embedding[None, None] + self.regime(regime)[:, None, None]
        return self.dropout(self.norm(h)).permute(0, 2, 1, 3), node_embedding


class MaskTemporalAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(WIDTH, 3 * WIDTH, bias=False)
        self.project = nn.Linear(WIDTH, WIDTH)
        self.norm = nn.LayerNorm(WIDTH)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, h, visible, residual_attention):
        b, s, t, d = h.shape
        q, k, v = self.qkv(h).reshape(b, s, t, 3, HEADS, d // HEADS).permute(3, 0, 1, 4, 2, 5).unbind(0)
        scores = (q.float() @ k.float().transpose(-1, -2)) / np.sqrt(d // HEADS)
        if residual_attention is not None:
            scores = scores + residual_attention
        key_valid = visible.any(-1).permute(0, 2, 1)[:, :, None, None, :]
        weights = masked_softmax(scores, key_valid)
        attended = (weights.to(v.dtype) @ v).transpose(2, 3).reshape(b, s, t, d)
        return self.norm(h + self.dropout(self.project(attended))), scores


class MaskSpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.summary = nn.Linear(2 * WIDTH, GRAPH_DIM)
        self.position = nn.Linear(WIDTH, GRAPH_DIM)
        self.qk = nn.Linear(GRAPH_DIM, 2 * CHEB_K * 16, bias=False)
        self.mask_pair = nn.Linear(GRAPH_DIM, 2 * CHEB_K * 8, bias=False)
        self.norm = nn.LayerNorm(GRAPH_DIM)

    def forward(self, h, visible, node_embedding):
        b, s, t, d = h.shape
        valid = visible.any(-1).permute(0, 2, 1).float()
        observed_summary = (h * valid[..., None]).sum(2) / valid.sum(2).clamp_min(1)[..., None]
        summary = self.summary(torch.cat((observed_summary, h.mean(2)), -1))
        # MagiNet의 관측 문맥 기반 missing spatial graph를 유지합니다.
        similarity = F.relu(summary.float() @ summary.float().transpose(-1, -2))
        missing_graph = (similarity.softmax(-1) > MGL_THRESHOLD).float()
        positioned = self.norm(summary + self.position(node_embedding)[None])
        q, k = self.qk(positioned).reshape(b, s, 2, CHEB_K, 16).permute(2, 0, 3, 1, 4).unbind(0)
        scores = (q.float() @ k.float().transpose(-1, -2)) / 4.
        a, z = self.mask_pair(positioned).reshape(b, s, 2, CHEB_K, 8).permute(2, 0, 3, 1, 4).unbind(0)
        pair_mask = torch.tanh((a.float() @ z.float().transpose(-1, -2)) / np.sqrt(8.))
        reliability = 0.25 + 0.75 * valid.mean(2)
        scores = scores + missing_graph[:, None] * pair_mask + reliability.log()[:, None, None, :]
        return scores.softmax(-1)


class MaskAwareChebConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.empty(CHEB_K, WIDTH, WIDTH))
        nn.init.xavier_uniform_(self.theta)

    def forward(self, h, spatial_attention, supports):
        weights = spatial_attention * supports[None]
        output = torch.einsum('bkij,bjtd,kdf->bitf', weights.to(h.dtype), h, self.theta.to(h.dtype))
        return F.relu(output)


def observation_context(visible, aux, neighbors):
    """Public observations only; physical neighbors exclude the station itself."""
    with torch.autocast(device_type=visible.device.type, enabled=False):
        joint = visible[..., :2].all(-1).float()
        counts = torch.einsum('ij,btj->bti', neighbors.float(), joint)
        degree = neighbors.float().sum(-1)[None, None]
        features = torch.cat((visible.float(), (counts / (1 + counts))[..., None],
                              (counts / degree.clamp_min(1))[..., None],
                              (degree > 0).expand_as(counts)[..., None].float(),
                              aux[..., 4:8].float().clamp(0, 4) / 4), -1)
    return features.permute(0, 2, 1, 3)


def gather_visible_spatial(h, spatial_attention, visible):
    with torch.autocast(device_type=h.device.type, enabled=False):
        weights = spatial_attention.float().mean(1)
        valid = visible.any(-1).permute(0, 2, 1).float()
        numerator = torch.einsum('bij,bjtd->bitd', weights, h.float() * valid[..., None])
        denominator = torch.einsum('bij,bjt->bit', weights, valid)
        return (numerator / denominator.clamp_min(1e-8)[..., None]).to(h.dtype)


class LearnedSpatialConv(nn.Module):
    def __init__(self, contextual=False):
        super().__init__()
        self.project = nn.Linear(WIDTH, WIDTH, bias=False)
        self.gate = nn.Parameter(torch.tensor(-3.))
        self.context_gate = nn.Linear(10, 1, bias=False) if contextual else None
        if self.context_gate is not None:
            nn.init.zeros_(self.context_gate.weight)

    def forward(self, h, spatial_attention, visible, context=None):
        # 관측 문맥으로 학습한 관계를 재사용하며 물리 supports를 곱하지 않습니다.
        gathered = gather_visible_spatial(h, spatial_attention, visible)
        gate = self.gate if self.context_gate is None else self.gate + self.context_gate(context).float()
        return gate.sigmoid() * self.project(gathered)


class ChannelSpatialReadout(nn.Module):
    """Channel-specific spatial residuals; the shared backbone has no extra learned graph."""
    def __init__(self):
        super().__init__()
        self.attention = MaskSpatialAttention()
        self.project = nn.Linear(WIDTH, WIDTH, bias=False)
        self.correction = nn.Linear(WIDTH, 2)
        self.gate = nn.Linear(10, 2)
        nn.init.zeros_(self.correction.weight)
        nn.init.zeros_(self.correction.bias)
        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, -3.)

    def forward(self, h, visible, node_embedding, context):
        attention = self.attention(h, visible, node_embedding)
        gathered = gather_visible_spatial(h, attention, visible)
        correction = self.correction(F.gelu(self.project(gathered))).float()
        # 관측소 전체가 비어 있는 시각에는 bias만으로 공간 보정을 만들지 않습니다.
        available = visible.any(-1).any(-1)[:, None, :, None]
        return correction * self.gate(context).float().sigmoid() * available


class GatedTemporalUnit(nn.Module):
    def __init__(self, kernel):
        super().__init__()
        self.conv = nn.Conv2d(WIDTH, 2 * WIDTH, kernel_size=(1, kernel), padding=(0, kernel // 2))

    def forward(self, h):
        a, b = self.conv(h).chunk(2, dim=1)
        return a.tanh() * b.sigmoid()


class MASTBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.temporal_attention = MaskTemporalAttention()
        self.spatial_attention = MaskSpatialAttention()
        self.spatial_conv = MaskAwareChebConv()
        self.temporal_direct = TEMPORAL_DIRECT
        self.learned_spatial = (LearnedSpatialConv(contextual=SPATIAL_MODE == 'context')
                                if LEARNED_GRAPH and SPATIAL_MODE != 'channel' else None)
        self.gates = nn.ModuleList(GatedTemporalUnit(k) for k in (3, 5, 7))
        self.time_mix = nn.Conv2d(3 * WIDTH, WIDTH, kernel_size=1)
        self.norm = nn.LayerNorm(WIDTH)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, h, visible, node_embedding, supports, residual_attention, context=None):
        attended, residual_attention = self.temporal_attention(h, visible, residual_attention)
        spatial_attention = self.spatial_attention(attended, visible, node_embedding)
        signal = attended if self.temporal_direct else h
        spatial = self.spatial_conv(signal, spatial_attention, supports)
        if self.learned_spatial is not None:
            spatial = spatial + self.learned_spatial(signal, spatial_attention, visible, context)
        channels = spatial.permute(0, 3, 1, 2)
        gated = self.time_mix(torch.cat([gate(channels) for gate in self.gates], dim=1)).permute(0, 2, 3, 1)
        output = self.norm(h + self.dropout(F.relu(spatial + gated)))
        return output, residual_attention


class MagiNetTask13(nn.Module):
    def __init__(self, station_count, panel_count, family_count, normalization):
        super().__init__()
        mean = torch.as_tensor(normalization['mean'], dtype=torch.float32)
        std = torch.as_tensor(normalization['std'], dtype=torch.float32)
        if mean.shape != (3,) or std.shape != (3,) or not torch.isfinite(mean).all() or not torch.isfinite(std).all() or (std <= 0).any():
            raise ValueError('입력 표준화의 평균·표준편차가 잘못되었습니다.')
        self.register_buffer('input_mean', mean)
        self.register_buffer('input_std', std)
        self.AMSTenc = AMSTEncoder(station_count, panel_count, family_count)
        self.MASTdec = nn.ModuleList(MASTBlock() for _ in range(BLOCKS))
        self.fuse = nn.Linear((BLOCKS + 1) * WIDTH, WIDTH)
        self.readout = nn.GRU(WIDTH, WIDTH // 2, batch_first=True, bidirectional=True)
        self.speed_head = nn.Sequential(nn.Linear(WIDTH, WIDTH // 2), nn.GELU(), nn.Linear(WIDTH // 2, 1))
        self.flow_head = nn.Sequential(nn.Linear(WIDTH, WIDTH // 2), nn.GELU(), nn.Linear(WIDTH // 2, 1))
        for head in (self.speed_head, self.flow_head):
            # 보정량은 양·음 모두 허용하며 시작 출력은 기존 보간과 같습니다.
            nn.init.zeros_(head[-1].weight)
            nn.init.zeros_(head[-1].bias)
        self.spatial_mode = SPATIAL_MODE
        self.channel_spatial = ChannelSpatialReadout() if SPATIAL_MODE == 'channel' else None

    def forward(self, values, visible, aux, base, slot, weekday, regime, ct):
        normalized = torch.where(visible, (values.float() - self.input_mean) / self.input_std, 0.)
        normalized_base = (base.float() - self.input_mean[:2]) / self.input_std[:2]
        normalized_aux = aux.float().clone()
        normalized_aux[..., 8:11] = (normalized_aux[..., 8:11] - self.input_mean) / self.input_std
        h, node_embedding = self.AMSTenc(normalized, visible, normalized_aux, normalized_base, slot, weekday, regime, ct)
        context = observation_context(visible, aux, ct['neighbors']) if self.spatial_mode != 'legacy' else None
        outputs, residual_attention = [h], None
        for block in self.MASTdec:
            h, residual_attention = block(h, visible, node_embedding, ct['supports'], residual_attention, context)
            outputs.append(h)
        h = F.gelu(self.fuse(torch.cat(outputs, dim=-1)))
        b, s, t, d = h.shape
        # cuDNN의 BF16 RNN 지원 여부와 무관하게 시간 readout은 FP32로 계산합니다.
        with torch.autocast(device_type=h.device.type, enabled=False):
            h, _ = self.readout(h.float().reshape(b * s, t, d))
        h = h.reshape(b, s, t, d).permute(0, 2, 1, 3)
        correction = torch.cat((self.speed_head(h), self.flow_head(h)), -1).float()
        if self.channel_spatial is not None:
            correction = correction + self.channel_spatial(h.permute(0, 2, 1, 3), visible,
                                                           node_embedding, context).permute(0, 2, 1, 3)
        # 입력 표준화와 별개로 출력·State 손실은 /25·/600 단위를 유지합니다.
        return (base.float() + correction).clamp_min(0.)


def tensor_context(context, device=None):
    device = DEVICE if device is None else device
    panel = context['panel']
    mapping = torch.as_tensor(context['mapping'], dtype=torch.long, device=device)
    neighbors = np.asarray(context['station_graph'], dtype=np.float32).copy()
    np.fill_diagonal(neighbors, 0)
    return dict(station_codes=torch.as_tensor(context['station_codes'], dtype=torch.long, device=device),
                panel_code=torch.tensor(context['panel_code'], dtype=torch.long, device=device),
                family_code=torch.tensor(context['family_code'], dtype=torch.long, device=device),
                direction_code=torch.tensor(context['direction_code'], dtype=torch.long, device=device),
                supports=torch.as_tensor(chebyshev_supports(context['station_graph']), device=device),
                neighbors=torch.as_tensor(neighbors, device=device),
                matrix=F.one_hot(mapping, num_classes=len(panel.links)).float(),
                lanes=torch.as_tensor(context['lanes'], device=device),
                link_lanes=torch.as_tensor(panel.lanes, device=device),
                lengths=torch.as_tensor(panel.lengths, device=device), free=torch.as_tensor(panel.free_speed, device=device),
                capacity=torch.as_tensor(panel.capacity, device=device), jam=torch.as_tensor(panel.jam_density, device=device))


def window_batch(d, starts, device=None):
    device = DEVICE if device is None else device
    lengths = [min(WINDOW, len(d['values']) - start) for start in starts]
    if len(set(lengths)) != 1:
        raise ValueError('한 batch의 시간 길이가 다릅니다.')
    return {key: torch.as_tensor(np.stack([d[key][start:start+length] for start, length in zip(starts, lengths)]),
                                 device=device, dtype=dtype)
            for key, dtype in (('values', torch.float32), ('visible', torch.bool), ('aux', torch.float32),
                                ('base', torch.float32), ('slot', torch.long), ('weekday', torch.long))} | {
               'regime': torch.full((len(starts),), int(d['regime']), dtype=torch.long, device=device)}


def panel_batch_size(context):
    name = context['panel'].name
    if name in PANEL_BATCH:
        return PANEL_BATCH[name]
    if BATCH_POLICY == 'adaptive':
        return 8 if len(context['mapping']) <= 128 else 4
    return BATCH_SIZE


@torch.inference_mode()
def maginet_predict(model, d, ct, batch_size=None):
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    model.eval()
    device = next(model.parameters()).device
    sums = np.zeros_like(d['base'], np.float64)
    weights = np.zeros((len(d['base']), 1, 1), np.float64)
    starts = window_starts(len(d['values']))
    for offset in range(0, len(starts), batch_size):
        chunk = starts[offset:offset+batch_size]
        batch = window_batch(d, chunk, device)
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP and device.type == 'cuda'):
            outputs = model(**batch, ct=ct).float().cpu().numpy()
        for start, output in zip(chunk, outputs):
            w = np.hanning(len(output) + 2)[1:-1, None, None]
            sums[start:start+len(output)] += output * w
            weights[start:start+len(output)] += w
    if (weights == 0).any():
        raise ValueError('겹치는 창 예측에 누락 시각이 있습니다.')
    result = (sums / weights).astype(np.float32)
    if not np.isfinite(result).all() or (result < 0).any():
        raise FloatingPointError('MagiNet 출력이 잘못되었습니다.')
    return result

def observability_mask(context, counts, ramp_valid, affected, consecutive):
    panel = context['panel']
    present = (counts > 0).any(0)
    topology_nodes = context['topology_nodes']
    observable = present & np.array([i in topology_nodes for i in range(len(panel.links))])
    for a, b in panel.edges.T:
        if a in topology_nodes and not present[a]:
            observable[b] = False
        if b in topology_nodes and not present[b]:
            observable[a] = False
    if context['mode'] == 'C':
        for link, _ in panel.ramp_map.values():
            observable[link] = False
    transition = (counts[:-1] > 0) & (counts[1:] > 0) & observable[None] & consecutive[:, None]
    if context['mode'] in ('A', 'B'):
        transition &= ramp_valid[:-1]
    return transition, (affected[:-1] | affected[1:])

def completed_link_fields(prediction_physical, d, context):
    values = d['observed'].copy()
    ti, si = d['ti'], d['si']
    values[ti, si, 0] = prediction_physical[:, 0]
    values[ti, si, 1] = prediction_physical[:, 1] * context['lanes'][si]
    links, counts = link_mean_numpy(values, d['active'], context['mapping'], len(context['panel'].links))
    affected = link_mean_numpy(d['target'][..., None].astype(float), d['active'],
                              context['mapping'], len(context['panel'].links))[0][..., 0] > 0
    return links, counts, affected

def physics_totals(prediction, d, context):
    panel = context['panel']
    links, counts, affected = completed_link_fields(prediction, d, context)
    truth = d['truth'][d['ti'], d['si']] * SCALE
    reference, _, _ = completed_link_fields(truth, d, context)
    n = links[..., 1] / np.maximum(links[..., 0], 1) * panel.lengths
    true_n = reference[..., 1] / np.maximum(reference[..., 0], 1) * panel.lengths
    good_n = (counts > 0) & affected
    error_n = (n - true_n)[good_n]
    transition, target_transition = observability_mask(context, counts, d['ramp_valid'], affected, d['consecutive'])
    error_dn_all = (np.diff(n, axis=0) - np.diff(true_n, axis=0))[transition]
    truth_dn_all = np.diff(true_n, axis=0)[transition]
    error_dn_target = (np.diff(n, axis=0) - np.diff(true_n, axis=0))[transition & target_transition]
    density = links[..., 1] / np.maximum(links[..., 0], 1)
    critical = panel.capacity / panel.free_speed
    wave = panel.capacity / np.maximum(panel.jam_density - critical, 1e-6)
    qfd = np.where(density <= critical, panel.free_speed * density,
                   wave * (panel.jam_density - density)).clip(0)
    good_fd = counts > 0
    fd_error = ((links[..., 1] - qfd) / panel.lanes)[good_fd]
    q_lane = (links[..., 1] / panel.lanes)[good_fd]
    fraction = link_mean_numpy(d['target'][..., None].astype(float), d['active'],
                               context['mapping'], len(panel.links))[0][..., 0]
    submitted = good_fd & (fraction > 0.5)
    return dict(n_sae=float(np.abs(error_n).sum()), n_sse=float(np.square(error_n).sum()), n_count=len(error_n),
                dn_sae=float(np.abs(error_dn_target).sum()), dn_sse=float(np.square(error_dn_target).sum()),
                dn_count=len(error_dn_target), lwr_num=float(np.abs(error_dn_all).sum()),
                lwr_den=float(np.abs(truth_dn_all).sum()), lwr_count=len(error_dn_all),
                fd_num=float(np.abs(fd_error).sum()), fd_den=float(np.abs(q_lane).sum()),
                fd_count=int(good_fd.sum()), submitted_count=int(submitted.sum()),
                low_flow=int((links[..., 1][submitted] < 50).sum()))

def state_totals(truth, prediction):
    error = np.asarray(prediction, np.float64) - truth
    return dict(count=len(error), speed_sse=float(np.square(error[:, 0]).sum()),
                flow_sse=float(np.square(error[:, 1]).sum()), speed_sae=float(np.abs(error[:, 0]).sum()),
                flow_sae=float(np.abs(error[:, 1]).sum()))

def summarize(records):
    daily = pd.DataFrame(records)
    if daily.empty:
        raise ValueError('평가 대상이 없습니다.')
    keys = ['candidate', 'alpha', 'panel', 'regime']
    sums = daily.groupby(keys, as_index=False).sum(numeric_only=True)
    expected = {(p, r) for p in daily.panel.unique() for r in range(3)}
    for _, part in sums.groupby('candidate'):
        if set(zip(part.panel, part.regime)) != expected or (part['count'] == 0).any():
            raise ValueError('평가의 panel×regime이 누락되었습니다.')
    sums['speed_rmse_kmh'] = np.sqrt(sums.speed_sse / sums['count'])
    sums['flow_rmse_per_lane'] = np.sqrt(sums.flow_sse / sums['count'])
    sums['speed_mae_kmh'] = sums.speed_sae / sums['count']
    sums['flow_mae_per_lane'] = sums.flow_sae / sums['count']
    sums['State'] = 0.54 * np.maximum(0, 1 - sums.speed_rmse_kmh / 25) + 0.46 * np.maximum(0, 1 - sums.flow_rmse_per_lane / 600)
    if 'lwr_num' in sums:
        sums['N_MAE_proxy'] = sums.n_sae / sums.n_count.replace(0, np.nan)
        sums['DN_MAE_proxy'] = sums.dn_sae / sums.dn_count.replace(0, np.nan)
        sums['LWR_proxy'] = np.maximum(0, 1 - sums.lwr_num / (sums.lwr_den + 1e-6))
        sums['FD_proxy'] = np.maximum(0, 1 - sums.fd_num / (sums.fd_den + 1e-6))
        sums.loc[sums.lwr_count == 0, 'LWR_proxy'] = 0.
        sums.loc[sums.fd_count == 0, 'FD_proxy'] = 0.
        sums['low_flow_fraction'] = sums.low_flow / sums.submitted_count.replace(0, np.nan)
        sums.loc[sums.low_flow_fraction > 0.20, 'FD_proxy'] = 0.
        sums['Physics_proxy'] = (sums.FD_proxy + 2 * sums.LWR_proxy) / 3
    metrics = ['State', 'speed_rmse_kmh', 'flow_rmse_per_lane', 'speed_mae_kmh', 'flow_mae_per_lane']
    metrics += [n for n in ('N_MAE_proxy', 'DN_MAE_proxy', 'FD_proxy', 'LWR_proxy', 'Physics_proxy') if n in sums]
    rows = []
    for name, part in sums.groupby('candidate', sort=False):
        per_panel = part.groupby('panel')[metrics].mean()
        row = dict(candidate=name, alpha=float(part.alpha.iloc[0]),
                   **{m: macro_mean(per_panel[m].to_dict()) for m in metrics})
        if 'FD_proxy' in row:
            row['selection_proxy'] = sum(row[k] * w for k, w in SELECT_WEIGHTS.items()) / sum(SELECT_WEIGHTS.values())
        rows.append(row)
    return pd.DataFrame(rows), sums, daily

def training_transitions(d, context):
    if '_transition' not in d:
        _, counts, affected = completed_link_fields(d['truth'][d['ti'], d['si']] * SCALE, d, context)
        transition, target_transition = observability_mask(context, counts, d['ramp_valid'], affected, d['consecutive'])
        d['_transition'] = transition & target_transition
    return d['_transition']


def joint_loss(output, d, starts, context, ct, physics_factor):
    device, length = output.device, output.shape[1]
    take = lambda key: torch.as_tensor(np.stack([d[key][s:s+length] for s in starts]), device=device)
    target, truth = take('target').bool(), take('truth').float()
    if not target.any():
        raise ValueError('학습 창에 대상 셀이 없습니다.')
    state_error = (output.float() - truth)[target]
    state = 0.54 * state_error[:, 0].square().mean() + 0.46 * state_error[:, 1].square().mean()
    zero = state * 0.
    losses = dict(State=state, N=zero, dN=zero, FD=zero)
    if physics_factor > 0:
        active, observed = take('active').float(), take('observed').float()
        units = torch.stack((torch.full_like(ct['lanes'], 25.), 600 * ct['lanes']), -1)
        predicted = torch.where(target[..., None], output.float() * units, observed)
        reference = torch.where(target[..., None], truth * units, observed)
        counts = torch.einsum('bts,sl->btl', active, ct['matrix'])
        aggregate = lambda values: torch.einsum('btsc,sl->btlc', values * active[..., None], ct['matrix']) / counts.clamp_min(1)[..., None]
        links, true_links = aggregate(predicted), aggregate(reference)
        affected = torch.einsum('bts,sl->btl', target.float(), ct['matrix']) > 0
        good_n = (counts > 0) & affected
        n = links[..., 1] / links[..., 0].clamp_min(1) * ct['lengths']
        true_n = true_links[..., 1] / true_links[..., 0].clamp_min(1) * ct['lengths']
        losses['N'] = (n - true_n)[good_n].abs().sum() / (true_n[good_n].abs().sum() + 1.)
        transition = training_transitions(d, context)
        good_dn = torch.as_tensor(np.stack([transition[s:s+length-1] for s in starts]), device=device)
        dn, true_dn = torch.diff(n, dim=1), torch.diff(true_n, dim=1)
        losses['dN'] = (dn - true_dn)[good_dn].abs().sum() / (true_dn[good_dn].abs().sum() + 1.)
        if LOSS_WEIGHTS['FD'] > 0:
            density = links[..., 1] / links[..., 0].clamp_min(1)
            critical = ct['capacity'] / ct['free']
            wave = ct['capacity'] / (ct['jam'] - critical).clamp_min(1e-6)
            qfd = torch.where(density <= critical, ct['free'] * density, wave * (ct['jam'] - density)).clamp_min(0)
            error = ((links[..., 1] - qfd) / ct['link_lanes'])[good_n]
            losses['FD'] = error.abs().sum() / ((links[..., 1] / ct['link_lanes'])[good_n].detach().abs().sum() + 1.)
    total = state + physics_factor * sum(LOSS_WEIGHTS[k] * losses[k] for k in LOSS_WEIGHTS)
    if not torch.isfinite(total):
        raise FloatingPointError('학습 손실에 NaN/Inf가 있습니다.')
    return total, {k: float(v.detach()) for k, v in losses.items()}


def error_slice_axes(d, context):
    ti, si = d['ti'], d['si']
    ages = d['aux'][ti, si, 4:6]
    gap = np.rint(ages.sum(-1) * 360)
    gap_bucket = np.select([np.any(ages >= 4., axis=-1), gap <= 10, gap <= 30, gap <= 60],
                           ['one_sided', '0-10', '10-30', '30-60'], default='60+')
    neighbors = context['station_graph'].astype(np.int32).copy()
    np.fill_diagonal(neighbors, 0)
    counts = (d['visible'][..., :2].all(-1).astype(np.int32) @ neighbors.T)[ti, si]
    neighbor_bucket = np.select([counts == 0, counts <= 2, counts <= 5], ['0', '1-2', '3-5'], default='6+')
    return {'hour_utc': d['slot'][ti] // 12, 'speed_gap_minutes': gap_bucket,
            'station': np.asarray(context['targets'].stations)[si], 'observed_neighbors': neighbor_bucket}


def error_slice_records(truth, prediction, axes, metadata):
    rows = []
    for dimension, buckets in axes.items():
        for bucket in np.unique(buckets):
            selected = buckets == bucket
            rows.append(dict(metadata, dimension=dimension, bucket=str(bucket),
                             **state_totals(truth[selected], prediction[selected])))
    return rows


def summarize_error_slices(records):
    keys = ['candidate', 'alpha', 'panel', 'regime', 'dimension', 'bucket']
    frame = pd.DataFrame(records).groupby(keys, as_index=False).sum(numeric_only=True)
    frame['speed_rmse_kmh'] = np.sqrt(frame.speed_sse / frame['count'])
    frame['flow_rmse_per_lane'] = np.sqrt(frame.flow_sse / frame['count'])
    frame['speed_mae_kmh'] = frame.speed_sae / frame['count']
    frame['flow_mae_per_lane'] = frame.flow_sae / frame['count']
    frame['slice_State'] = (0.54 * np.maximum(0, 1 - frame.speed_rmse_kmh / 25)
                            + 0.46 * np.maximum(0, 1 - frame.flow_rmse_per_lane / 600))
    return frame


def evaluate(entries, contexts, tensors, model, period, alphas, folder, collect_diagnostics=False,
             postprocessors=None):
    started, records, slice_records = time.perf_counter(), [], []
    selected = [e for e in entries if e['period'] == period]
    if not selected:
        raise ValueError(f'평가 날짜가 없습니다: {period}')
    for number, entry in enumerate(selected, 1):
        d, context = load_day(entry['path']), contexts[entry['panel']]
        output = maginet_predict(model, d, tensors[entry['panel']], batch_size=panel_batch_size(context))
        if postprocessors:
            output = postprocessors[entry['panel']](d, output)
        ti, si = d['ti'], d['si']
        truth, base, prediction = d['truth'][ti, si] * SCALE, d['base'][ti, si] * SCALE, output[ti, si] * SCALE
        axes = error_slice_axes(d, context) if collect_diagnostics else None
        for alpha in alphas:
            mixed = (1 - alpha) * base + alpha * prediction
            records.append(dict(candidate=f'alpha_{float(alpha)!r}', alpha=float(alpha),
                                panel=entry['panel'], date=entry['date'],
                                regime=entry['regime'], **state_totals(truth, mixed), **physics_totals(mixed, d, context)))
            if collect_diagnostics:
                slice_records.extend(error_slice_records(truth, mixed, axes,
                    {key: records[-1][key] for key in ('candidate', 'alpha', 'panel', 'date', 'regime')}))
        if number == 1 or number % 20 == 0 or number == len(selected):
            print(f'평가 {period}: {number}/{len(selected)}일', flush=True)
    macro, grouped, daily = summarize(records)
    folder.mkdir(parents=True, exist_ok=True)
    for name, frame in (('macro', macro), ('panel_regime', grouped), ('daily', daily)):
        frame.to_csv(folder / f'{period}_{name}.csv', index=False)
    if collect_diagnostics:
        summarize_error_slices(slice_records).to_csv(folder / f'{period}_error_slices.csv', index=False)
    if (grouped.lwr_count == 0).any():
        print('주의: LWR 대리 평가 대상이 없는 그룹이 있습니다. panel_regime의 lwr_count를 확인해 주세요.')
    return macro, grouped, daily, time.perf_counter() - started


def notebook_signature(contexts, entries, normalization):
    settings = dict(architecture='maginet-task13-residual-v2',
                    input_normalization={k: normalization[k] for k in ('mode', 'mean', 'std')},
                    output_scale=SCALE.tolist(), aux_names=list(AUX_NAMES), seed=SEED, window=WINDOW, stride=STRIDE,
                    width=WIDTH, blocks=BLOCKS, heads=HEADS, cheb_k=CHEB_K, graph_dim=GRAPH_DIM,
                    dropout=DROPOUT, mgl=MGL_THRESHOLD, batch=BATCH_SIZE, updates=UPDATES_PER_GROUP,
                    lr=LR, weight_decay=WEIGHT_DECAY, patience=PATIENCE,
                    warmup=WARMUP_EPOCHS, ramp=PHYSICS_RAMP_EPOCHS, loss_weights=LOSS_WEIGHTS,
                    select_weights=SELECT_WEIGHTS, periods=PERIODS, release=str(RELEASE_ROOT.resolve()),
                    training_dates={n: sorted(e['date'] for e in entries if e['panel'] == n and e['period'] == 'fit')
                                    for n in contexts},
                    panels={name: dict(stations=c['targets'].stations, links=c['targets'].links,
                                      station_codes=c['station_codes'].tolist(), panel_code=c['panel_code'],
                                      family=c['family_code'], direction=c['direction_code'], mode=c['mode'],
                                      dates=c['dates'], profile_hash=hashlib.sha256(c['profile'].tobytes()).hexdigest())
                            for name, c in contexts.items()})
    if TEMPORAL_DIRECT or MASK_RESAMPLE or LEARNED_GRAPH:
        settings['improvements'] = dict(temporal_direct=TEMPORAL_DIRECT, learned_graph=LEARNED_GRAPH,
                                        mask_resample='fit-link-v2' if MASK_RESAMPLE else None)
    if SPATIAL_MODE != 'legacy':
        settings['improvements']['spatial_mode'] = SPATIAL_MODE
    if CHECKPOINT_METRIC != 'selection_proxy':
        settings['checkpoint_metric'] = CHECKPOINT_METRIC
    return hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest(), settings


def run_signature(contexts, entries, normalization):
    _, settings = notebook_signature(contexts, entries, normalization)
    settings.pop('release')
    settings['format_version'] = 3
    settings['days_per_regime'] = DAYS_PER_REGIME
    # 기본 fixed 설정은 이전 로컬 체크포인트의 서명을 유지합니다.
    if BATCH_POLICY != 'fixed' or PANEL_BATCH:
        settings.update(batch_policy=BATCH_POLICY, panel_batch=dict(PANEL_BATCH),
                        panel_batches={n: panel_batch_size(c) for n, c in contexts.items()})
    inputs = []
    for entry in sorted(entries, key=lambda e: (e['panel'], e['period'], e['date'])):
        with zipfile.ZipFile(entry['path']) as archive:
            arrays = sorted((m.filename, m.file_size, m.CRC) for m in archive.infolist())
        inputs.append((entry['panel'], entry['period'], entry['date'], arrays))
    settings['input_hash'] = hashlib.sha256(json.dumps(inputs).encode()).hexdigest()
    settings['network_hashes'] = network_hashes(contexts)
    return hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest(), settings


def network_hashes(contexts):
    hashes = {}
    for name, context in contexts.items():
        digest = hashlib.sha256()
        for values in (context['station_graph'], context['mapping'], context['lanes'],
                       context['panel'].free_speed, context['panel'].lanes, context['panel'].lengths,
                       context['panel'].capacity, context['panel'].jam_density, context['panel'].edges):
            array = np.ascontiguousarray(values)
            digest.update(str((array.shape, array.dtype)).encode())
            digest.update(array.tobytes())
        digest.update(json.dumps(context['panel'].ramp_map, sort_keys=True).encode())
        hashes[name] = digest.hexdigest()
    return hashes


def save_checkpoint(payload, path):
    temporary = path.with_suffix('.pt.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)


def optimizer_step(model, optimizer, scaler):
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=not scaler.is_enabled())
    scaler.step(optimizer)
    scaler.update()


def load_best_maginet(entries, contexts, folder):
    best_path = folder / 'best_maginet.pt'
    if not best_path.exists() or not (folder / 'history.csv').exists():
        raise FileNotFoundError(f'저장 모델·history.csv가 필요합니다. 먼저 TRAIN_MODEL=True로 학습해 주세요: {folder}')
    normalization = fit_input_normalization(entries, CACHE_METADATA_DIR / 'normalization', legacy_folder=folder)
    signature, _ = run_signature(contexts, entries, normalization)
    saved = torch.load(best_path, map_location=DEVICE, weights_only=True)
    if saved['signature'] != signature:
        raise ValueError('저장 모델과 설정·패널·기간이 다릅니다. 해당 실행의 설정을 복원해 주세요.')
    model = MagiNetTask13(max(int(c['station_codes'].max()) for c in contexts.values()) + 1,
                         max(c['panel_code'] for c in contexts.values()) + 1,
                         max(c['family_code'] for c in contexts.values()) + 1, normalization).to(DEVICE)
    model.load_state_dict(saved['model'])
    model.eval()
    print({'loaded_best_epoch': saved['epoch'], 'checkpoint': str(best_path)})
    return model, pd.read_csv(folder / 'history.csv')


def train_maginet(entries, contexts, tensors, folder, epochs=None, resume=None):
    epochs = EPOCHS if epochs is None else epochs
    resume = RESUME if resume is None else resume
    reset_seed()
    folder.mkdir(parents=True, exist_ok=True)
    best_path, last_path = folder / 'best_maginet.pt', folder / 'last_maginet.pt'
    if not resume and any(p.exists() for p in (best_path, last_path, folder / 'best_state_maginet.pt')):
        raise FileExistsError(f'기존 학습을 보호합니다. RESUME=True 또는 새 REPORT_DIR를 사용해 주세요: {folder}')
    normalization = fit_input_normalization(entries, CACHE_METADATA_DIR / 'normalization', legacy_folder=folder)
    signature, settings = run_signature(contexts, entries, normalization)
    station_count = max(int(c['station_codes'].max()) for c in contexts.values()) + 1
    panel_count = max(c['panel_code'] for c in contexts.values()) + 1
    family_count = max(c['family_code'] for c in contexts.values()) + 1
    model = MagiNetTask13(station_count, panel_count, family_count, normalization).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
    best_path, last_path = folder / 'best_maginet.pt', folder / 'last_maginet.pt'
    start_epoch, best, bad_epochs, best_epoch, history = 1, -float('inf'), 0, 0, []
    if resume:
        if not last_path.exists() or not best_path.exists():
            raise FileNotFoundError('재개하려면 last_maginet.pt와 best_maginet.pt가 모두 필요합니다.')
        state = torch.load(last_path, map_location=DEVICE, weights_only=True)
        if state['signature'] != signature:
            raise ValueError('설정·패널·학습 날짜가 달라 재개할 수 없습니다. REPORT_DIR를 바꾸어 새로 학습해 주세요.')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        start_epoch, best, bad_epochs, best_epoch, history = state['epoch'] + 1, state['best'], state['bad_epochs'], state['best_epoch'], state['history']
        torch.set_rng_state(state['rng_cpu'].cpu())
        if DEVICE.type == 'cuda' and state['rng_cuda']:
            torch.cuda.set_rng_state_all([v.cpu() for v in state['rng_cuda']])
        if epochs < start_epoch:
            raise ValueError(f'EPOCHS를 마지막 완료 epoch {start_epoch-1}보다 크게 지정해 주세요.')
    elif best_path.exists() or last_path.exists():
        raise FileExistsError(f'기존 학습을 보호합니다. RESUME=True 또는 새 REPORT_DIR를 사용해 주세요: {folder}')
    fit_entries = [e for e in entries if e['period'] == 'fit']
    pools = {(name, r): [e for e in fit_entries if e['panel'] == name and e['regime'] == r]
             for name in contexts for r in range(3)}
    if any(not p for p in pools.values()):
        raise ValueError('공동 학습에 panel×regime별 날짜가 모두 필요합니다.')
    metadata = checkpoint_metadata(contexts, normalization, settings)
    # 재개 설정과 학습 그룹을 확인한 뒤 결과·캐시를 각각 저장합니다.
    (folder / 'config.json').write_text(json.dumps(settings, indent=2, ensure_ascii=False))
    (folder / 'input_normalization.json').write_text(json.dumps(normalization, indent=2, ensure_ascii=False))
    manifest_key = hashlib.sha256(json.dumps(entries, sort_keys=True).encode()).hexdigest()[:16]
    manifest_dir = CACHE_METADATA_DIR / 'manifests'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = manifest_dir / f'cache_manifest-{manifest_key}.csv'
    temporary = manifest_path.with_suffix('.csv.tmp')
    pd.DataFrame(entries).to_csv(temporary, index=False)
    temporary.replace(manifest_path)
    direction_count = {n: sum(p.rsplit('_', 1)[0] == n.rsplit('_', 1)[0] for p in contexts) for n in contexts}
    panel_batches = {n: panel_batch_size(c) for n, c in contexts.items()}
    print({'panel_batches': panel_batches}, flush=True)
    began = time.perf_counter()
    prior_elapsed = history[-1]['elapsed_seconds'] if history else 0.
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        rng = np.random.default_rng(SEED + epoch)
        mask_rng = np.random.default_rng(np.random.SeedSequence([SEED, epoch, 1]))
        groups = list(pools)
        rng.shuffle(groups)
        totals = dict(State=0., N=0., dN=0., FD=0.)
        updates, train_windows, train_started = 0, 0, time.perf_counter()
        physics_factor = min(1., max(0., (epoch - WARMUP_EPOCHS) / max(PHYSICS_RAMP_EPOCHS, 1)))
        for name, regime in groups:
            pool = pools[(name, regime)]
            probabilities = np.array([e['targets'] for e in pool], np.float64)
            probabilities /= probabilities.sum()
            for step in range(UPDATES_PER_GROUP):
                if step % 4 == 0:
                    entry = pool[int(rng.choice(len(pool), p=probabilities))]
                    d = load_day(entry['path'])
                    if MASK_RESAMPLE:
                        d = resample_fit_mask(d, contexts[name], mask_rng, entry['period'])
                    starts = [s for s in window_starts(len(d['values'])) if d['target'][s:s+WINDOW].any()]
                    if not starts:
                        raise ValueError('대상 셀이 있는 학습 창이 없습니다.')
                chunk = rng.choice(starts, size=panel_batches[name], replace=True).tolist()
                batch = window_batch(d, chunk)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                    output = model(**batch, ct=tensors[name])
                loss, terms = joint_loss(output.float(), d, chunk, contexts[name], tensors[name], physics_factor)
                # 도로군마다 방향 수가 달라도 누적 가중치가 같도록 맞춥니다.
                scaler.scale(loss / direction_count[name]).backward()
                optimizer_step(model, optimizer, scaler)
                updates += 1
                train_windows += len(chunk)
                for k in totals:
                    totals[k] += terms[k]
            print(f'epoch {epoch}/{epochs}: {name}/{REGIMES[regime]} 학습 완료', flush=True)
        train_seconds = time.perf_counter() - train_started
        macro, _, _, eval_seconds = evaluate(entries, contexts, tensors, model, 'early', (1.,), folder / 'early')
        proxy = float(macro.iloc[0].selection_proxy)
        score = float(macro.iloc[0][CHECKPOINT_METRIC])
        if not np.isfinite(score):
            raise FloatingPointError('체크포인트 선택 점수가 유한하지 않습니다.')
        scheduler.step(score)
        if score > best + 1e-7:
            best, bad_epochs, best_epoch = score, 0, epoch
            payload = dict(model=model.state_dict(), signature=signature, epoch=epoch, score=score,
                           checkpoint_metric=CHECKPOINT_METRIC, **metadata)
            save_checkpoint(payload, best_path)
            if CHECKPOINT_METRIC == 'State':
                save_checkpoint(payload, folder / 'best_state_maginet.pt')
        else:
            bad_epochs += 1
        row = dict(epoch=epoch, **{f'loss_{k}': v / updates for k, v in totals.items()},
                   optimizer_updates=updates, train_windows=train_windows,
                   early_State=float(macro.iloc[0].State), early_selection_proxy=proxy,
                   early_checkpoint_score=score,
                   physics_factor=physics_factor, lr=optimizer.param_groups[0]['lr'],
                   train_seconds=train_seconds, eval_seconds=eval_seconds,
                   elapsed_seconds=prior_elapsed + time.perf_counter() - began)
        history.append(row)
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        save_checkpoint(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                        scaler=scaler.state_dict(), signature=signature, epoch=epoch, best=best,
                        bad_epochs=bad_epochs, best_epoch=best_epoch, history=history,
                        rng_cpu=torch.get_rng_state(), rng_cuda=torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else [],
                        **metadata), last_path)
        print({'epoch': epoch, 'State': row['early_State'], 'selection_proxy': proxy,
               'checkpoint_metric': CHECKPOINT_METRIC, 'checkpoint_score': score,
               'best_epoch': best_epoch, 'physics_factor': physics_factor}, flush=True)
        if bad_epochs >= PATIENCE and epoch >= WARMUP_EPOCHS + PHYSICS_RAMP_EPOCHS + 2:
            break
    best_state = torch.load(best_path, map_location=DEVICE, weights_only=True)
    if best_state['signature'] != signature:
        raise ValueError('저장된 best 체크포인트의 설정이 다릅니다.')
    model.load_state_dict(best_state['model'])
    model.eval()
    print({'best_epoch': best_state['epoch'], 'checkpoint_metric': CHECKPOINT_METRIC,
           'best_early_score': best_state['score']})
    return model, pd.DataFrame(history)

def predict_split(contexts, model, tensors, alpha, split, folder, postprocessors=None):
    if split not in ('validation', 'private') or not 0 <= alpha <= 1:
        raise ValueError('추론 split 또는 alpha가 잘못되었습니다.')
    folder.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, c in contexts.items():
        template_path = c['release'] / 'task1' / name / split / 'sample_submission_state.csv'
        template = pd.read_csv(template_path, dtype={k: str for k in KEYS})
        if template.empty or template.duplicated(KEYS).any() or set(template.panel) != {name}:
            raise ValueError(f'잘못된 제출 템플릿: {template_path}')
        files = dated_files(c['release'] / 'corridors' / name / split / 'mainline_states_masked')
        if not files:
            raise FileNotFoundError(f'공개 masked 입력이 없습니다: {name}/{split}')
        first = StationDay.load(next(iter(files.values())), c['panel'])
        target_index = TargetIndex(c['release'], name, first, split)
        if (first.station_ids, first.station_links) != (c['targets'].stations, c['targets'].links):
            raise ValueError(f'추론 관측소 구성이 다릅니다: {name}')
        out = template.copy()
        out[['speed_kmh', 'flow_vph']] = np.nan
        for number, (date, group) in enumerate(template.groupby(template.timestamp.str[:10], sort=False), 1):
            if date not in files:
                raise ValueError(f'추론 날짜가 누락되었습니다: {name}/{date}')
            d = make_day(c, date, split=split, target_index=target_index, input_files=files)
            ti, si = d['ti'], d['si']
            base = d['base'][ti, si]
            if alpha > 0:
                output = maginet_predict(model, d, tensors[name], batch_size=panel_batch_size(c))
                neural = (postprocessors[name](d, output) if postprocessors else output)[ti, si]
            else:
                neural = base
            values = ((1 - alpha) * base + alpha * neural) * SCALE
            local = pd.DataFrame(dict(timestamp=d['timestamps'][ti], station_id=np.asarray(c['targets'].stations)[si],
                                      speed_kmh=values[:, 0], flow_vph=values[:, 1] * c['lanes'][si]))
            aligned = group[['timestamp', 'station_id']].merge(local, on=['timestamp', 'station_id'],
                                                               how='left', validate='one_to_one', sort=False)
            if aligned[['speed_kmh', 'flow_vph']].isna().any().any():
                raise ValueError(f'추론 대상 키가 일치하지 않습니다: {name}/{date}')
            out.loc[group.index, ['speed_kmh', 'flow_vph']] = aligned[['speed_kmh', 'flow_vph']].to_numpy()
            if number == 1 or number % 10 == 0:
                print(f'CSV 예측 {name}/{split}: {number}일', flush=True)
        values = out[['speed_kmh', 'flow_vph']].to_numpy()
        if not out[KEYS].equals(template[KEYS]) or not np.isfinite(values).all() or (values < 0).any():
            raise ValueError(f'출력 값 또는 템플릿 키가 잘못되었습니다: {name}')
        path = folder / f'task1-{name}-{split}.csv'
        temporary = path.with_suffix('.csv.tmp')
        out.to_csv(temporary, index=False)
        temporary.replace(path)
        paths.append(str(path))
    return paths


def checkpoint_metadata(contexts, normalization, settings):
    """Store inference inputs as plain containers and CPU tensors (weights_only safe)."""
    packed = {}
    for name, context in contexts.items():
        panel = context['panel']
        network = {}
        for key in PanelInfo.__dataclass_fields__:
            value = getattr(panel, key)
            network[key] = torch.from_numpy(value.copy()) if isinstance(value, np.ndarray) else value
        packed[name] = dict(network=network, stations=context['targets'].stations,
                            station_links=context['targets'].links, dates=context['dates'], mode=context['mode'],
                            panel_code=int(context['panel_code']), family_code=int(context['family_code']),
                            direction_code=int(context['direction_code']), topology_nodes=sorted(context['topology_nodes']),
                            cache_key=context.get('cache_key'))
        for key in ('mapping', 'lanes', 'station_codes', 'station_graph', 'profile'):
            packed[name][key] = torch.from_numpy(np.asarray(context[key]).copy())
    return dict(format_version=3, settings=settings, normalization=normalization, contexts=packed,
                experiment=EXPERIMENT, runtime=dict(torch=str(torch.__version__), device=str(DEVICE),
                                                   precision=str(AMP_DTYPE) if USE_AMP else 'float32',
                                                   cpu_threads=torch.get_num_threads()))


def unpack_contexts(packed, release):
    contexts = {}
    for name, saved in packed.items():
        network = {k: v.cpu().numpy().copy() if isinstance(v, torch.Tensor) else v
                   for k, v in saved['network'].items()}
        context = {k: saved[k] for k in ('dates', 'mode', 'panel_code', 'family_code', 'direction_code')}
        context.update(release=release, panel=PanelInfo(**network), topology_nodes=set(saved['topology_nodes']),
                       targets=SimpleNamespace(stations=saved['stations'], links=saved['station_links']),
                       cache_key=saved.get('cache_key'))
        for key in ('mapping', 'lanes', 'station_codes', 'station_graph', 'profile'):
            context[key] = saved[key].cpu().numpy().copy()
        contexts[name] = context
    return contexts


SETTING_NAMES = dict(input_normalization='INPUT_NORMALIZATION', seed='SEED', window='WINDOW', stride='STRIDE',
                     width='WIDTH', blocks='BLOCKS', heads='HEADS', cheb_k='CHEB_K', graph_dim='GRAPH_DIM',
                     dropout='DROPOUT', mgl='MGL_THRESHOLD', batch='BATCH_SIZE', updates='UPDATES_PER_GROUP',
                     lr='LR', weight_decay='WEIGHT_DECAY', patience='PATIENCE', warmup='WARMUP_EPOCHS',
                     ramp='PHYSICS_RAMP_EPOCHS', loss_weights='LOSS_WEIGHTS', select_weights='SELECT_WEIGHTS',
                     periods='PERIODS', days_per_regime='DAYS_PER_REGIME',
                     batch_policy='BATCH_POLICY', panel_batch='PANEL_BATCH', checkpoint_metric='CHECKPOINT_METRIC')
DEFAULT_CONFIG = {key: globals()[key] for key in SETTING_NAMES.values()}


def restore_settings(settings):
    if settings['architecture'] != 'maginet-task13-residual-v2':
        raise ValueError('지원하지 않는 MagiNet 구조입니다.')
    globals().update(BATCH_POLICY='fixed', PANEL_BATCH={}, CHECKPOINT_METRIC='selection_proxy')
    for saved_key, variable in SETTING_NAMES.items():
        if saved_key in settings:
            value = settings[saved_key]
            globals()[variable] = value['mode'] if saved_key == 'input_normalization' else value
    improvements = settings.get('improvements', {})
    globals().update(TEMPORAL_DIRECT=bool(improvements.get('temporal_direct')),
                     MASK_RESAMPLE=bool(improvements.get('mask_resample')),
                     LEARNED_GRAPH=bool(improvements.get('learned_graph')),
                     SPATIAL_MODE=improvements.get('spatial_mode', 'legacy'))
    if (SPATIAL_MODE not in ('legacy', 'context', 'channel')
            or SPATIAL_MODE != 'legacy' and not LEARNED_GRAPH
            or CHECKPOINT_METRIC not in ('selection_proxy', 'State')):
        raise ValueError('저장된 공간 경로·체크포인트 선택 기준이 잘못되었습니다.')


def load_local_checkpoint(path, release, saved=None):
    saved = torch.load(path, map_location='cpu', weights_only=True) if saved is None else saved
    if saved.get('format_version') != 3:
        raise ValueError('Colab 체크포인트는 먼저 import-colab 명령으로 가져와 주세요.')
    signature = hashlib.sha256(json.dumps(saved['settings'], sort_keys=True).encode()).hexdigest()
    if signature != saved['signature']:
        raise ValueError('체크포인트 설정 서명이 일치하지 않습니다.')
    restore_settings(saved['settings'])
    contexts = unpack_contexts(saved['contexts'], release)
    model = MagiNetTask13(max(int(c['station_codes'].max()) for c in contexts.values()) + 1,
                         max(c['panel_code'] for c in contexts.values()) + 1,
                         max(c['family_code'] for c in contexts.values()) + 1, saved['normalization']).to(DEVICE)
    model.load_state_dict(saved['model'])
    return model.eval(), contexts, saved


def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_release(path=None):
    if path is None:
        pointer = PROJECT_ROOT / 'data' / 'release_root.txt'
        path = pointer.read_text(encoding='utf-8-sig').strip() if pointer.is_file() else RELEASE_ROOT
    path = Path(path).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    path = path.resolve()
    if not (path / 'corridors').is_dir() or not (path / 'task1').is_dir():
        raise FileNotFoundError(f'corridors/와 task1/이 있는 데이터 경로를 지정해 주세요: {path}')
    return path


def checkpoint_folder(path):
    return path.parent.parent if path.parent.name == 'full' else path.parent


def configure(args):
    globals().update(DEFAULT_CONFIG)
    globals()['SPATIAL_MODE'] = 'legacy'
    globals()['DATE_OVERRIDES'] = {}
    experiment = args.experiment or 'temporal_direct'
    report_dir = (args.output_dir or PROJECT_ROOT / 'reports' / 'maginet' / experiment).resolve()
    checkpoint = (args.checkpoint or report_dir / 'full' / 'best_maginet.pt').resolve()
    source = None
    if args.command in ('evaluate', 'predict') or args.command == 'train' and args.resume:
        source_path = report_dir / 'full' / 'last_maginet.pt' if args.command == 'train' else checkpoint
        source = torch.load(source_path, map_location='cpu', weights_only=True)
        if source.get('format_version') != 3:
            raise ValueError('Colab 체크포인트는 먼저 import-colab 명령으로 가져와 주세요.')
        if args.command == 'train':
            restore_settings(source['settings'])
        experiment = source['experiment']
        if args.output_dir is None and args.command in ('evaluate', 'predict'):
            report_dir = checkpoint_folder(checkpoint)
        if args.command == 'train':
            globals()['DATE_OVERRIDES'] = {n: c['dates'] for n, c in source['settings']['panels'].items()}
    elif args.command == 'import-colab':
        source_folder = args.source_run / 'full' if (args.source_run / 'full' / 'config.json').is_file() else args.source_run
        old_settings = json.loads((source_folder / 'config.json').read_text(encoding='utf-8'))
        restore_settings(old_settings)
        globals()['DATE_OVERRIDES'] = {n: c['dates'] for n, c in old_settings['panels'].items()}
        if args.experiment is None:
            for name, (dn, temporal, resample, learned) in EXPERIMENTS.items():
                if (LOSS_WEIGHTS['dN'], TEMPORAL_DIRECT, MASK_RESAMPLE, LEARNED_GRAPH) == (dn, temporal, resample, learned):
                    experiment = name
                    break
        if args.output_dir is None:
            report_dir = PROJECT_ROOT / 'reports' / 'maginet' / experiment
    else:
        dn, temporal, resample, learned = EXPERIMENTS[experiment]
        globals().update(LOSS_WEIGHTS=dict(N=0.05, dN=dn, FD=0.), TEMPORAL_DIRECT=temporal,
                         MASK_RESAMPLE=resample, LEARNED_GRAPH=learned)
        if experiment in SPATIAL_EXPERIMENTS:
            globals().update(SPATIAL_MODE=SPATIAL_EXPERIMENTS[experiment], CHECKPOINT_METRIC='State')
    if args.checkpoint_metric is not None:
        globals()['CHECKPOINT_METRIC'] = 'State' if args.checkpoint_metric == 'state' else 'selection_proxy'
    for argument, variable in (('input_normalization', 'INPUT_NORMALIZATION'), ('seed', 'SEED'),
                               ('batch_size', 'BATCH_SIZE'), ('updates_per_group', 'UPDATES_PER_GROUP'),
                               ('epochs', 'EPOCHS'), ('patience', 'PATIENCE'), ('lr', 'LR'),
                               ('days_per_regime', 'DAYS_PER_REGIME'), ('width', 'WIDTH'),
                               ('blocks', 'BLOCKS'), ('heads', 'HEADS'), ('batch_policy', 'BATCH_POLICY')):
        value = getattr(args, argument)
        if value is not None:
            globals()[variable] = value
    if args.panel_batch is not None:
        overrides = {}
        for item in args.panel_batch:
            try:
                name, size = item.split('=', 1)
                size = int(size)
            except ValueError as error:
                raise ValueError('--panel-batch는 PANEL=양의정수 형식이어야 합니다.') from error
            if not name or size < 1 or name in overrides:
                raise ValueError('--panel-batch의 패널 중복·이름·배치 크기를 확인해 주세요.')
            overrides[name] = size
        globals()['PANEL_BATCH'] = overrides
    if source and args.experiment and args.experiment != source['experiment']:
        raise ValueError('저장 모델과 --experiment가 다릅니다.')
    if args.command == 'smoke':
        globals().update(DAYS_PER_REGIME=1 if args.days_per_regime is None else args.days_per_regime,
                         UPDATES_PER_GROUP=1 if args.updates_per_group is None else args.updates_per_group,
                         EPOCHS=2 if args.epochs is None else args.epochs)
        if args.output_dir is None:
            report_dir = report_dir / 'smoke'
    if BATCH_POLICY == 'adaptive' and args.batch_size is not None:
        raise ValueError('adaptive에서는 --batch-size 대신 --panel-batch로 배치를 지정해 주세요.')
    if (min(BATCH_SIZE, UPDATES_PER_GROUP, EPOCHS, WIDTH, BLOCKS, HEADS) < 1 or WIDTH % HEADS
            or WIDTH % 2 or not np.isfinite(LR) or LR <= 0 or PATIENCE < 1 or args.cpu_threads < 1
            or DAYS_PER_REGIME is not None and DAYS_PER_REGIME < 1):
        raise ValueError('모델 크기·학습량·학습률·날짜 수 설정을 확인해 주세요.')
    device = args.device
    if device == 'auto':
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA를 사용할 수 없습니다. --device cpu로 실행할 수 있습니다.')
    if args.precision == 'bf16' and (device != 'cuda' or not torch.cuda.is_bf16_supported()):
        raise ValueError('이 장치에서는 BF16을 사용할 수 없습니다.')
    amp = device == 'cuda' and args.precision != 'fp32'
    dtype = torch.bfloat16 if device == 'cuda' and torch.cuda.is_bf16_supported() else torch.float16
    cache_dir = (args.cache_dir or PROJECT_ROOT / 'data' / 'maginet_cache').resolve()
    selected = args.panel
    if source and args.command == 'train':
        selected = selected or sorted(source['contexts'])
    elif args.command == 'import-colab':
        selected = selected or sorted(old_settings['panels'])
    globals().update(EXPERIMENT=experiment, RELEASE_ROOT=resolve_release(args.release_root), REPORT_DIR=report_dir,
                     REPORT_BASE=report_dir.parent, CACHE_STORE_DIR=cache_dir, CACHE_DIR=cache_dir,
                     CACHE_ROOT=cache_dir, CACHE_CONTAINER=cache_dir, CACHE_METADATA_DIR=cache_dir / 'metadata',
                     CACHE_MODE=args.cache_mode, SAVED_CACHE_ZIP=args.cache_zip,
                     CACHE_PROFILE_DIR=args.legacy_profile_dir or (source_folder if args.command == 'import-colab'
                                                                  else cache_dir / 'legacy_profiles'),
                     PANELS=selected, DEVICE=torch.device(device), USE_AMP=amp, AMP_DTYPE=dtype,
                     TRAIN_MODEL=args.command in ('train', 'smoke'), RESUME=args.resume)
    torch.set_num_threads(args.cpu_threads)
    reset_seed()
    load_day.cache_clear()
    args.checkpoint = checkpoint
    args.saved_checkpoint = source
    if (args.select_retrieval or args.retrieval_selection is not None) and args.output_dir is None:
        globals()['REPORT_DIR'] = REPORT_DIR / 'retrieval'
    print(dict(command=args.command, experiment=experiment, device=device,
               precision=str(dtype) if amp else 'float32', release=str(RELEASE_ROOT),
               cache=str(cache_dir), reports=str(REPORT_DIR)), flush=True)


def prepare_inputs(smoke=False):
    names = available_panels(RELEASE_ROOT)
    if smoke and PANELS is None:
        names = names[:1]
    unknown = set(PANEL_BATCH) - set(names)
    if unknown:
        raise ValueError(f'학습에 포함되지 않은 --panel-batch 패널입니다: {sorted(unknown)}')
    root, container = (restore_input_cache(names, SAVED_CACHE_ZIP, CACHE_DIR / 'imported') if SAVED_CACHE_ZIP
                       else (CACHE_DIR, CACHE_DIR))
    globals().update(CACHE_ROOT=root, CACHE_CONTAINER=container)
    contexts, entries, offset = {}, [], 0
    for code, name in enumerate(names):
        context = prepare_panel(RELEASE_ROOT, name, code, offset)
        offset += len(context['targets'].stations)
        contexts[name] = context
    attach_vocabulary(contexts)
    identity = dict(format='maginet-local-input-v1', networks=network_hashes(contexts),
                    panels={n: dict(stations=c['targets'].stations, links=c['targets'].links,
                                    fit_dates=c['dates']['fit'],
                                    profile_hash=hashlib.sha256(c['profile'].tobytes()).hexdigest())
                            for n, c in contexts.items()})
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]
    destination = CACHE_DIR / 'inputs' / key
    for name, context in contexts.items():
        context['cache_key'] = key
        for period in ('fit', 'early', 'blend'):
            for date in context['dates'][period]:
                source = root / name / period / f'{date}.npz'
                target = destination / name / period / f'{date}.npz'
                if source.is_file() and not target.exists():
                    target.parent.mkdir(parents=True, exist_ok=True)
                    temporary = target.with_suffix('.npz.tmp')
                    shutil.copyfile(source, temporary)
                    temporary.replace(target)
        entries.extend(cache_context(context, destination))
    globals()['CACHE_ROOT'] = destination
    return contexts, entries


def evaluation_inputs(contexts, period):
    entries = []
    for name, context in contexts.items():
        folder = RELEASE_ROOT / 'corridors' / name / 'train'
        context['files'] = dated_files(folder / 'mainline_states')
        context['masked'] = dated_files(folder / 'mainline_states_masked')
        dates = context['dates'][period]
        if not dates:
            raise ValueError(f'체크포인트의 {name}/{period} 날짜가 없습니다.')
        first = StationDay.load(context['files'][dates[0]], context['panel'])
        target_index = TargetIndex(RELEASE_ROOT, name, first)
        if (target_index.stations, target_index.links) != (context['targets'].stations, context['targets'].links):
            raise ValueError(f'체크포인트의 관측소와 로컬 데이터가 다릅니다: {name}')
        context['targets'] = target_index
        root = CACHE_DIR / 'inputs' / context['cache_key'] if context.get('cache_key') else CACHE_ROOT
        entries.extend(cache_context(context, root, periods=(period,)))
    return entries


def select_alpha(entries, contexts, model, checkpoint):
    tensors = {n: tensor_context(c) for n, c in contexts.items()}
    macro, _, _, _ = evaluate(entries, contexts, tensors, model, 'blend', ALPHAS, REPORT_DIR,
                               collect_diagnostics=True)
    best = macro.sort_values([CHECKPOINT_METRIC, 'alpha'], ascending=[False, True]).iloc[0]
    selection = dict(alpha=float(best.alpha), selection_proxy=float(best.selection_proxy),
                     State=float(best.State), criterion=dict(State=1.) if CHECKPOINT_METRIC == 'State' else SELECT_WEIGHTS,
                     selected_on='2031-01-16~31',
                     panels=sorted(contexts), checkpoint_sha256=file_digest(checkpoint))
    path = REPORT_DIR / 'selected_alpha.json'
    path.write_text(json.dumps(selection, indent=2, ensure_ascii=False), encoding='utf-8')
    print(macro.to_string(index=False), flush=True)
    return selection['alpha']


def prediction_alpha(args, contexts):
    if args.alpha is not None:
        return args.alpha
    selection = args.selection or checkpoint_folder(args.checkpoint) / 'selected_alpha.json'
    saved = json.loads(selection.read_text(encoding='utf-8'))
    alpha = float(saved['alpha'])
    if (not 0 <= alpha <= 1 or saved['checkpoint_sha256'] != file_digest(args.checkpoint)
            or not set(contexts) <= set(saved['panels'])):
        raise ValueError('선택한 alpha의 값·체크포인트·패널이 다릅니다.')
    return alpha


def retrieval_memories(contexts):
    from historical_retrieval import HistoricalRetrieval
    entries = evaluation_inputs(contexts, 'fit')
    memories = {}
    for name in contexts:
        memories[name] = HistoricalRetrieval(e['path'] for e in entries if e['panel'] == name)
        print(f'과거 검색 준비: {name}, fit {len(memories[name].dates)}일', flush=True)
    return memories


def retrieval_metrics(frame, weights):
    weights = np.broadcast_to(weights, (len(frame), 2))
    mix = np.stack([1 - weights, weights], -1)
    sse = np.einsum('gci,gcij,gcj->gc', mix, np.stack(frame.gram), mix)
    rms = np.sqrt(np.maximum(sse, 0) / frame['count'].to_numpy()[:, None])
    state = .54 * np.maximum(0, 1 - rms[:, 0] / 25) + .46 * np.maximum(0, 1 - rms[:, 1] / 600)
    values = pd.DataFrame(dict(State=state, speed_rmse_kmh=rms[:, 0], flow_rmse_per_lane=rms[:, 1]), index=frame.index)
    return {k: macro_mean(v.to_dict()) for k, v in values.groupby(level='panel').mean().items()}


def select_retrieval(entries, contexts, tensors, model, memories, checkpoint, folder):
    from historical_retrieval import CONFIG, METHODS
    from scipy.optimize import minimize_scalar
    records = []
    for number, entry in enumerate(entries, 1):
        if entry['period'] != 'early':
            raise ValueError('검색 혼합 비율은 early 구간에서만 선택합니다.')
        day = load_day(entry['path'])
        output = maginet_predict(model, day, tensors[entry['panel']], batch_size=panel_batch_size(contexts[entry['panel']]))
        candidates = memories[entry['panel']].candidates(day, output)
        ti, si = day['ti'], day['si']
        truth = day['truth'][ti, si].astype(np.float64) * SCALE
        baseline_error = output[ti, si].astype(np.float64) * SCALE - truth
        for method, candidate in candidates.items():
            errors = np.stack([baseline_error, candidate[ti, si].astype(np.float64) * SCALE - truth], -1)
            records.append(dict(panel=entry['panel'], regime=entry['regime'], method=method,
                                count=len(truth), gram=np.einsum('nci,ncj->cij', errors, errors)))
        if number == 1 or number % 20 == 0 or number == len(entries):
            print(f'검색 비율 선택 early: {number}/{len(entries)}일', flush=True)
    comparisons = []
    for method in METHODS:
        frame = pd.DataFrame([r for r in records if r['method'] == method]).groupby(['panel', 'regime']).agg(
            count=('count', 'sum'), gram=('gram', 'sum'))
        if len(frame) != 3 * len(contexts):
            raise ValueError('검색 비율 선택에 panel×regime이 누락됐습니다.')
        weights = []
        for channel in (0, 1):
            def loss(w):
                return -retrieval_metrics(frame, np.eye(2)[channel] * w)['State']
            optimum = minimize_scalar(loss, bounds=(0, 1), method='bounded', options=dict(xatol=1e-7))
            if not optimum.success:
                raise RuntimeError('검색 혼합 비율 최적화에 실패했습니다.')
            weights.append(min((0., 1., float(optimum.x)), key=loss))
        comparisons.append(dict(method=method, speed_weight=weights[0], flow_weight=weights[1],
                                **retrieval_metrics(frame, weights), baseline_State=retrieval_metrics(frame, 0.)['State']))
    best = max(comparisons, key=lambda row: row['State'])
    selection = dict(format='maginet-retrieval-v1', config=CONFIG, method=best['method'],
                     weights=[best['speed_weight'], best['flow_weight']], selected_on='early',
                     checkpoint_sha256=file_digest(checkpoint),
                     panels={n: dict(fit_dates=memory.dates, memory_sha256=memory.sha256) for n, memory in memories.items()})
    folder.mkdir(parents=True, exist_ok=True)
    (folder / 'selected_retrieval.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')
    pd.DataFrame(comparisons).to_csv(folder / 'retrieval_early_comparison.csv', index=False)
    print(pd.DataFrame(comparisons).to_string(index=False), flush=True)
    return selection


def retrieval_postprocessors(path, contexts, memories, checkpoint, folder):
    from historical_retrieval import CONFIG, METHODS
    saved = json.loads(path.read_text(encoding='utf-8'))
    weights = np.asarray(saved['weights'], dtype=float)
    if (saved.get('format') != 'maginet-retrieval-v1' or saved['config'] != CONFIG
            or saved['method'] not in METHODS or weights.shape != (2,) or not np.isfinite(weights).all()
            or (weights < 0).any() or (weights > 1).any()
            or saved['checkpoint_sha256'] != file_digest(checkpoint) or not set(contexts) <= set(saved['panels'])):
        raise ValueError('검색 선택 파일의 설정·비율·체크포인트·패널이 다릅니다.')
    for name, memory in memories.items():
        if saved['panels'][name]['memory_sha256'] != memory.sha256:
            raise ValueError(f'검색 비율을 선택한 fit 자료가 다릅니다: {name}')
    folder.mkdir(parents=True, exist_ok=True)
    (folder / 'retrieval_run.json').write_text(json.dumps(saved, indent=2), encoding='utf-8')
    return {name: partial(memory.apply, method=saved['method'], weights=weights) for name, memory in memories.items()}


def import_colab(contexts, entries, source_run):
    source = source_run / 'full' if (source_run / 'full' / 'config.json').is_file() else source_run
    original = json.loads((source / 'config.json').read_text(encoding='utf-8'))
    normalization = fit_input_normalization(entries, CACHE_METADATA_DIR / 'normalization')
    _, expected = notebook_signature(contexts, entries, normalization)
    expected['release'] = original.get('release')
    if json.dumps(expected, sort_keys=True) != json.dumps(original, sort_keys=True):
        raise ValueError('Colab 설정과 로컬 데이터·기간·과거 평균·모델 설정이 다릅니다.')
    old_signature = hashlib.sha256(json.dumps(original, sort_keys=True).encode()).hexdigest()
    signature, settings = run_signature(contexts, entries, normalization)
    target = REPORT_DIR / 'full'
    if (target / 'best_maginet.pt').exists() or (target / 'last_maginet.pt').exists():
        raise FileExistsError('가져오기 결과 폴더에 체크포인트가 이미 있습니다.')
    payloads = {}
    for name in ('best_maginet.pt', 'last_maginet.pt'):
        path = source / name
        if path.is_file():
            state = torch.load(path, map_location='cpu', weights_only=True)
            if state['signature'] != old_signature:
                raise ValueError(f'Colab config.json과 체크포인트 서명이 다릅니다: {path}')
            state.update(signature=signature, **checkpoint_metadata(contexts, normalization, settings))
            payloads[name] = state
    if 'best_maginet.pt' not in payloads:
        raise FileNotFoundError(source / 'best_maginet.pt')
    target.mkdir(parents=True, exist_ok=True)
    for name, state in payloads.items():
        save_checkpoint(state, target / name)
    for name, value in (('config.json', settings), ('input_normalization.json', normalization)):
        (target / name).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    if (source / 'history.csv').is_file():
        shutil.copyfile(source / 'history.csv', target / 'history.csv')
    print(f'Colab 모델을 별도 로컬 폴더에 가져왔습니다: {target}', flush=True)


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('command', choices=('prepare', 'train', 'evaluate', 'predict', 'smoke', 'import-colab'))
    parser.add_argument('--release-root', type=Path, help='기본값: data/release_root.txt')
    parser.add_argument('--cache-dir', type=Path)
    parser.add_argument('--cache-mode', choices=('build', 'reuse'), default='build')
    parser.add_argument('--cache-zip', type=Path)
    parser.add_argument('--legacy-profile-dir', type=Path)
    parser.add_argument('--output-dir', type=Path)
    parser.add_argument('--checkpoint', type=Path)
    parser.add_argument('--panel', action='append')
    parser.add_argument('--experiment', choices=tuple(EXPERIMENTS))
    parser.add_argument('--input-normalization', choices=('fixed', 'standard'))
    parser.add_argument('--epochs', type=int)
    parser.add_argument('--batch-size', type=int)
    parser.add_argument('--batch-policy', choices=('fixed', 'adaptive'),
                        help='adaptive: 관측소 128개 이하 배치 8, 초과 4; 기본 fixed')
    parser.add_argument('--panel-batch', action='append', metavar='PANEL=SIZE',
                        help='패널별 배치 재지정; 반복 가능, batch-policy보다 우선')
    parser.add_argument('--updates-per-group', type=int)
    parser.add_argument('--days-per-regime', type=int)
    parser.add_argument('--seed', type=int)
    parser.add_argument('--width', type=int)
    parser.add_argument('--blocks', type=int)
    parser.add_argument('--heads', type=int)
    parser.add_argument('--lr', type=float)
    parser.add_argument('--patience', type=int)
    parser.add_argument('--checkpoint-metric', choices=('state', 'proxy'),
                        help='최적 모델·scheduler·조기 종료 기준; 새 공간 실험 기본 state, 기존 실험 proxy')
    parser.add_argument('--resume', action='store_true')
    parser.add_argument('--device', choices=('auto', 'cpu', 'cuda'), default='auto')
    parser.add_argument('--precision', choices=('auto', 'fp32', 'bf16'), default='auto')
    parser.add_argument('--cpu-threads', type=int, default=4)
    parser.add_argument('--period', choices=('early', 'blend', 'february'), default='blend')
    parser.add_argument('--select-alpha', action='store_true')
    parser.add_argument('--alpha', type=float)
    parser.add_argument('--selection', type=Path)
    parser.add_argument('--split', choices=('validation', 'private', 'both'), default='validation')
    parser.add_argument('--source-run', type=Path)
    parser.add_argument('--select-retrieval', action='store_true', help='early에서 과거 검색 방법과 채널별 비율 선택')
    parser.add_argument('--retrieval-selection', type=Path, help='고정한 selected_retrieval.json으로 평가·추론')
    args = parser.parse_args(argv)
    if args.command in ('evaluate', 'predict'):
        training_only = ('input_normalization', 'seed', 'updates_per_group', 'epochs', 'patience', 'lr',
                         'days_per_regime', 'width', 'blocks', 'heads', 'checkpoint_metric')
        supplied = ['--' + key.replace('_', '-') for key in training_only if getattr(args, key) is not None]
        if supplied:
            parser.error(f'평가·추론에서는 저장된 학습 설정을 사용합니다. 지정할 수 없는 옵션: {", ".join(supplied)}')
    if args.command in ('evaluate', 'predict', 'import-colab') and (
            args.batch_policy is not None or args.panel_batch is not None):
        parser.error('패널별 배치는 train·smoke·prepare에서 지정하며 평가·추론은 저장된 설정을 사용합니다.')
    if args.command == 'import-colab' and args.source_run is None:
        parser.error('import-colab에는 --source-run이 필요합니다.')
    if args.alpha is not None and not 0 <= args.alpha <= 1:
        parser.error('--alpha는 0~1이어야 합니다.')
    if args.select_alpha and (args.command != 'evaluate' or args.period != 'blend'):
        parser.error('--select-alpha는 evaluate --period blend에서만 사용할 수 있습니다.')
    if args.resume and args.command != 'train':
        parser.error('--resume은 train에서만 사용할 수 있습니다.')
    retrieval = args.select_retrieval or args.retrieval_selection is not None
    if args.select_retrieval and (args.command != 'evaluate' or args.period != 'early'):
        parser.error('--select-retrieval은 evaluate --period early에서만 사용할 수 있습니다.')
    if retrieval and (args.command not in ('evaluate', 'predict') or args.select_alpha or args.selection
                      or args.alpha is not None and args.alpha != 1. or args.select_retrieval and args.retrieval_selection):
        parser.error('검색 옵션은 evaluate·predict에서만 쓰며 기존 alpha 선택과 함께 지정하지 않습니다.')
    if retrieval:
        args.alpha = 1.
    configure(args)
    if args.command in ('prepare', 'train', 'smoke', 'import-colab'):
        if args.command in ('train', 'smoke') and not RESUME:
            if any((REPORT_DIR / 'full' / n).exists() for n in
                   ('best_maginet.pt', 'last_maginet.pt', 'best_state_maginet.pt')):
                raise FileExistsError('기존 체크포인트가 있습니다. --resume 또는 새 --output-dir를 사용해 주세요.')
        contexts, entries = prepare_inputs(smoke=args.command == 'smoke')
        if args.command == 'prepare':
            normalization = fit_input_normalization(entries, CACHE_METADATA_DIR / 'normalization')
            print(dict(cached_days=len(entries), panels=sorted(contexts), normalization=normalization), flush=True)
        elif args.command == 'import-colab':
            import_colab(contexts, entries, args.source_run)
        else:
            tensors = {n: tensor_context(c) for n, c in contexts.items()}
            model, history = train_maginet(entries, contexts, tensors, REPORT_DIR / 'full')
            print(history.tail().to_string(index=False), flush=True)
            checkpoint = REPORT_DIR / 'full' / 'best_maginet.pt'
            select_alpha(entries, contexts, model, checkpoint)
    else:
        model, contexts, _ = load_local_checkpoint(args.checkpoint, RELEASE_ROOT, saved=args.saved_checkpoint)
        if args.batch_size is not None:
            globals().update(BATCH_SIZE=args.batch_size, BATCH_POLICY='fixed', PANEL_BATCH={})
        if args.panel:
            unknown = set(args.panel) - set(contexts)
            if unknown:
                raise ValueError(f'체크포인트에 없는 패널입니다: {sorted(unknown)}')
            contexts = {n: contexts[n] for n in sorted(set(args.panel))}
        postprocessors = None
        if retrieval:
            memories = retrieval_memories(contexts)
            if args.select_retrieval:
                entries = evaluation_inputs(contexts, 'early')
                select_retrieval(entries, contexts, {n: tensor_context(c) for n, c in contexts.items()}, model,
                                 memories, args.checkpoint, REPORT_DIR)
                return
            postprocessors = retrieval_postprocessors(args.retrieval_selection, contexts, memories,
                                                      args.checkpoint, REPORT_DIR)
        if args.command == 'evaluate':
            entries = evaluation_inputs(contexts, args.period)
            if args.select_alpha:
                select_alpha(entries, contexts, model, args.checkpoint)
            else:
                alpha = prediction_alpha(args, contexts) if args.period == 'february' else (
                    args.alpha if args.alpha is not None else 1.)
                alphas = tuple(sorted({0., alpha}))
                macro, _, _, _ = evaluate(entries, contexts, {n: tensor_context(c) for n, c in contexts.items()},
                                           model, args.period, alphas, REPORT_DIR, collect_diagnostics=True,
                                           postprocessors=postprocessors)
                print(macro.to_string(index=False), flush=True)
        else:
            alpha = prediction_alpha(args, contexts)
            splits = ('validation', 'private') if args.split == 'both' else (args.split,)
            tensors = {n: tensor_context(c) for n, c in contexts.items()}
            for split in splits:
                paths = predict_split(contexts, model, tensors, alpha, split, REPORT_DIR / 'submissions',
                                      postprocessors=postprocessors)
                print('\n'.join(paths), flush=True)


if __name__ == '__main__':
    main()
''')


내장 코드 로드: maginet (2340줄)


In [24]:
# src/run.py — 원본 cdf888214fe8a54b73302f3988b764b74b3f30a4
load_source('run', r'''"""Train FAST-style models and combine them with the official baselines.

The organizer repository is read as an external checkout; no organizer data or
code is copied into this project. See MODEL_PLAN.md for the validation limits.
"""

from __future__ import annotations

import argparse
import os
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd

from data import PanelInfo, StationDay, link_features, station_window_features

STATE_KEYS = ["panel", "timestamp", "station_id", "link_id", "mask_regime"]
QUEUE_KEYS = ["window_id", "timestamp", "link_id"]
ODME_KEYS = ["panel", "departure_time", "path_id"]
STATE_COLUMNS = STATE_KEYS + ["speed_kmh", "flow_vph"]
QUEUE_COLUMNS = QUEUE_KEYS + ["queue_pred"]
ODME_COLUMNS = ODME_KEYS + ["origin_zone", "destination_zone", "path_flow"]


def panels(release: Path, selected: list[str] | None) -> list[str]:
    available = sorted(p.name for p in (release / "corridors").iterdir() if p.is_dir())
    if selected:
        unknown = set(selected) - set(available)
        if unknown:
            raise ValueError(f"Unknown panels: {sorted(unknown)}")
        return sorted(set(selected))
    return available


def train_files(release: Path, panel: str, period: str) -> list[Path]:
    from learning import period_files
    return period_files(release, panel, period)


def day_masks(day: StationDay, panel: PanelInfo, seed: int) -> tuple[np.ndarray, np.ndarray]:
    generator = np.random.default_rng(seed)
    rate = (0.20, 0.30, 0.50)[seed % 3]
    hidden_links = generator.random((len(day.timestamps), len(panel.links))) < rate
    visible = day.eligible & ~hidden_links[:, day.station_to_link]
    targets = day.eligible & hidden_links[:, day.station_to_link]
    return visible, targets


def center_windows(steps: int):
    for start in range(0, steps, 12):
        index = np.clip(np.arange(start - 30, start + 42), 0, steps - 1)
        yield index, slice(30, 30 + min(12, steps - start)), slice(start, min(start + 12, steps))


def tensors(device: str, *arrays: np.ndarray):
    import torch
    return [torch.as_tensor(array, device=device) for array in arrays]


def fit_state(args: argparse.Namespace) -> None:
    from learning import train
    train(args, "state")


def queue_windows(day: StationDay, panel: PanelInfo, per_condition: int, rng: np.random.Generator):
    steps, nodes = len(day.timestamps), len(panel.links)
    speed = np.zeros((steps, nodes), dtype=np.float32)
    count = np.zeros((steps, nodes), dtype=np.float32)
    for station, link in enumerate(day.station_to_link):
        good = day.eligible[:, station]
        speed[good, link] += day.speed[good, station]
        count[good, link] += 1
    valid = count > 0
    speed /= np.maximum(count, 1)
    queue = (speed * 25.0 <= panel.cutoffs[None, :]) & valid
    candidates: dict[str, list[int]] = {"onset": [], "ongoing": []}
    for origin in range(12, steps - 6):
        # The release gives [T-60, T); targets start at T+5, not at T.
        future = queue[origin + 1:origin + 7]
        if not future.any():
            continue
        history = queue[origin - 12:origin]
        if not history.any():
            candidates["onset"].append(origin)
        elif history[-1].any():
            candidates["ongoing"].append(origin)
    chosen = []
    for condition, origins in candidates.items():
        if origins:
            chosen.extend((condition, int(origin)) for origin in rng.choice(origins, min(per_condition, len(origins)), replace=False))
    for condition, origin in chosen:
        indices = slice(origin - 12, origin)
        history = link_features(day.timestamps[indices], speed[indices], np.zeros_like(speed[indices]),
                                valid[indices], np.arange(nodes), panel)
        future = slice(origin + 1, origin + 7)
        yield condition, history, queue[future], speed[future], valid[future]


def mean_iou(logits: list[np.ndarray], labels: list[np.ndarray], valid: list[np.ndarray], threshold: float) -> float:
    scores = []
    for output, truth, mask in zip(logits, labels, valid):
        predicted = (1.0 / (1.0 + np.exp(-np.clip(output, -30, 30))) >= threshold) & mask
        actual = truth & mask
        union = np.logical_or(predicted, actual).sum()
        scores.append(float(np.logical_and(predicted, actual).sum() / union) if union else 0.0)
    return float(np.mean(scores)) if scores else 0.0


def fit_queue(args: argparse.Namespace) -> None:
    from learning import train
    train(args, "queue")


def load_model(path: Path, task: str, device: str):
    import torch
    from fast_model import QueueModel, StateModel
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint["task"] != task:
        raise ValueError(f"Expected {task} checkpoint: {path}")
    if checkpoint.get("format_version") == 2:
        from learning import model_from_checkpoint
        return model_from_checkpoint(checkpoint, device).eval(), checkpoint
    model = (StateModel if task == "state" else QueueModel)(checkpoint["width"], checkpoint["blocks"])
    model.load_state_dict(checkpoint["weights"])
    return model.to(device).eval(), checkpoint


def predict_state(release: Path, split: str, checkpoint: Path, output: Path) -> None:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, config = load_model(checkpoint, "state", device)
    if config.get("format_version") == 2:
        from learning import predict_state_v2
        predict_state_v2(model, config, release, split, output)
        return
    parts = []
    for name in panels(release, None):
        panel = PanelInfo.load(release, name)
        template_path = release / "task1" / name / split / "sample_submission_state.csv"
        template = pd.read_csv(template_path, dtype=str)
        target_by_date = {date: group.copy() for date, group in template.groupby(template.timestamp.str[:10])}
        files = sorted((release / "corridors" / name / split / "mainline_states_masked").glob("**/*.parquet"))
        found = set()
        for path in files:
            day = StationDay.load(path, panel)
            date = day.timestamps[0][:10]
            if date not in target_by_date:
                continue
            found.add(date)
            visible = day.eligible
            prediction = np.zeros((*day.speed.shape, 2), dtype=np.float32)
            station_link = torch.as_tensor(day.station_to_link, device=device)
            with torch.no_grad():
                for index, middle, absolute in center_windows(len(day.timestamps)):
                    inputs, context = station_window_features(day, panel, visible, index)
                    x, station = tensors(device, inputs[None], context[None])
                    values = model(x, station, station_link, panel.segments, config["bidirectional"])
                    prediction[absolute] = values[0, middle].cpu().numpy()
            target = target_by_date[date]
            timestamp_index = {value: i for i, value in enumerate(day.timestamps)}
            station_index = {value: i for i, value in enumerate(day.station_ids)}
            if not set(target.timestamp).issubset(timestamp_index) or not set(target.station_id).issubset(station_index):
                raise ValueError(f"Template keys absent from masked observations: {name} {date}")
            ti = target.timestamp.map(timestamp_index).to_numpy(dtype=int)
            si = target.station_id.map(station_index).to_numpy(dtype=int)
            if any(day.station_links[s] != link for s, link in zip(si, target.link_id)):
                raise ValueError(f"Station/link mismatch: {name} {date}")
            target["speed_kmh"] = prediction[ti, si, 0] * 25.0
            target["flow_vph"] = prediction[ti, si, 1] * 600.0 * day.lanes[si]
            parts.append(target[STATE_COLUMNS])
        if found != set(target_by_date):
            raise ValueError(f"Missing masked days for {name} {split}: {sorted(set(target_by_date) - found)}")
    result = pd.concat(parts, ignore_index=True)
    if result[STATE_KEYS].duplicated().any() or result[["speed_kmh", "flow_vph"]].isna().any().any():
        raise ValueError("Invalid State predictions")
    output.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output, index=False)


def predict_queue(release: Path, split: str, checkpoint: Path, output: Path) -> None:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, config = load_model(checkpoint, "queue", device)
    if config.get("format_version") == 2:
        from learning import predict_queue_v2
        predict_queue_v2(model, config, release, split, output)
        return
    parts = []
    for name in panels(release, None):
        root = release / "task2" / name / split
        if not (root / "sample_submission_queue.csv").exists():
            continue
        template = pd.read_csv(root / "sample_submission_queue.csv", dtype=str)
        history = pd.read_parquet(root / "window_history.parquet", columns=["window_id", "timestamp", "link_id", "speed_kmh", "is_score_eligible"])
        history["window_id"] = history.window_id.astype(str)
        history["link_id"] = history.link_id.astype(str)
        history["timestamp"] = history.timestamp.astype(str)
        panel = PanelInfo.load(release, name)
        windows = {key: group for key, group in history.groupby("window_id")}
        for window_id, target in template.groupby("window_id", sort=False):
            if window_id not in windows:
                raise ValueError(f"No history for {window_id}")
            frame = windows[window_id]
            times = sorted(frame.timestamp.unique().tolist())
            if len(times) != 12 or times[-1] >= min(target.timestamp):
                raise ValueError(f"Unexpected Task 2 history boundary: {window_id}")
            nodes = len(panel.links)
            speed = np.zeros((12, nodes), dtype=np.float32)
            count = np.zeros((12, nodes), dtype=np.float32)
            time_index = {value: i for i, value in enumerate(times)}
            for row in frame.itertuples(index=False):
                if row.link_id in panel.link_index and bool(row.is_score_eligible) and pd.notna(row.speed_kmh):
                    index = time_index[row.timestamp], panel.link_index[row.link_id]
                    speed[index] += float(row.speed_kmh) / 25.0
                    count[index] += 1
            valid = count > 0
            speed /= np.maximum(count, 1)
            x = link_features(times, speed, np.zeros_like(speed), valid, np.arange(nodes), panel)
            with torch.no_grad():
                logits, _ = model(tensors(device, x[None])[0], panel.segments, config["bidirectional"])
            result = target.copy()
            future = sorted(result.timestamp.unique().tolist())
            if len(future) != 6 or not set(result.link_id).issubset(panel.link_index):
                raise ValueError(f"Unexpected Task 2 targets: {window_id}")
            future_index = result.timestamp.map({value: i for i, value in enumerate(future)}).to_numpy(dtype=int)
            link_index = result.link_id.map(panel.link_index).to_numpy(dtype=int)
            probability = torch.sigmoid(logits[0]).cpu().numpy()
            result["queue_pred"] = (probability[future_index, link_index] >= config["threshold"]).astype(int)
            parts.append(result[QUEUE_COLUMNS])
    if not parts:
        raise ValueError(f"No Task 2 templates for {split}")
    result = pd.concat(parts, ignore_index=True)
    if result[QUEUE_KEYS].duplicated().any():
        raise ValueError("Duplicate queue keys")
    output.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output, index=False)


def odme_problem(release: Path, name: str, split: str):
    from scipy.sparse import csr_matrix
    network = release / "corridors" / name / "network"
    root = release / "task4" / name / split
    template = pd.read_csv(root / "sample_submission_path_flow.csv", dtype=str)
    if template.empty or template[ODME_COLUMNS[:-1]].isna().any().any():
        raise ValueError(f"Empty or incomplete ODME template: {name} {split}")
    if not template.panel.eq(name).all() or template.departure_time.nunique() != 1:
        raise ValueError(f"Mismatched ODME template period/panel: {name} {split}")
    if template[ODME_KEYS].duplicated().any() or template.path_id.duplicated().any():
        raise ValueError(f"Duplicate ODME template paths: {name} {split}")
    counts = pd.read_csv(root / "synthetic_link_counts.csv", dtype={"link_id": str})
    prior = pd.read_csv(root / "synthetic_weak_prior.csv", dtype={"path_id": str, "departure_time": str})
    incidence = pd.read_csv(network / "path_link_incidence.csv", dtype={"path_id": str, "link_id": str})
    paths = template.path_id.tolist()
    measured = counts.dropna(subset=["count"])
    if measured.link_id.duplicated().any() or prior.path_id.duplicated().any():
        raise ValueError("Duplicate measured links or priors")
    measured = measured[measured.link_id.isin(incidence.link_id)]
    if measured.empty:
        raise ValueError(f"No measured ODME links: {name} {split}")
    row = {link: i for i, link in enumerate(measured.link_id)}
    column = {path: i for i, path in enumerate(paths)}
    included = incidence[incidence.link_id.isin(row) & incidence.path_id.isin(column)]
    matrix = csr_matrix((np.ones(len(included)),
                         (included.link_id.map(row), included.path_id.map(column))),
                        shape=(len(row), len(column)))
    matrix.data[:] = 1.0  # Repeated incidence rows still mean one traversal.
    prior_values = pd.to_numeric(prior.path_flow, errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(prior_values).all() or (prior_values < 0).any():
        raise ValueError(f"Invalid ODME prior: {name} {split}")
    if "departure_time" in prior and not prior.departure_time.eq(template.departure_time.iloc[0]).all():
        raise ValueError(f"Mismatched ODME prior period: {name} {split}")
    base = pd.Series(prior_values, index=prior.path_id).reindex(paths, fill_value=0).to_numpy(dtype=np.float64)
    observed = pd.to_numeric(measured["count"], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(observed).all() or not np.isfinite(base).all() or (observed < 0).any() or (base < 0).any():
        raise ValueError(f"Invalid counts/prior: {name} {split}")
    return template, matrix, observed, base


def odme_prior_variance(base, power=0, floor_ratio=0.1):
    base = np.asarray(base, dtype=np.float64)
    if base.ndim != 1 or not base.size or not np.isfinite(base).all() or (base < 0).any():
        raise ValueError("ODME prior must be a nonempty finite nonnegative vector")
    if power not in (0, 1, 2) or not np.isfinite(floor_ratio) or floor_ratio <= 0:
        raise ValueError("ODME prior power must be 0, 1, or 2, and floor ratio must be positive")
    if power == 0:
        return np.ones_like(base)
    positive = base[base > 0]
    scale = float(np.median(positive)) if positive.size else 1.0
    variance = (base / scale + floor_ratio) ** power
    variance /= np.median(variance)
    if not np.isfinite(variance).all() or (variance <= 0).any():
        raise ValueError("ODME prior variance is outside the numerical range")
    return variance


def solve_odme(matrix, observed, base, lam, prior_power=0, prior_floor_ratio=0.1):
    from scipy.optimize import lsq_linear, minimize
    from scipy.sparse import csr_matrix
    if lam < 0 or not np.isfinite(lam):
        raise ValueError("ODME lambda must be finite and nonnegative")
    variance = odme_prior_variance(base, prior_power, prior_floor_ratio)
    base = np.asarray(base, dtype=np.float64)
    observed = np.asarray(observed, dtype=np.float64)
    design = csr_matrix(matrix, dtype=np.float64)
    if (observed.ndim != 1 or not observed.size or design.shape != (len(observed), len(base))
            or not np.isfinite(observed).all() or (observed < 0).any()
            or not np.isfinite(design.data).all() or (design.data < 0).any()):
        raise ValueError("Invalid ODME matrix/count dimensions or values")
    if lam == 0:
        result = lsq_linear(design, observed, bounds=(0, np.inf), tol=1e-8, max_iter=2000)
        if not result.success:
            raise RuntimeError(f"ODME solver did not converge: {result.message}")
        return result.x
    precision = lam / variance
    if not np.isfinite(precision).all() or (precision <= 0).any():
        raise ValueError("ODME prior precision is outside the numerical range")
    # The dual has one variable per measured link, independent of path count.
    def objective(dual):
        flow = np.maximum(base - np.asarray(design.T @ dual).ravel() / precision, 0)
        cost = 0.5 * (dual @ dual) + observed @ dual + 0.5 * np.dot(precision * (flow - base), flow + base)
        return cost, dual + observed - design @ flow
    result = minimize(objective, np.zeros(len(observed)), jac=True, method="BFGS",
                      options={"gtol": 1e-5, "maxiter": 2000})
    flow = np.maximum(base - np.asarray(design.T @ result.x).ravel() / precision, 0)
    residual, difference = design @ flow - observed, flow - base
    primal = 0.5 * (residual @ residual + np.dot(precision * difference, difference))
    # Clipped flows minimize the Lagrangian; this gap avoids subtracting large costs.
    gap = 0.5 * np.sum((result.x - residual) ** 2)
    if not np.isfinite(flow).all() or not np.isfinite(primal + gap) or gap > 1e-7 * (1 + primal):
        raise RuntimeError(f"ODME solver did not converge: {result.message}")
    return flow


def predict_odme(release: Path, split: str, output: Path, lam: float = 0.05,
                 prior_power=0, prior_floor_ratio=0.1, selected=None, prior_only=False) -> None:
    parts = []
    for name in panels(release, selected):
        template, matrix, observed, base = odme_problem(release, name, split)
        solution = base.copy() if prior_only else solve_odme(matrix, observed, base, lam, prior_power, prior_floor_ratio)
        template["path_flow"] = solution
        parts.append(template[ODME_COLUMNS])
    if not parts:
        raise ValueError(f"No ODME panels for {split}")
    result = pd.concat(parts, ignore_index=True)
    output.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output, index=False)


def odme_structure(template, matrix):
    columns = matrix.tocsc()
    signatures = {}
    for index in range(columns.shape[1]):
        links = tuple(columns.indices[columns.indptr[index]:columns.indptr[index + 1]])
        signatures[links] = signatures.get(links, 0) + 1
    od_sizes = template.groupby(["origin_zone", "destination_zone"]).size()
    return {"measured_links": matrix.shape[0], "paths": matrix.shape[1],
            "unobserved_paths": int(np.sum(np.diff(columns.indptr) == 0)),
            "distinct_measured_path_signatures": len(signatures),
            "ambiguous_path_groups": sum(size > 1 for size in signatures.values()),
            "paths_in_ambiguous_groups": sum(size for size in signatures.values() if size > 1),
            "od_pairs": len(od_sizes), "od_pairs_with_multiple_paths": int((od_sizes > 1).sum()),
            "max_paths_per_od": int(od_sizes.max()),
            "destinations": int(template.destination_zone.nunique())}


def odme_metrics(template, matrix, observed, base, estimate, fit):
    loaded = np.asarray(matrix @ estimate).ravel()
    metrics = {}
    for label, mask in (("fit", fit), ("hidden", ~fit)):
        residual = loaded[mask] - observed[mask]
        normalized_l1 = float(np.sum(np.abs(residual)) / max(float(observed[mask].sum()), 1e-9))
        rmse = float(np.sqrt(np.mean(residual ** 2)))
        metrics.update({f"{label}_link_normalized_l1": normalized_l1,
                        f"{label}_link_s_link": max(0.0, 1.0 - normalized_l1),
                        f"{label}_link_rmse_vph": rmse,
                        f"{label}_link_normalized_rmse": rmse / max(float(observed[mask].mean()), 1)})
    movement = estimate - base
    prior_total, estimate_total = float(base.sum()), float(estimate.sum())
    destinations = template.destination_zone.to_numpy()
    prior_attr = pd.Series(base).groupby(destinations).sum().to_numpy() / max(prior_total, 1e-9)
    estimate_attr = pd.Series(estimate).groupby(destinations).sum().to_numpy() / max(estimate_total, 1e-9)
    metrics.update({"prior_movement_l1": float(np.abs(movement).sum()),
                    "prior_relative_l1": float(np.abs(movement).sum() / max(prior_total, 1e-9)),
                    "prior_relative_l2": float(np.linalg.norm(movement) / max(np.linalg.norm(base), 1)),
                    "destination_distribution_l1_from_prior": float(np.abs(estimate_attr - prior_attr).sum())
                    if prior_total > 0 and estimate_total > 0 else None,
                    "zero_path_fraction": float(np.mean(estimate == 0))})
    return metrics


def validate_odme(args):
    from learning import family_mean, stable_seed, write_json
    if args.repeats < 1 or not args.lambdas or not args.prior_powers:
        raise ValueError("ODME validation needs candidates and at least one repeat")
    if any(not np.isfinite(lam) or lam < 0 for lam in args.lambdas):
        raise ValueError("ODME lambdas must be finite and nonnegative")
    configurations = sorted({(float(lam), power) for lam in args.lambdas for power in args.prior_powers}
                            | {(0.05, 0)})
    candidates = [{"method": "prior", "lambda": None, "prior_power": None}]
    candidates += [{"method": "nnls", "lambda": lam, "prior_power": power} for lam, power in configurations]
    rows, structures = [], {}
    for name in panels(args.release_root, args.panel):
        template, matrix, observed, base = odme_problem(args.release_root, name, args.split)
        if len(observed) < 2:
            raise ValueError("ODME link holdout needs at least two measured links")
        odme_prior_variance(base, 0, args.prior_floor_ratio)
        structures[name] = odme_structure(template, matrix)
        for repeat in range(args.repeats):
            # The first mask stays comparable with earlier lambda-only reports.
            seed = stable_seed(args.seed, name, args.split) if repeat == 0 else stable_seed(args.seed, name, args.split, repeat)
            rng = np.random.default_rng(seed)
            hidden = rng.choice(len(observed), max(1, len(observed) // 5), replace=False)
            fit = np.ones(len(observed), bool)
            fit[hidden] = False
            for candidate in candidates:
                estimate = base.copy() if candidate["method"] == "prior" else solve_odme(
                    matrix[fit], observed[fit], base, candidate["lambda"], candidate["prior_power"], args.prior_floor_ratio)
                rows.append({"panel": name, "repeat": repeat, **candidate,
                             "hidden_link_indices": hidden.tolist(),
                             **odme_metrics(template, matrix, observed, base, estimate, fit)})
    if not rows:
        raise ValueError("No ODME panels to validate")
    summaries = []
    for candidate in candidates:
        matches = [row for row in rows if all(row[key] == value for key, value in candidate.items())]
        summary = dict(candidate)
        for metric in ("hidden_link_normalized_l1", "hidden_link_normalized_rmse", "hidden_link_s_link",
                       "fit_link_normalized_l1", "prior_movement_l1", "prior_relative_l1", "prior_relative_l2",
                       "destination_distribution_l1_from_prior", "zero_path_fraction"):
            usable = [row for row in matches if row[metric] is not None]
            panel_values = {}
            for row in usable:
                panel_values.setdefault(row["panel"], []).append(row[metric])
            summary[metric] = family_mean({name: float(np.mean(values)) for name, values in panel_values.items()}) if usable else None
        repeat_errors = [family_mean({row["panel"]: row["hidden_link_normalized_l1"] for row in matches
                                     if row["repeat"] == repeat}) for repeat in range(args.repeats)]
        summary["hidden_link_normalized_l1_std"] = float(np.std(repeat_errors))
        summary["worst_panel_hidden_link_normalized_l1"] = max(
            float(np.mean([row["hidden_link_normalized_l1"] for row in matches if row["panel"] == name]))
            for name in structures)
        summaries.append(summary)
    best = min(summaries, key=lambda row: row["hidden_link_normalized_l1"])
    report = {"split": args.split, "proxy_only": True, "selection_metric": "hidden_link_normalized_l1",
              "seed": args.seed, "repeats": args.repeats, "prior_floor_ratio": args.prior_floor_ratio,
              "best_proxy_candidate": {key: best[key] for key in ("method", "lambda", "prior_power")},
              "candidates": summaries, "structure": structures, "panels": rows}
    write_json(args.output, report)
    print(report)


def ensemble_predictions(release, split, checkpoints, output, task):
    import torch
    if len(checkpoints) == 1:
        (predict_state if task == "state" else predict_queue)(release, split, checkpoints[0], output)
        return
    keys, values = (STATE_KEYS, ["speed_kmh", "flow_vph"]) if task == "state" else (QUEUE_KEYS, ["queue_pred"])
    reference, total, thresholds = None, None, []
    for number, path in enumerate(checkpoints):
        member = output.with_name(f"{output.stem}.member{number}.csv")
        if task == "state":
            predict_state(release, split, path, member)
        else:
            from learning import predict_queue_v2
            device = "cuda" if torch.cuda.is_available() else "cpu"
            model, config = load_model(path, task, device)
            if config.get("format_version") != 2:
                raise ValueError("Queue probability averaging requires v2 checkpoints")
            predict_queue_v2(model, config, release, split, member, probabilities=True)
            thresholds.append(config["threshold"])
        frame = pd.read_csv(member, dtype={key: str for key in keys}).sort_values(keys).reset_index(drop=True)
        if frame[keys].duplicated().any():
            raise ValueError("Duplicate ensemble keys")
        if reference is None:
            reference = frame[keys]
            total = frame[values].to_numpy(np.float64)
        elif not frame[keys].equals(reference):
            raise ValueError("Ensemble members have different target keys")
        else:
            total += frame[values].to_numpy(np.float64)
    total /= len(checkpoints)
    if task == "queue":
        total = (total >= np.mean(thresholds)).astype(int)
    result = reference.copy()
    result[values] = total
    result.to_csv(output, index=False)


def official(python: str, script: Path, *arguments: str) -> None:
    subprocess.run([python, str(script), *arguments], check=True)


def external_state_inputs(release: Path, paths: list[Path]) -> dict[str, pd.DataFrame]:
    parts = []
    for path in paths:
        frame = pd.read_csv(path, dtype={key: str for key in STATE_KEYS})
        if set(STATE_COLUMNS) - set(frame):
            raise ValueError(f"Missing Task 1 columns: {path}")
        parts.append(frame[STATE_COLUMNS])
    state = pd.concat(parts, ignore_index=True)
    if state.empty or state[STATE_KEYS].isna().any().any() or state.duplicated(STATE_KEYS).any():
        raise ValueError("Empty, incomplete, or duplicate external Task 1 keys")
    for column in ("speed_kmh", "flow_vph"):
        state[column] = pd.to_numeric(state[column], errors="coerce")
    values = state[["speed_kmh", "flow_vph"]].to_numpy()
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("External Task 1 values must be finite and nonnegative")
    result = {}
    for split in ("validation", "private"):
        templates = [pd.read_csv(release / "task1" / name / split / "sample_submission_state.csv",
                                 usecols=STATE_KEYS, dtype=str) for name in panels(release, None)]
        expected = pd.concat(templates, ignore_index=True)
        aligned = expected.merge(state, on=STATE_KEYS, how="left", validate="one_to_one", sort=False)
        if aligned[["speed_kmh", "flow_vph"]].isna().any().any():
            raise ValueError(f"External Task 1 files do not cover every {split} template key")
        result[split] = aligned[STATE_COLUMNS]
    if sum(len(frame) for frame in result.values()) != len(state):
        raise ValueError("External Task 1 files contain keys outside validation/private templates")
    return result


def make_submission(args: argparse.Namespace) -> None:
    official_root = args.official_root.resolve()
    release = args.release_root.resolve()
    for required in (official_root / "src" / "merge_submissions.py", release / "submission_key.csv"):
        if not required.exists():
            raise FileNotFoundError(required)
    state_inputs = external_state_inputs(release, args.state_csv) if args.state_csv else None
    combined: dict[str, list[Path]] = {"state": [], "queue": [], "odme": []}
    for split in ("validation", "private"):
        folder = args.output_dir / split
        folder.mkdir(parents=True, exist_ok=True)
        state, queue, odme = (folder / f"{task}.csv" for task in combined)
        if state_inputs is not None:
            state_inputs[split].to_csv(state, index=False)
        elif args.state_checkpoint:
            ensemble_predictions(release, split, args.state_checkpoint, state, "state")
        else:
            official(sys.executable, official_root / "src" / "task1" / "build_task1_baseline_submission.py",
                     "--release-root", str(release), "--split", split, "--output", str(state))
        if args.queue_checkpoint:
            ensemble_predictions(release, split, args.queue_checkpoint, queue, "queue")
        else:
            official(sys.executable, official_root / "src" / "task2" / "build_task2_persistence_submission.py",
                     "--release-root", str(release), "--split", split, "--output", str(queue))
        predict_odme(release, split, odme, args.odme_lambda, args.odme_prior_power, args.odme_prior_floor_ratio)
        for task, path in zip(combined, (state, queue, odme)):
            combined[task].append(path)
    merged = {}
    for task, files in combined.items():
        table = pd.concat([pd.read_csv(path, dtype=str) for path in files], ignore_index=True)
        keys = {"state": STATE_KEYS, "queue": QUEUE_KEYS, "odme": ODME_KEYS}[task]
        if table[keys].duplicated().any():
            raise ValueError(f"Duplicate {task} keys across splits")
        destination = args.output_dir / f"{task}_both.csv"
        table.to_csv(destination, index=False)
        merged[task] = destination
    temporary = args.output_dir / "submission.pending.csv"
    command = [sys.executable, str(official_root / "src" / "merge_submissions.py"),
               "--state", str(merged["state"]), "--queue", str(merged["queue"]),
               "--odme", str(merged["odme"]), "--key", str(release / "submission_key.csv"),
               "--output", str(temporary)]
    completed = subprocess.run(command, check=True, capture_output=True, text=True)
    print(completed.stdout)
    if "WARNING:" in completed.stdout or "Every scored cell was filled from your files." not in completed.stdout:
        temporary.unlink(missing_ok=True)
        raise RuntimeError("Official merger found missing predictions; upload CSV was not finalized")
    os.replace(temporary, args.output_dir / "submission.csv")
    print(f"Complete submission: {args.output_dir / 'submission.csv'}")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    commands = parser.add_subparsers(dest="command", required=True)
    for name in ("train-state", "train-queue", "submit", "evaluate", "validate-odme", "predict-odme"):
        command = commands.add_parser(name)
        command.add_argument("--release-root", type=Path, required=True)
        if name.startswith("train"):
            command.add_argument("--panel", action="append")
            command.add_argument("--checkpoint", type=Path, required=True)
            command.add_argument("--epochs", type=int, default=75, help="Total epoch budget, including resumed epochs")
            command.add_argument("--width", type=int, default=64)
            command.add_argument("--blocks", type=int, default=2)
            command.add_argument("--heads", type=int, default=4)
            command.add_argument("--dropout", type=float, default=0.1)
            command.add_argument("--backend", choices=["mamba2", "mamba1", "scan"], default="mamba2")
            command.add_argument("--graph", action="store_true")
            command.add_argument("--graph-mode", choices=["mean", "reliability"], default="mean")
            command.add_argument("--missing-aware", action="store_true", help="Learn separate missing-channel embeddings")
            command.add_argument("--station-decoder", choices=["linear", "gated", "split"], default="linear")
            command.add_argument("--adaptive-graph-k", type=int, default=0, help="0 disables learned edges; otherwise select at least 2 neighbors")
            command.add_argument("--wavelet-levels", type=int, choices=[0, 1, 2, 3], default=0, help="Haar trend/detail levels; 0 disables wavelet")
            command.add_argument("--wavelet-input", choices=["signals", "embedded"], help="New training defaults to signals; resume/transfer inherits the checkpoint")
            command.add_argument("--state-regime-balance", action="store_true", help="Fit-only macro regime MSE weighting for State")
            command.add_argument("--lr", type=float, default=1e-3)
            command.add_argument("--seed", type=int, default=42)
            command.add_argument("--max-days", type=int)
            command.add_argument("--batch-size", type=int, default=0, help="0 selects a panel-size-dependent batch")
            command.add_argument("--panel-batch", action="append", default=[], metavar="PANEL=N")
            command.add_argument("--update-windows", type=int,
                                 help="Windows per optimizer update; 0 selects State=8/Queue=48; omitted keeps legacy updates")
            command.add_argument("--workers", type=int, choices=[0, 1, 2], default=1)
            command.add_argument("--cpu-threads", type=int, default=2)
            command.add_argument("--precision", choices=["auto", "fp32", "bf16"], default="auto")
            command.add_argument("--bf16", action="store_true", help="Use CUDA bfloat16 autocast for model forward passes")
            command.add_argument("--patience", type=int, default=5, help="0 disables early stopping")
            command.add_argument("--scheduler-patience", type=int, default=2)
            command.add_argument("--grad-clip", type=float, default=1.0)
            command.add_argument("--log-every", type=int, default=25)
            command.add_argument("--resume", type=Path)
            command.add_argument("--init-state", type=Path, help="Initialize Queue encoder from a v2 State checkpoint")
            command.add_argument("--lambda-n", type=float, default=0.0)
            command.add_argument("--lambda-dn", type=float, default=0.0)
            command.add_argument("--lambda-fd", type=float, default=0.0)
            command.add_argument("--windows-per-day", type=int, default=6)
            command.add_argument("--normal-windows", type=int, default=1)
            command.add_argument("--normal-weight", type=float, default=0.2)
            command.add_argument("--validation-windows", type=int, default=5, help="Fixed maximum per condition/panel")
            command.add_argument("--window-gap-minutes", type=int, default=360)
            command.add_argument("--history-coverage", type=float, default=0.5)
            command.add_argument("--future-coverage", type=float, default=0.5)
            command.add_argument("--label-temperature", type=float, default=0.0, help="km/h; 0 uses hard labels")
            command.add_argument("--uncertainty-margin", type=float, default=0.0, help="km/h; 0 disables uncertainty weighting")
            command.add_argument("--soft-iou-weight", type=float, default=0.2)
            command.add_argument("--speed-weight", type=float, default=0.1)
            command.add_argument("--forward-only", dest="bidirectional", action="store_false")
            command.set_defaults(bidirectional=True)
            command.set_defaults(queue_origin=False, queue_resample=False, queue_condition_gap=False,
                                 queue_changes=False, queue_head="linear", queue_validation_spread=False)
            if name == "train-queue":
                command.add_argument("--queue-origin", action="store_true", help="Append the released masked observation at T")
                command.add_argument("--queue-resample", action="store_true", help="Resample fit windows each epoch; keep evaluation fixed")
                command.add_argument("--queue-condition-gap", action="store_true", help="Apply fit window spacing within each condition")
                command.add_argument("--queue-changes", action="store_true", help="Add threshold margin and masked 5/15-minute changes")
                command.add_argument("--queue-head", choices=["linear", "pooled-mlp"], default="linear")
                command.add_argument("--queue-validation-spread", action="store_true", help="Spread fixed validation windows across dates")
        elif name == "evaluate":
            command.add_argument("--checkpoint", type=Path, required=True)
            command.add_argument("--period", choices=["select", "holdout"], default="select")
            command.add_argument("--output", type=Path, required=True)
            command.add_argument("--panel", action="append")
            command.add_argument("--max-days", type=int)
            command.add_argument("--workers", type=int, choices=[0, 1, 2], default=1)
            command.add_argument("--bootstrap", type=int, default=0)
            command.add_argument("--validation-windows", type=int, help="Override the saved per-condition Queue validation limit")
            command.add_argument("--queue-validation-spread", action="store_true", default=None,
                                 help="Evaluate Queue on fixed windows spread across dates")
        elif name == "validate-odme":
            command.add_argument("--split", choices=["validation", "private"], default="validation")
            command.add_argument("--panel", action="append")
            command.add_argument("--lambdas", type=float, nargs="+", default=[0.01, 0.05, 0.2])
            command.add_argument("--prior-powers", type=int, choices=[0, 1, 2], nargs="+", default=[0])
            command.add_argument("--prior-floor-ratio", type=float, default=0.1)
            command.add_argument("--repeats", type=int, default=3)
            command.add_argument("--seed", type=int, default=42)
            command.add_argument("--output", type=Path, required=True)
        elif name == "predict-odme":
            command.add_argument("--split", choices=["validation", "private"], required=True)
            command.add_argument("--panel", action="append")
            command.add_argument("--odme-lambda", type=float, default=0.05)
            command.add_argument("--prior-power", type=int, choices=[0, 1, 2], default=0)
            command.add_argument("--prior-floor-ratio", type=float, default=0.1)
            command.add_argument("--prior-only", action="store_true", help="Write the released prior without solving")
            command.add_argument("--output", type=Path, required=True)
        else:
            command.add_argument("--official-root", type=Path, required=True)
            command.add_argument("--output-dir", type=Path, required=True)
            state_source = command.add_mutually_exclusive_group()
            state_source.add_argument("--state-checkpoint", type=Path, action="append")
            state_source.add_argument("--state-csv", type=Path, action="append",
                                      help="Task 1 CSVs covering all validation/private targets, including MagiNet outputs")
            command.add_argument("--queue-checkpoint", type=Path, action="append")
            command.add_argument("--odme-lambda", type=float, default=0.05)
            command.add_argument("--odme-prior-power", type=int, choices=[0, 1, 2], default=0)
            command.add_argument("--odme-prior-floor-ratio", type=float, default=0.1)
    args = parser.parse_args()
    if args.command == "train-state":
        fit_state(args)
    elif args.command == "train-queue":
        fit_queue(args)
    elif args.command == "evaluate":
        from learning import evaluate
        evaluate(args)
    elif args.command == "validate-odme":
        validate_odme(args)
    elif args.command == "predict-odme":
        predict_odme(args.release_root, args.split, args.output, args.odme_lambda,
                     args.prior_power, args.prior_floor_ratio, args.panel, args.prior_only)
    else:
        make_submission(args)


if __name__ == "__main__":
    main()
''')


내장 코드 로드: run (696줄)


## 4. 실행 준비와 Task 2 backend 점검

작은 입력으로 Task 2의 순전파·역전파를 먼저 확인합니다. 선택한 backend가 이 GPU에서 실패하면 긴 MagiNet 학습 전에 중단합니다. smoke의 `scan` 결과를 Mamba2 성능으로 해석하지 마세요.


In [25]:
maginet = MODULES["maginet"]
learning = MODULES["learning"]
runner = MODULES["run"]
fast_model = MODULES["fast_model"]

def call_queue(arguments):
    previous = sys.argv[:]
    print("Queue:", shlex.join(map(str, arguments)), flush=True)
    try:
        sys.argv = ["task123-notebook", *map(str, arguments)]
        runner.main()
    finally:
        sys.argv = previous
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def call_maginet(arguments):
    print("MagiNet:", shlex.join(map(str, arguments)), flush=True)
    try:
        maginet.main(list(map(str, arguments)))
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def training_action(best, last, budget, resume):
    """Choose fresh training, resume, or reuse for one task independently."""
    if budget < 1:
        raise ValueError("목표 epoch는 1 이상이어야 합니다.")
    if not best.exists() and not last.exists():
        return "new"
    if not resume:
        raise FileExistsError(f"기존 모델을 보호합니다. RESUME=True 또는 새 결과 폴더를 사용해 주세요: {best.parent}")
    if not best.is_file() or not last.is_file():
        raise FileNotFoundError(f"재개에는 best와 last 체크포인트가 모두 필요합니다: {best}, {last}")
    saved = torch.load(last, map_location="cpu", weights_only=True)
    epoch, bad_epochs = saved["epoch"], saved["bad_epochs"]
    if saved.get("format_version") == 3:
        settings = saved["settings"]
        stopped = (bad_epochs >= settings["patience"]
                   and epoch >= settings["warmup"] + settings["ramp"] + 2)
    elif saved.get("format_version") == 2 and saved.get("task") == "queue":
        patience = saved["training_args"]["patience"]
        stopped = patience > 0 and bad_epochs >= patience
    else:
        raise ValueError(f"지원하지 않는 재개 체크포인트입니다: {last}")
    return "complete" if epoch >= budget or stopped else "resume"

panel_args = [value for panel in (selected_panels or []) for value in ("--panel", panel)]
queue_eval_args = ["--queue-validation-spread", "--validation-windows",
                   str(2 if RUN_MODE == "smoke" else VALIDATION_WINDOWS)]
device = "cuda" if torch.cuda.is_available() else "cpu"
width = 8 if RUN_MODE == "smoke" else 64
features = len(MODULES["data"].queue_feature_names("--queue-changes" in QUEUE_FLAGS))
probe = fast_model.TrafficQueueModel(features=features, node_count=2, width=width, blocks=1,
            heads=2 if RUN_MODE == "smoke" else 4, backend=EFFECTIVE_BACKEND,
            queue_head="pooled-mlp" if "pooled-mlp" in QUEUE_FLAGS else "linear").to(device)
try:
    with torch.autocast(device, dtype=torch.bfloat16, enabled=device == "cuda" and QUEUE_PRECISION == "bf16"):
        logits, speed = probe(torch.zeros(1, 13 if "--queue-origin" in QUEUE_FLAGS else 12, 2, features, device=device),
                              torch.tensor([1, 2], device=device), ((0, 2),),
                              torch.tensor([[0], [1]], device=device))
        loss = logits.float().square().mean() + speed.float().square().mean()
    loss.backward()
    assert logits.shape == speed.shape == (1, 6, 2)
    assert torch.isfinite(loss) and all(torch.isfinite(p.grad).all() for p in probe.parameters() if p.grad is not None)
finally:
    del probe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
(RESULTS / "notebook_run.json").write_text(json.dumps({
    "source_commit": SOURCE_COMMIT, "source_sha256": SOURCE_HASHES,
    "mode": RUN_MODE, "panels": selected_panels, "experiment": MAGINET_EXPERIMENT,
    "queue_backend": EFFECTIVE_BACKEND, "queue_flags": QUEUE_FLAGS,
    "torch": str(torch.__version__), "precision": QUEUE_PRECISION,
    "state_epochs": SMOKE_EPOCHS if RUN_MODE == "smoke" else MAGINET_EPOCHS,
    "queue_epochs": SMOKE_EPOCHS if RUN_MODE == "smoke" else QUEUE_EPOCHS,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Task 2 backend 순전파·역전파 확인 완료")


Task 2 backend 순전파·역전파 확인 완료


## 5. Task 1·3 학습

학습은 6~12월, 체크포인트 선택은 1월 전반, 보간 혼합 선택은 1월 후반입니다. `best_maginet.pt`는 예측용이고 `last_maginet.pt`는 재개용입니다. 새 학습은 기존 체크포인트를 덮어쓰지 않습니다. 마지막 모델 저장 직후 중단돼 보간 선택 파일이 없거나 현재 모델과 맞지 않으면 학습 모드의 평가 셀에서 다시 생성합니다. 다른 공간 구조·손실을 비교할 때는 새 실험 폴더에서 시작해 주세요.


In [26]:
MAGINET_CHECKPOINT = MAGINET_DIR / "full" / "best_maginet.pt"
if RUN_MODE != "predict":
    budget = SMOKE_EPOCHS if RUN_MODE == "smoke" else MAGINET_EPOCHS
    action = training_action(MAGINET_CHECKPOINT, MAGINET_DIR / "full/last_maginet.pt", budget, RESUME)
    print("Task 1·3 학습 상태:", action, "(complete는 저장된 모델을 사용합니다.)")
    if action != "complete":
        args = ["train", "--release-root", RELEASE, "--cache-dir", CACHE_DIR,
                "--output-dir", MAGINET_DIR, "--experiment", MAGINET_EXPERIMENT,
                "--epochs", budget, "--batch-size", MAGINET_BATCH_SIZE,
                "--cpu-threads", CPU_THREADS, "--precision", "auto", *panel_args]
        if action == "new":
            args += ["--checkpoint-metric", "state", "--seed", SEED]
        if RUN_MODE == "smoke":
            args += ["--days-per-regime", 1, "--updates-per-group", 1,
                     "--width", 8, "--blocks", 1, "--heads", 2]
        if action == "resume":
            args += ["--resume"]
        call_maginet(args)
if not MAGINET_CHECKPOINT.is_file():
    raise FileNotFoundError(MAGINET_CHECKPOINT)
print("Task 1·3 체크포인트:", MAGINET_CHECKPOINT)


Task 1·3 학습 상태: new (complete는 저장된 모델을 사용합니다.)
MagiNet: train --release-root /kaggle/working/trafficflow_task123/data/release --cache-dir /kaggle/working/trafficflow_task123/cache/maginet --output-dir /kaggle/working/trafficflow_task123/reports/full/maginet/temporal_direct_dn003_channel_graph --experiment temporal_direct_dn003_channel_graph --epochs 50 --batch-size 2 --cpu-threads 2 --precision auto --checkpoint-metric state --seed 42
{'command': 'train', 'experiment': 'temporal_direct_dn003_channel_graph', 'device': 'cuda', 'precision': 'torch.bfloat16', 'release': '/kaggle/working/trafficflow_task123/data/release', 'cache': '/kaggle/working/trafficflow_task123/cache/maginet', 'reports': '/kaggle/working/trafficflow_task123/reports/full/maginet/temporal_direct_dn003_channel_graph'}
과거 평균 계산 D12_I405_N: 1/214 days
과거 평균 계산 D12_I405_N: 30/214 days
과거 평균 계산 D12_I405_N: 60/214 days
과거 평균 계산 D12_I405_N: 90/214 days
과거 평균 계산 D12_I405_N: 120/214 days
과거 평균 계산 D12_I405_N: 150/214 days
과거 평균 계

KeyboardInterrupt: 

## 6. Task 2 학습

Task 1의 양방향 복원값을 Task 2에 넣지 않습니다. 공개 `T` 관측은 masked 파일에서만 읽고 미래는 제외합니다. 대상은 `D12_I405_N/S`를 제외한 8개 패널입니다. 저장된 모델의 입력·출력층·표본 설정을 바꿔 재개할 수는 없습니다. `QUEUE_FLAGS`를 바꾸면 `QUEUE_TAG`도 바꾸어 새 학습을 시작해 주세요.


In [27]:
if RUN_MODE != "predict":
    budget = SMOKE_EPOCHS if RUN_MODE == "smoke" else QUEUE_EPOCHS
    last = QUEUE_CHECKPOINT.with_name("queue.last.pt")
    action = training_action(QUEUE_CHECKPOINT, last, budget, RESUME)
    print("Task 2 학습 상태:", action, "(complete는 저장된 모델을 사용합니다.)")
    if action != "complete":
        args = ["train-queue", "--release-root", RELEASE, "--checkpoint", QUEUE_CHECKPOINT,
                "--backend", EFFECTIVE_BACKEND, "--epochs", budget,
                "--precision", QUEUE_PRECISION, "--cpu-threads", CPU_THREADS,
                "--seed", SEED, "--workers", 0, *queue_eval_args, *QUEUE_FLAGS, *panel_args]
        if RUN_MODE == "smoke":
            args += ["--max-days", 2, "--width", 8, "--heads", 2, "--blocks", 1,
                     "--batch-size", 2, "--windows-per-day", 1, "--normal-windows", 1]
        if action == "resume":
            args += ["--resume", last]
        call_queue(args)
if not QUEUE_CHECKPOINT.is_file():
    raise FileNotFoundError(QUEUE_CHECKPOINT)
print("Task 2 체크포인트:", QUEUE_CHECKPOINT)


Task 2 학습 상태: new (complete는 저장된 모델을 사용합니다.)
Queue: train-queue --release-root /kaggle/working/trafficflow_task123/data/release --checkpoint /kaggle/working/trafficflow_task123/reports/full/queue/enhanced-mamba2/queue.pt --backend mamba2 --epochs 75 --precision bf16 --cpu-threads 2 --seed 42 --workers 0 --queue-validation-spread --validation-windows 20 --queue-origin --queue-resample --queue-condition-gap --queue-changes --queue-head pooled-mlp
Preparing fit-only means: D12_I5_N days=214
fit-only means D12_I5_N: 50/214
fit-only means D12_I5_N: 100/214
fit-only means D12_I5_N: 150/214
fit-only means D12_I5_N: 200/214
Preparing fit-only means: D12_I5_S days=214
fit-only means D12_I5_S: 50/214
fit-only means D12_I5_S: 100/214
fit-only means D12_I5_S: 150/214
fit-only means D12_I5_S: 200/214
Preparing fit-only means: D7_I10_E days=214
fit-only means D7_I10_E: 50/214
fit-only means D7_I10_E: 100/214
fit-only means D7_I10_E: 150/214
fit-only means D7_I10_E: 200/214
Preparing fit-only means: 

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import shutil

# Input 패널에 표시된 실제 경로로 바꿔 주세요.
previous = Path("/kaggle/input/이전노트북이름/trafficflow_task123")

for relative in ("cache/maginet", "data/processed_v2"):
    src = previous / relative
    dst = WORK_ROOT / relative

    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("캐시 복원:", dst)
    else:
        print("저장된 캐시 없음:", src)

## 7. 평가·보간 비교·과거 검색

`blend_macro.csv`의 `alpha_0.0`은 시간 보간, `alpha_1.0`은 모델 단독입니다. 검색을 켜면 별도 `retrieval/` 아래에 검색을 적용한 모델과 보간의 비교를 저장합니다. 검색 방법·두 채널의 혼합 비율은 1월 전반에서 선택하며, Task 2에는 검색 결과를 넣지 않습니다. Queue는 모델에 저장된 임계값으로 평가합니다. `RUN_MODE="predict"`에서는 이 셀의 평가와 검색 비율 선택을 자동으로 건너뜁니다.


In [ ]:
if RUN_MODE == "predict":
    print("예측 모드: 평가·재선택 없이 저장된 모델과 혼합 선택값을 사용합니다.")
else:
    eval_base = ["--release-root", RELEASE, "--cache-dir", CACHE_DIR,
                 "--checkpoint", MAGINET_CHECKPOINT, "--cpu-threads", CPU_THREADS, *panel_args]

    alpha_selection = MAGINET_DIR / "selected_alpha.json"
    try:
        selected_alpha = json.loads(alpha_selection.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        selected_alpha = {}
    saved_model = torch.load(MAGINET_CHECKPOINT, map_location="cpu", weights_only=True)
    needed_panels = set(selected_panels or saved_model["contexts"])
    del saved_model
    if (selected_alpha.get("checkpoint_sha256") != maginet.file_digest(MAGINET_CHECKPOINT)
            or not needed_panels <= set(selected_alpha.get("panels", []))):
        call_maginet(["evaluate", *eval_base, "--period", "blend", "--select-alpha"])

    call_maginet(["evaluate", *eval_base, "--period", "blend", "--alpha", 1])
    if USE_RETRIEVAL:
        call_maginet(["evaluate", *eval_base, "--period", "early", "--select-retrieval"])
        call_maginet(["evaluate", *eval_base, "--period", "blend", "--retrieval-selection", RETRIEVAL_SELECTION])
    call_queue(["evaluate", "--release-root", RELEASE, "--checkpoint", QUEUE_CHECKPOINT,
                "--period", "select", "--output", QUEUE_DIR / "select.json", *queue_eval_args])
    if RUN_FEBRUARY:
        extra = ["--retrieval-selection", RETRIEVAL_SELECTION] if USE_RETRIEVAL else []
        call_maginet(["evaluate", *eval_base, "--period", "february", *extra])
        call_queue(["evaluate", "--release-root", RELEASE, "--checkpoint", QUEUE_CHECKPOINT,
                    "--period", "holdout", "--output", QUEUE_DIR / "february.json", *queue_eval_args])
    display(pd.read_csv(MAGINET_DIR / "blend_macro.csv"))
    if USE_RETRIEVAL:
        display(pd.read_csv(MAGINET_DIR / "retrieval" / "retrieval_early_comparison.csv"))
        display(pd.read_csv(MAGINET_DIR / "retrieval" / "blend_macro.csv"))
    queue_report = json.loads((QUEUE_DIR / "select.json").read_text())
    print("Queue 대리 검증:", {k: v for k, v in queue_report.items() if k not in {"daily_records", "baseline"}})
    print("동일 표본 비교용 window_signature:", queue_report.get("window_signature"))


## 8. validation·private 예측

Task 1 CSV는 총 교통량 `flow_vph`로 저장하며 Task 3에도 사용합니다. Queue는 저장된 임계값으로 0/1을 출력합니다. smoke 모델은 해당 패널의 형식 점검용이며 전체 제출로 사용하지 않습니다. `RUN_MODE="predict"`에서는 평가를 건너뛰고 저장된 보간·검색 선택값을 사용합니다. 검색을 끄면 1월 평가 원본·train masked 자료는 필요하지 않으며, Queue의 도로 구성 확인을 위한 train 원본 최소 하루는 유지해 주세요. 검색을 사용할 경우에는 같은 fit 자료와 저장된 검색 선택 파일이 있어야 합니다.


In [ ]:
STATE_FILES = []
QUEUE_FILES = []
if MAKE_PREDICTIONS:
    args = ["predict", "--release-root", RELEASE, "--checkpoint", MAGINET_CHECKPOINT,
            "--cache-dir", CACHE_DIR, "--split", "both", "--cpu-threads", CPU_THREADS, *panel_args]
    if USE_RETRIEVAL:
        args += ["--retrieval-selection", RETRIEVAL_SELECTION]
    call_maginet(args)
    state_folder = MAGINET_DIR / ("retrieval/submissions" if USE_RETRIEVAL else "submissions")
    saved = torch.load(MAGINET_CHECKPOINT, map_location="cpu", weights_only=True)
    predicted_panels = selected_panels or sorted(saved["contexts"])
    STATE_FILES = [state_folder / f"task1-{panel}-{split}.csv"
                   for split in ("validation", "private") for panel in predicted_panels]
    if not all(p.is_file() for p in STATE_FILES):
        raise FileNotFoundError("Task 1 예측 파일이 모두 생성되지 않았습니다.")
    del saved
    for split in ("validation", "private"):
        path = QUEUE_DIR / f"queue-{split}.csv"
        runner.predict_queue(RELEASE, split, QUEUE_CHECKPOINT, path)
        QUEUE_FILES.append(path)
    print("Task 1 CSV:", *map(str, STATE_FILES), sep="\n")
    print("Task 2 CSV:", *map(str, QUEUE_FILES), sep="\n")


## 9. 최종 제출 병합

전체 패널 모델에서만 제출을 만듭니다. `TASK4_FILES`가 있으면 기존 Task 4 출력을 사용하고, 없으면 `BUILD_TASK4_BASELINE=True`일 때 원래 NNLS를 사용합니다. Task 4를 바꾸는 실험은 수행하지 않습니다. 모든 task의 자연 키를 `submission_key.csv`에 연결하며 누락·중복·템플릿 밖 키·음수·비유한 값·Queue의 0/1 위반을 거부합니다. 완성 전에는 `.pending.csv`만 쓰고, 전부 맞을 때만 `submission.csv`로 확정합니다.


In [ ]:
TASK_KEYS = {"state": runner.STATE_KEYS, "queue": runner.QUEUE_KEYS, "odme": runner.ODME_KEYS}
TASK_VALUES = {"state": ["speed_kmh", "flow_vph"], "queue": ["queue_pred"], "odme": ["path_flow"]}
OUTPUT_COLUMNS = ["submission_id", "task", "speed_kmh", "flow_vph", "queue_pred", "path_flow"]

def merge_submission(files, key_path, output, chunksize=250_000):
    tables, used = {}, {}
    for task in TASK_KEYS:
        if not files[task]:
            raise ValueError(f"{task} CSV가 필요합니다.")
        columns = TASK_KEYS[task] + TASK_VALUES[task]
        frame = pd.concat([pd.read_csv(p, dtype=str, usecols=columns) for p in files[task]], ignore_index=True)
        if frame.empty or frame[TASK_KEYS[task]].isna().any().any() or frame.duplicated(TASK_KEYS[task]).any():
            raise ValueError(f"{task}: 비었거나 누락·중복 키가 있습니다.")
        for column in TASK_VALUES[task]:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        values = frame[TASK_VALUES[task]].to_numpy(float)
        if not np.isfinite(values).all() or (values < 0).any() or (task == "queue" and not np.isin(values, [0, 1]).all()):
            raise ValueError(f"{task}: 잘못된 예측값입니다.")
        tables[task] = frame.set_index(TASK_KEYS[task])
        used[task] = np.zeros(len(frame), bool)
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    pending = output.with_suffix(".pending.csv")
    written = 0
    chunks = pd.read_csv(key_path, dtype=str, chunksize=chunksize)
    try:
        for chunk in chunks:
            if chunk[["submission_id", "task"]].isna().any().any() or not chunk.task.isin(TASK_KEYS).all():
                raise ValueError("제출 키의 ID 또는 task가 잘못되었습니다.")
            out = chunk[["submission_id", "task"]].reset_index(drop=True).copy()
            for column in OUTPUT_COLUMNS[2:]:
                out[column] = 0.
            chunk = chunk.reset_index(drop=True)
            for task, table in tables.items():
                mask = out.task.eq(task)
                if not mask.any():
                    continue
                wanted = pd.MultiIndex.from_frame(chunk.loc[mask, TASK_KEYS[task]])
                indices = table.index.get_indexer(wanted)
                if (indices < 0).any():
                    raise ValueError(f"{task}: 제출 템플릿에 필요한 예측이 누락됐습니다.")
                if len(np.unique(indices)) != len(indices) or used[task][indices].any():
                    raise ValueError(f"{task}: 제출 키에 자연 키가 중복됐습니다.")
                out.loc[mask, TASK_VALUES[task]] = table.iloc[indices][TASK_VALUES[task]].to_numpy()
                used[task][indices] = True
            out[OUTPUT_COLUMNS].to_csv(pending, index=False, mode="w" if written == 0 else "a", header=written == 0)
            written += len(out)
            print(f"제출 병합: {written:,}행", flush=True)
        if written == 0 or any(not mask.all() for mask in used.values()):
            raise ValueError("제출 키가 비었거나 템플릿 밖의 예측 키가 있습니다.")
        pending.replace(output)
    except BaseException:
        pending.unlink(missing_ok=True)
        raise
    finally:
        chunks.close()
    return written

SUBMISSION_PATH = WORK_ROOT.parent / "submission.csv"
if MAKE_SUBMISSION and MAKE_PREDICTIONS:
    state_saved = torch.load(MAGINET_CHECKPOINT, map_location="cpu", weights_only=True)
    queue_saved = torch.load(QUEUE_CHECKPOINT, map_location="cpu", weights_only=True)
    state_coverage = set(selected_panels or state_saved["contexts"])
    required_queue = {p.name for p in (RELEASE / "task2").iterdir()
                      if p.name not in {"D12_I405_N", "D12_I405_S"}
                      and (p / "validation" / "sample_submission_queue.csv").is_file()}
    full_coverage = state_coverage == set(all_panels) and required_queue <= set(queue_saved["trained_panels"])
    del state_saved, queue_saved
    if RUN_MODE == "smoke" or not full_coverage:
        print("점검용 또는 일부 패널 모델입니다. 패널별 CSV를 확인하고 전체 학습 후 제출을 생성해 주세요.")
    else:
        odme_files = list(map(Path, TASK4_FILES))
        if not odme_files:
            if not BUILD_TASK4_BASELINE:
                raise ValueError("TASK4_FILES를 지정하거나 BUILD_TASK4_BASELINE을 켜 주세요.")
            for split in ("validation", "private"):
                path = RESULTS / f"odme-baseline-{split}.csv"
                runner.predict_odme(RELEASE, split, path, 0.05, 0, 0.1)
                odme_files.append(path)
        count = merge_submission({"state": STATE_FILES, "queue": QUEUE_FILES, "odme": odme_files},
                                 RELEASE / "submission_key.csv", SUBMISSION_PATH)
        print(f"완성된 제출: {SUBMISSION_PATH} ({count:,}행)")


## 10. 저장할 결과

Kaggle의 **Save Version → Save & Run All** 또는 실행한 세션의 Output 저장으로 `trafficflow_task123/reports/`를 보존해 주세요. 새 세션에서 이 Output을 Input으로 연결할 수 있습니다. 캐시는 다시 생성할 수 있으며, 재개 시 입력 내용·설정이 원본과 맞는지 검사합니다.

- `reports/<smoke|full>/maginet/<실험>/full/`: `best_maginet.pt`, `last_maginet.pt`, 설정·학습 기록
- 같은 실험의 `selected_alpha.json`, `retrieval/selected_retrieval.json`: 저장한 체크포인트에 맞는 혼합 선택값
- `reports/<smoke|full>/queue/<이름>-<backend>/`: `queue.pt`, `queue.last.pt`, epoch 로그, 평가, 예측
- `/kaggle/working/submission.csv`: 전체 task·패널의 키가 일치할 때만 생성되는 업로드 파일

소스 수정이나 GPU·정밀도 변경 후에는 이전 실행과 같은 수치가 나온다고 가정하지 마세요. State·`selection_proxy`·Physics 대리 지표·공식 Kaggle 종합 점수는 서로 구분합니다.
